# Face-forgery detection on FaceForensics++ and Google's DeepFake Detection set
### A reproducible image- and video-level benchmark for Kaggle (2 × T4)

**What the notebook does, top to bottom, without manual steps**

1. Finds FaceForensics++ (FF++) and Google/Jigsaw's DeepFake Detection set (DFD) among the attached Kaggle inputs, checks the video counts against the published ones and assigns the **official FF++ train/val/test split** (720/140/140 source videos, embedded below from the FaceForensics repository).
2. Samples frames from every video, finds the face with OpenCV's **YuNet** detector (in a frame with several people, the manipulated face is found by comparing with the counterpart video) and stores one square face crop per frame (1.3 × the face box, 224 × 224 pixels).
3. Trains two kinds of detector on FF++ c23: an ImageNet-pretrained **EfficientNet-B4**, fine-tuned end to end (three seeds per configuration), and a **linear probe on frozen CLIP ViT-L/14** image embeddings. Both GPUs work in parallel, one training job each.
4. Tests every detector on what it has not seen: the FF++ test videos, the **same videos at c40** compression, a **held-out manipulation method** (leave-one-method-out, plus FaceShifter, which no detector is trained on) and the whole **DFD** set (other actors, other scenes, other face-swap pipeline).
5. Repeats the training with DFD's training actors added (actor-disjoint DFD split), to measure what DFD is worth as **additional training data**.
6. Reports video-level AUC with **95% bootstrap intervals** (test videos resampled by source pair for FF++ and two-way by actor for DFD, EfficientNet seeds resampled too), and frame-level AUC, AP, EER, accuracy at a threshold fixed on the validation split, calibration error and Grad-CAM maps.
7. Writes every figure as **600 DPI PNG and vector PDF**, every table as CSV, and packs them into **one ZIP file** in `/kaggle/working`, with a download link at the end.

**How to run it on Kaggle**

1. *File → Import notebook* and choose this file. In the right-hand panel: *Session options → Accelerator → GPU T4 x2*, and **Internet on** (needed in every run: the YuNet face detector comes from GitHub, the EfficientNet-B4 and CLIP weights from Hugging Face).
2. *Add Input → Datasets*, and add these four (the first is required, the others are strongly recommended):
   `abbasaliyev/ffpp-core-c23-videos` (FF++ c23, 6,000 videos, 11 GB) · `abbasaliyev/ffpp-core-c40-videos` (FF++ c40, 1.5 GB) · `abbasaliyev/ffpp-dfd-c23-videos` (DFD c23, 3,431 videos, 24 GB) · `abbasaliyev/ffpp-dfd-c40-videos` (DFD c40, 3.4 GB).
   Also recognised: `xdxd003/ff-c23`, `sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset`, and the folder layout written by the official FaceForensics download script. If an input is missing, the notebook tries to attach it by itself.
3. Recommended first: set `QUICK = True` in the first code cell and *Save Version → Save & Run All*. It runs the whole pipeline on a small subset in roughly 30 to 60 minutes (an estimate) and proves that the inputs, the downloads, both GPUs and the ZIP work. Its outputs are stamped QUICK TEST and are not results.
4. Then set `QUICK = False` and *Save & Run All* again. By estimates from published hardware figures (not measured: no GPU was available to the author), the full run takes roughly 6 to 9 hours: 2 to 3.5 hours of face extraction on the CPUs, then 3 to 5 hours of training on the two GPUs. The notebook measures its own speed, starts no work it cannot finish before Kaggle's 12-hour limit, and writes the ZIP in any case. If jobs are left over, attach this notebook's own output as an input and run it again: the same extraction plan is used, finished jobs are reused, and only the missing ones run.
5. Download `ffpp_dfd_benchmark_artifacts.zip` from the notebook's *Output* tab (or with the link printed at the end of an interactive session).

`DRY_RUN = True` rehearses everything in a few minutes on synthetic videos with tiny random networks, on CPU, without any dataset. Everything it writes is stamped DRY RUN.

**Data terms.** FaceForensics++ and the DeepFake Detection set are distributed under the *FaceForensics Terms of Use*, which allow non-commercial research and education only. The Kaggle copies listed above are **unofficial mirrors** (their uploader says so). Whether a mirror is acceptable for a publication is the team's decision; the official route is the request form at [github.com/ondyari/FaceForensics](https://github.com/ondyari/FaceForensics), whose download script writes a layout this notebook also reads.

**What has and has not been verified.** The notebook was run end to end in rehearsal mode (synthetic videos in the real folder layouts, tiny random networks, CPU) with two sets of library versions, including a check that a mark planted on every manipulated face is found again by the detectors with AUC 1.0, and with simulated failures (a crashed worker, the time limit, a resumed session, missing inputs). It has **not** been run on Kaggle, on a GPU, with the pretrained weights or on the real videos by its author; run times are estimates that the notebook replaces with its own measurements.

In [ ]:
# =============================================================================
# 1. ENVIRONMENT AND RUN MODE
# =============================================================================
import os, sys, json, time, math, glob, hashlib, zipfile, shutil, traceback, warnings, platform, datetime
import re, random, subprocess, itertools, threading
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- run mode ----------------------------------------------------------------
# DRY_RUN = True : a few minutes on SYNTHETIC videos with tiny random networks, on CPU. Needs no dataset,
#                  no GPU and no internet. Every figure and table is stamped DRY RUN. Nothing in it is a result.
# QUICK   = True : the real pipeline on a small subset of the real videos (roughly 30-60 min on 2 x T4).
#                  Proves that inputs, downloads, GPUs and the ZIP work. Stamped QUICK TEST. Not a result.
# both False     : the benchmark.
DRY_RUN = False
QUICK = False
DRY_RUN = DRY_RUN or os.environ.get("FFB_DRY_RUN", "0") == "1"
QUICK = (QUICK or os.environ.get("FFB_QUICK", "0") == "1") and not DRY_RUN
MODE = "dryrun" if DRY_RUN else ("quick" if QUICK else "full")

def _process_start():
    """When this Python process (the notebook's kernel) started: Kaggle's session clock runs from about then, not from
    whenever this cell is reached."""
    try:
        import psutil
        return float(psutil.Process(os.getpid()).create_time())
    except Exception:
        return time.time()
T_START = min(time.time(), _process_start())
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = os.environ.get("FFB_WORK_DIR") or ("/kaggle/working" if ON_KAGGLE else os.path.abspath("ffb_work"))
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

def _first_writable(cands):
    for c in cands:
        try:
            os.makedirs(c, exist_ok=True)
            probe = os.path.join(c, ".ffb_probe")
            open(probe, "w").close(); os.remove(probe)
            return c
        except Exception:
            continue
    raise RuntimeError("no writable scratch directory")

# Face crops are large (about 17 GB in the full run) and are NOT kept as notebook output: they go to the
# session's scratch disk. Everything small that a later session can reuse goes to STATE in /kaggle/working.
TEMP = _first_writable([os.environ.get("FFB_TEMP_DIR") or "/kaggle/temp/ffb", "/tmp/ffb_temp", os.path.join(WORK, "_ffb_temp")])
INPUT_ROOT = os.environ.get("FFB_INPUT_ROOT") or "/kaggle/input"
if DRY_RUN:
    INPUT_ROOT = os.path.join(TEMP, "dryrun_input")
SUFFIX = {"dryrun": "_dryrun", "quick": "_quick", "full": ""}[MODE]
OUT = os.path.join(WORK, "ffb_outputs" + SUFFIX)          # what goes into the ZIP; emptied at the start of every run
STATE = os.path.join(WORK, "ffb_state" + SUFFIX)          # finished jobs and features; kept, so that a later run can resume
CROPS_DIR = os.path.join(TEMP, "crops" + SUFFIX)
if os.path.isdir(OUT):
    shutil.rmtree(OUT, ignore_errors=True)
SRC = os.path.join(WORK, "ffb_src")
DIRS = {k: os.path.join(OUT, k) for k in ("figures/png", "figures/pdf", "tables", "results", "metadata", "logs")}
for d in list(DIRS.values()) + [SRC, STATE, CROPS_DIR]:
    os.makedirs(d, exist_ok=True)
if SRC not in sys.path:
    sys.path.insert(0, SRC)

LOG_PATH = os.path.join(DIRS["logs"], "run.log")
def log(*a):
    msg = " ".join(str(x) for x in a)
    line = f"[{(time.time() - T_START) / 60:7.1f} min] {msg}"
    print(line, flush=True)
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass

def elapsed_min():
    return (time.time() - T_START) / 60.0

import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
N_CPU = os.cpu_count() or 2
# GPU workers: one per GPU. Without a GPU (rehearsal) the same code runs with CPU workers.
N_WORKERS = N_GPU if N_GPU > 0 else int(os.environ.get("FFB_CPU_WORKERS", "2" if DRY_RUN else "1"))
WORKER_DEVICE = "cuda" if N_GPU > 0 else "cpu"

RUN = {
    "notebook": "ffpp_dfd_face_forgery_benchmark_kaggle_t4x2", "notebook_version": "1.0", "mode": MODE,
    "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "dry_run": bool(DRY_RUN), "quick": bool(QUICK), "on_kaggle": bool(ON_KAGGLE),
    "python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "n_gpu": N_GPU, "gpu_names": [torch.cuda.get_device_name(i) for i in range(N_GPU)], "n_cpu": N_CPU,
    "stages": OrderedDict(), "ok": {}, "warnings": [], "data": {},
}

def _versions():
    out = {}
    for name in ("cv2", "scipy", "sklearn", "matplotlib", "timm", "transformers", "huggingface_hub", "kagglehub", "PIL"):
        try:
            out[name] = __import__(name).__version__
        except Exception as e:
            out[name] = f"not importable ({type(e).__name__})"
    return out
RUN["versions"] = _versions()

def warn(msg):
    RUN["warnings"].append(str(msg))
    log("WARNING:", msg)

def _machine_start_gap_min():
    """Minutes between the start of this session's machine (its first process) and of this kernel; None if unknown."""
    try:
        import psutil
        return (T_START - float(psutil.Process(1).create_time())) / 60.0
    except Exception:
        return None
if ON_KAGGLE and not DRY_RUN:
    _gap = _machine_start_gap_min()
    if _gap is not None and 15 <= _gap <= 12 * 60:
        warn(f"this kernel started {_gap:.0f} min after this session's machine. The time limits are counted from the kernel's start; if the "
             "session has been running all that time (a kernel restart inside an interactive session), Kaggle may stop it before the ZIP is "
             "written. For a long run use Save Version -> Save & Run All, which starts a fresh session.")

ST = {}

def peak_gb():
    try:
        import resource
        return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)
    except Exception:
        return 0.0

def out_of_time(late=False):
    """After HARD_STOP_MIN no new stage starts (a running one stops at its next check); tables and figures
    may still run until FINAL_STOP_MIN; the export always runs."""
    return "CFG" in globals() and elapsed_min() > float(CFG.get("FINAL_STOP_MIN" if late else "HARD_STOP_MIN", 1e9))

class SkipStage(Exception):
    """A stage that has nothing to do (for example, an optional corpus is not attached)."""

class TimeUp(Exception):
    """Raised inside a long stage when the session limit is close. What the stage finished is kept."""

def check_time(what="", late=False):
    if out_of_time(late=late):
        raise TimeUp(f"session time limit is close; stopped {what}".strip())

def run_stage(name, fn, needs=(), late_ok=False):
    """Run one stage. A failure is recorded and reported, never raised: a committed Kaggle run that ends in an
    exception keeps no output, so every stage after the data check is fenced and the export always runs."""
    missing = [k for k in needs if not RUN["ok"].get(k)]
    if missing:
        RUN["stages"][name] = {"status": "skipped", "reason": "needs " + ", ".join(missing)}
        RUN["ok"][name] = False
        log(f"-- stage {name}: SKIPPED (needs {', '.join(missing)})")
        return None
    if name != "export" and out_of_time(late=late_ok):
        RUN["stages"][name] = {"status": "skipped", "reason": "session time limit is close; skipped so that the outputs can still be written"}
        RUN["ok"][name] = False
        log(f"-- stage {name}: SKIPPED (session time limit is close)")
        return None
    t0 = time.time()
    try:
        if name in os.environ.get("FFB_FAIL_STAGE", "").split(","):          # test hook
            raise RuntimeError("failure injected by the FFB_FAIL_STAGE test hook")
        out = fn()
        RUN["stages"][name] = {"status": "ok", "minutes": round((time.time() - t0) / 60, 2), "peak_memory_gb": peak_gb()}
        RUN["ok"][name] = True
        ST[name] = out
        log(f"-- stage {name}: ok ({(time.time() - t0) / 60:.1f} min; peak host memory so far {peak_gb():.1f} GB)")
        return out
    except SkipStage as e:
        RUN["stages"][name] = {"status": "skipped", "reason": str(e)}
        RUN["ok"][name] = False
        log(f"-- stage {name}: SKIPPED ({e})")
        return None
    except TimeUp as e:
        RUN["stages"][name] = {"status": "stopped", "reason": str(e), "minutes": round((time.time() - t0) / 60, 2)}
        RUN["ok"][name] = False
        log(f"-- stage {name}: STOPPED ({e}); what it had finished is kept")
        return None
    except Exception as e:
        tb = traceback.format_exc()
        RUN["stages"][name] = {"status": "FAILED", "error": f"{type(e).__name__}: {e}", "traceback": tb[-4000:], "minutes": round((time.time() - t0) / 60, 2)}
        RUN["ok"][name] = False
        log(f"-- stage {name}: FAILED  {type(e).__name__}: {e}")
        print(tb[-2500:])
        return None

log(f"mode {MODE} | work {WORK} | scratch {TEMP} | input {INPUT_ROOT}")
log(f"torch {torch.__version__} | GPUs: {RUN['gpu_names'] or 'none'} | CPU cores: {N_CPU} | workers: {N_WORKERS} x {WORKER_DEVICE}")
log(" | ".join(f"{k} {v}" for k, v in RUN["versions"].items()))
if N_GPU != 2 and not DRY_RUN:
    warn(f"this notebook was written for 2 x T4; found {N_GPU} GPU(s). It adapts, but timings will differ.")
if N_GPU == 0 and not DRY_RUN and os.environ.get("FFB_ALLOW_CPU", "0") != "1":          # (the variable is a test hook)
    raise RuntimeError("No GPU. Choose Session options -> Accelerator -> GPU T4 x2. (Set DRY_RUN = True for a rehearsal on CPU.)")

In [ ]:
# =============================================================================
# 2. CONFIGURATION AND PROTOCOL
# =============================================================================
# The official FaceForensics++ split: 360 / 70 / 70 pairs of source videos (720 / 140 / 140 videos), copied from
# dataset/splits/{train,val,test}.json of github.com/ondyari/FaceForensics (MIT-licensed code repository).
# A manipulated video "A_B" belongs to the split of its pair (A, B).
FFPP_SPLITS = json.loads('{"train":["071_054","087_081","881_856","187_234","645_688","754_758","811_920","710_788","628_568","312_021","950_836","059_050","524_580","751_752","918_934","604_703","296_293","518_131","536_540","969_897","372_413","357_432","809_799","092_098","302_323","981_985","512_495","088_060","795_907","535_587","297_270","838_810","850_764","476_400","268_269","033_097","226_491","784_769","195_442","678_460","320_328","451_449","409_382","556_588","027_009","196_310","241_210","295_099","043_110","753_789","716_712","508_831","005_010","276_185","498_433","294_292","105_180","984_967","318_334","356_324","344_020","289_228","022_489","137_165","095_053","999_960","481_469","534_490","543_559","150_153","598_178","475_265","671_677","204_230","863_853","561_998","163_031","655_444","038_125","735_774","184_205","499_539","717_684","878_866","127_129","286_267","032_944","681_711","236_237","989_993","537_563","814_871","509_525","221_206","808_829","696_686","431_447","737_719","609_596","408_424","976_954","156_243","434_438","627_658","025_067","635_642","523_541","572_554","215_208","651_835","975_978","792_903","931_936","846_845","899_914","209_016","398_457","797_844","360_437","738_804","694_767","790_014","657_644","374_407","728_673","193_030","876_891","553_545","331_260","873_872","109_107","121_093","143_140","778_798","983_113","504_502","709_390","940_941","894_848","311_387","562_626","330_162","112_892","765_867","124_085","665_679","414_385","555_516","072_037","086_090","202_348","341_340","333_377","082_103","569_921","750_743","211_177","770_791","329_327","613_685","007_132","304_300","860_905","986_994","378_368","761_766","232_248","136_285","601_653","693_698","359_317","246_258","500_592","776_676","262_301","307_365","600_505","833_826","361_448","473_366","885_802","277_335","667_446","522_337","018_019","430_459","886_877","456_435","239_218","771_849","065_089","654_648","151_225","152_149","229_247","624_570","290_240","011_805","461_250","251_375","639_841","602_397","028_068","338_336","964_174","782_787","478_506","313_283","659_749","690_689","893_913","197_224","253_183","373_394","803_017","305_513","051_332","238_282","621_546","401_395","510_528","410_411","049_946","663_231","477_487","252_266","952_882","315_322","216_164","061_080","603_575","828_830","723_704","870_001","201_203","652_773","108_052","272_396","040_997","988_966","281_474","077_100","146_256","972_718","303_309","582_172","222_168","884_968","217_117","118_120","242_182","858_861","101_096","697_581","763_930","839_864","542_520","122_144","687_615","544_532","721_715","179_212","591_605","275_887","996_056","825_074","530_594","757_573","611_760","189_200","392_339","734_699","977_075","879_963","910_911","889_045","962_929","515_519","062_066","937_888","199_181","785_736","079_076","155_576","748_355","819_786","577_593","464_463","439_441","574_547","747_854","403_497","965_948","726_713","943_942","160_928","496_417","700_813","756_503","213_083","039_058","781_806","620_619","351_346","959_957","264_271","006_002","391_406","631_551","501_326","412_274","641_662","111_094","166_167","130_139","938_987","055_147","990_008","013_883","614_616","772_708","840_800","415_484","287_426","680_486","057_070","590_034","194_235","291_874","902_901","343_363","279_298","393_405","674_744","244_822","133_148","636_578","637_427","041_063","869_780","733_935","259_345","069_961","783_916","191_188","526_436","123_119","207_908","796_740","815_730","173_171","383_353","458_722","533_450","618_629","646_643","531_549","428_466","859_843","692_610"],"val":["720_672","939_115","284_263","402_453","820_818","762_832","834_852","922_898","104_126","106_198","159_175","416_342","857_909","599_585","443_514","566_617","472_511","325_492","816_649","583_558","933_925","419_824","465_482","565_589","261_254","992_980","157_245","571_746","947_951","926_900","493_538","468_470","915_895","362_354","440_364","640_638","827_817","793_768","837_890","004_982","192_134","745_777","299_145","742_775","586_223","483_370","779_794","971_564","273_807","991_064","664_668","823_584","656_666","557_560","471_455","042_084","979_875","316_369","091_116","023_923","702_612","904_046","647_622","958_956","606_567","632_548","927_912","350_349","595_597","727_729"],"test":["953_974","012_026","078_955","623_630","919_015","367_371","847_906","529_633","418_507","227_169","389_480","821_812","670_661","158_379","423_421","352_319","579_701","488_399","695_422","288_321","705_707","306_278","865_739","995_233","755_759","467_462","314_347","741_731","970_973","634_660","494_445","706_479","186_170","176_190","380_358","214_255","454_527","425_485","388_308","384_932","035_036","257_420","924_917","114_102","732_691","550_452","280_249","842_714","625_650","024_073","044_945","896_128","862_047","607_683","517_521","682_669","138_142","552_851","376_381","000_003","048_029","724_725","608_675","386_154","220_219","801_855","161_141","949_868","880_135","429_404"]}')
FFPP_SPLITS_SHA256 = "ac5a0539ec24b8cfefb19b5a1f6c0aae9b75239a7c0202abe2e8973cb326b9f1"
if hashlib.sha256(json.dumps(FFPP_SPLITS, separators=(",", ":"), sort_keys=True).encode()).hexdigest() != FFPP_SPLITS_SHA256:
    raise RuntimeError("the embedded FaceForensics++ split was modified; restore it from dataset/splits/*.json of github.com/ondyari/FaceForensics")
assert [len(FFPP_SPLITS[s]) for s in ("train", "val", "test")] == [360, 70, 70]

CLASSIC = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]      # the four methods of the original FF++ benchmark
ALL_METHODS = CLASSIC + ["FaceShifter"]                                  # FaceShifter was added in 2020; no detector here is trained on it
SHORT = {"Deepfakes": "DF", "Face2Face": "F2F", "FaceSwap": "FS", "NeuralTextures": "NT", "FaceShifter": "FSh", "DFD": "DFD", "real": "real"}

CFG = dict(
    SEED=20261020,

    # ---- data ---------------------------------------------------------------
    # Kaggle datasets that the notebook tries to attach by itself when nothing of that kind is found under
    # /kaggle/input. Their contents were described by their uploaders, not inspected by the author of this
    # notebook; the notebook counts what it finds and compares the counts with the published ones.
    KAGGLE_DATASETS=dict(
        ffpp_c23=["abbasaliyev/ffpp-core-c23-videos", "xdxd003/ff-c23"],
        ffpp_c40=["abbasaliyev/ffpp-core-c40-videos"],
        dfd_c23=["abbasaliyev/ffpp-dfd-c23-videos", "sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset"],
        dfd_c40=["abbasaliyev/ffpp-dfd-c40-videos"],
    ),
    ALLOW_KAGGLEHUB=True,
    COMPRESSION_OVERRIDES={},          # {"<folder name of the copy>": "c23" or "c40"} if a copy's folder names do not say it
    PUBLISHED=dict(ffpp_real=1000, ffpp_per_method=1000, dfd_real=363, dfd_fake=3068),
    DFD_TRAIN_ACTOR_SHARE=0.5,         # DFD actors are split in two halves; fakes whose two actors fall in different halves are not used

    # ---- frames and faces -----------------------------------------------------
    DECODE_WINDOW=300,                 # frames are sampled evenly from the first 300 frames (10-12 s) of a video
    FRAMES=dict(train_real=16, train_fake=4, val=8, test=16, dfd=8),   # per video; real training videos get more frames to balance the classes
    CROP_SIZE=224, CROP_SCALE=1.3,     # square crop of 1.3 x the face box, resized to 224 x 224
    DET_SCORE=0.6, DET_MAX_SIDE=640,   # YuNet confidence threshold; detection runs on a copy at most 640 px on its longer side
    EXTRACT_PROCESSES=None,            # None: one per CPU core
    DISK_RESERVE_GB=4.0,

    # ---- detectors ------------------------------------------------------------
    # EfficientNet-B4: adapted from DeepfakeBench's efficientnetb4.yaml. The same: Adam, lr 2e-4, weight decay 5e-4,
    # batch 32, at most 10 epochs, flip/rotation/blur/brightness-contrast/JPEG augmentation each with p = 0.5.
    # Different (see section 6): 224 px (256 there), timm's ImageNet weights and normalisation (0.5/0.5 there),
    # 16 real / 4 manipulated frames per training video with a class-weighted loss (32 frames, natural ratio there),
    # and the epoch chosen on validation video AUC (ties broken by the lower validation loss) with patience 3 (a fixed number of epochs there).
    EFFNET=dict(arch="efficientnet_b4", pretrained=True, epochs=10, patience=3, batch_size=32, eval_batch_size=128,
                lr=2e-4, weight_decay=5e-4, max_steps_per_epoch=0),
    SEEDS=[0, 1, 2],
    CLIP=dict(model="openai/clip-vit-large-patch14", image_size=224, batch_size=128,
              mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]),
    PROBE_C_GRID=[0.001, 0.01, 0.1, 1.0, 10.0],
    CONFIGS=["main4", "lomo_Deepfakes", "lomo_Face2Face", "lomo_FaceSwap", "lomo_NeuralTextures", "ffdfd"],

    # ---- statistics -----------------------------------------------------------
    N_BOOT=2000, CI_LEVEL=0.95, ECE_BINS=15, FRAMES_CURVE=[1, 2, 4, 8, 16],

    # ---- time (minutes after the notebook's kernel started) ---------------------
    EXTRACT_BUDGET_MIN=260.0,          # face extraction starts no new video after this (the cut rules of section 5 apply first)
    HARD_STOP_MIN=650.0,               # no new stage after this; Kaggle ends a session at 720 min (counted from the session start, not from this cell)
    FINAL_STOP_MIN=690.0,              # tables and figures may run until here; the export always runs
    EVAL_RESERVE_MIN=45.0,             # GPU workers stop this long before HARD_STOP_MIN
    LOADER_WORKERS=None,               # data-loading processes per GPU worker; None: (CPU cores - 1) / GPU workers

    DPI=600,
    ZIP_NAME="ffpp_dfd_benchmark_artifacts.zip",
)

if QUICK:
    # Roughly 30 to 60 minutes on 2 x T4 (an estimate). Same code, a small subset. Stamped QUICK TEST; not the benchmark.
    CFG.update(SEEDS=[0], CONFIGS=["main4", "lomo_NeuralTextures", "ffdfd"], N_BOOT=500,
               FRAMES=dict(train_real=8, train_fake=2, val=4, test=8, dfd=4),
               EFFNET=dict(CFG["EFFNET"], epochs=2, patience=2), EXTRACT_BUDGET_MIN=40.0)
    CFG["QUICK_SUBSET"] = dict(train_pairs=40, val_pairs=10, test_pairs=10, dfd_actors=8)
if DRY_RUN:
    # Minutes on CPU. Tiny random networks, synthetic videos. Stamped DRY RUN; nothing in it means anything.
    CFG.update(SEEDS=[0, 1], N_BOOT=300, CROP_SIZE=96, DECODE_WINDOW=40,
               FRAMES=dict(train_real=4, train_fake=2, val=2, test=4, dfd=2),
               EFFNET=dict(CFG["EFFNET"], arch="efficientnet_b0", pretrained=False, epochs=2, patience=2, batch_size=16, eval_batch_size=64, max_steps_per_epoch=25),
               CLIP=dict(CFG["CLIP"], model="tiny-random-clip", image_size=96, batch_size=64),
               FRAMES_CURVE=[1, 2, 4], HARD_STOP_MIN=60.0, FINAL_STOP_MIN=70.0, EVAL_RESERVE_MIN=10.0, EXTRACT_BUDGET_MIN=20.0)
for _k in ("HARD_STOP_MIN", "FINAL_STOP_MIN", "EVAL_RESERVE_MIN", "EXTRACT_BUDGET_MIN", "N_BOOT"):        # test hooks
    if os.environ.get("FFB_" + _k):
        CFG[_k] = type(CFG[_k])(float(os.environ["FFB_" + _k]))
if os.environ.get("FFB_CFG_JSON"):
    CFG.update(json.loads(os.environ["FFB_CFG_JSON"]))
if CFG["LOADER_WORKERS"] is None:
    CFG["LOADER_WORKERS"] = int(max(0 if DRY_RUN else 1, min(4, (N_CPU - 1) // max(N_WORKERS, 1))))

def config_methods(name):
    """FF++ manipulation methods a training configuration sees (in training and in validation)."""
    if name in ("main4", "ffdfd"):
        return list(CLASSIC)
    if name.startswith("lomo_"):
        return [m for m in CLASSIC if m != name[5:]]
    raise ValueError(name)

CONFIG_DOC = {
    "main4": "FF++ c23 train split: originals + Deepfakes, Face2Face, FaceSwap, NeuralTextures",
    "ffdfd": "main4 + the DFD training actors (c23), originals and fakes",
    **{f"lomo_{m}": f"main4 without {m}: {m} is held out of training and validation" for m in CLASSIC},
}
DETECTORS = {"effnet": "EfficientNet-B4, fine-tuned" if not DRY_RUN else "EfficientNet-B0 (random init, rehearsal)",
             "clip": "CLIP ViT-L/14, frozen + logistic probe" if not DRY_RUN else "tiny random CLIP + logistic probe (rehearsal)"}

PROTOCOL = dict(
    title="Face-forgery detection on FaceForensics++ and DFD: benchmark protocol",
    labels="1 = manipulated (fake), 0 = real; scores are logits of 'fake'",
    splits=dict(ffpp="official FF++ split (720/140/140 source videos); a manipulated video A_B follows its pair", ffpp_splits_sha256=FFPP_SPLITS_SHA256,
                dfd=f"actor-disjoint halves (share {CFG['DFD_TRAIN_ACTOR_SHARE']}) drawn with seed {CFG['SEED']}; fakes of two actors from different halves are not used for the DFD-train/DFD-test comparison"),
    configs={k: CONFIG_DOC[k] for k in CFG["CONFIGS"]},
    detectors=DETECTORS, effnet=CFG["EFFNET"], seeds=CFG["SEEDS"], clip=CFG["CLIP"], probe_c_grid=CFG["PROBE_C_GRID"],
    frames=CFG["FRAMES"], decode_window=CFG["DECODE_WINDOW"], crop=dict(size=CFG["CROP_SIZE"], scale=CFG["CROP_SCALE"], detector="YuNet (OpenCV); the largest face, tracked by overlap; in a frame with several faces, the face that differs clearly most from the counterpart video (the original of a manipulated video, or a manipulated version of an original)"),
    model_selection=f"chosen on the validation split of the configuration, by video-level AUC: the EfficientNet epoch (a tie within 1e-4 goes to the lower class-balanced validation loss; patience {CFG['EFFNET']['patience']}) and the probe's C (a tie goes to the smaller C); test data are never used for any choice",
    threshold="decision threshold for accuracy-type metrics: maximises balanced accuracy on the validation videos, then fixed",
    video_score="mean of the frame logits of a video's valid face crops",
    leakage_rule="a (run, test set) pair is not evaluated if any of its videos (at any compression) was used for training or validation of that run",
    metrics=["video AUC (primary)", "frame AUC", "AP", "EER", "accuracy / balanced accuracy / TPR / TNR at the validation threshold", f"ECE ({CFG['ECE_BINS']} bins)"],
    intervals=f"{int(CFG['CI_LEVEL'] * 100)}% percentile intervals from {CFG['N_BOOT']} bootstrap resamples of the test videos by cluster (FF++: source-video pairs; "
              "DFD: two-way, by both actors of a video); for EfficientNet the training seeds are resampled as well (hierarchical bootstrap), and intervals "
              "with the trained models held fixed are reported alongside; comparisons are not corrected for multiplicity",
    mode=MODE,
)
_proto_bytes = json.dumps(PROTOCOL, sort_keys=True, default=str).encode()
PROTOCOL_SHA = hashlib.sha256(_proto_bytes).hexdigest()
with open(os.path.join(DIRS["metadata"], "protocol.json"), "w") as f:
    json.dump(dict(PROTOCOL, sha256=PROTOCOL_SHA, written_utc=RUN["started_utc"]), f, indent=1, default=str)
RUN["protocol_sha256"] = PROTOCOL_SHA
log(f"protocol written (sha256 {PROTOCOL_SHA[:16]}...) | configurations: {', '.join(CFG['CONFIGS'])} | seeds: {CFG['SEEDS']}")

## 3. Libraries

Three small modules are written to `ffb_src/` and imported, so that the CPU processes that extract faces and the GPU worker processes run exactly the same code as the notebook:

* `ffb_video.py`: frame sampling, YuNet face detection, face selection and square cropping;
* `ffb_metrics.py`: AUC (also for 2,000 bootstrap weightings at once), ROC, EER, average precision, threshold choice, calibration error, cluster bootstrap;
* `ffb_worker.py`: the GPU worker that computes CLIP embeddings and fine-tunes EfficientNet.

The next code cell checks a few of their results against scikit-learn and against values known by construction.

In [ ]:
%%writefile ffb_src/ffb_video.py
# 3a. library module (written to ffb_src/ and imported below)
"""Video decoding, face detection and face cropping for the FF++ / DFD benchmark.

Everything here is plain numpy + OpenCV so that it can run in CPU worker processes.
Workers write their crops straight into a shared .npy memmap at offsets that the
notebook assigns in advance, and return only small metadata rows.
"""
import os
import time

import numpy as np
import cv2

YUNET_FILE = "face_detection_yunet_2023mar.onnx"
YUNET_SHA256 = "8f2383e4dd3cfbb4553ea8718107fc0423210dc964f9f4280604804ed2552fa4"   # Git LFS object id in opencv/opencv_zoo
YUNET_URLS = [
    "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
]


# ----------------------------------------------------------------------------- frames
def sample_indices(n_frames, k, window):
    """k frame indices spread evenly over the first `window` frames (all frames if the video is shorter).

    The centre of each of k equal bins is taken, so the first and the last frame are not favoured.
    If the frame count is unknown (0 or negative), the window itself is used and a short video
    simply yields fewer frames.
    """
    k = int(k)
    if k <= 0:
        return []
    n = int(n_frames) if n_frames and n_frames > 0 else int(window)
    w = max(1, min(n, int(window)))
    if k >= w:
        return list(range(w))
    idx = np.floor((np.arange(k) + 0.5) * w / k).astype(int)
    return sorted(set(int(i) for i in np.clip(idx, 0, w - 1)))


def open_video(path):
    """VideoCapture with FFmpeg's decoder limited to one thread when this OpenCV supports it: several videos are
    decoded in parallel processes, one per CPU core, so threads inside each decoder would only compete."""
    if hasattr(cv2, "CAP_PROP_N_THREADS") and hasattr(cv2, "CAP_FFMPEG"):
        try:
            cap = cv2.VideoCapture(path, cv2.CAP_FFMPEG, [cv2.CAP_PROP_N_THREADS, 1])
            if cap.isOpened():
                return cap
        except Exception:
            pass
    return cv2.VideoCapture(path)


def video_info(cap):
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    wh = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0))
    return n, fps, wh


def frame_count(path):
    cap = open_video(path)
    if not cap.isOpened():
        return -1, 0.0, (0, 0)
    out = video_info(cap)
    cap.release()
    return out


def read_frames(path, indices, cap=None):
    """Decode a video sequentially and keep the frames at `indices` (BGR uint8).

    grab() advances without colour conversion; retrieve() is called only for the wanted frames.
    `cap`: an already opened capture positioned at the first frame (it is released here).
    Returns (list of (index, frame), number of frames decoded, error text).
    """
    if not indices:
        if cap is not None:
            cap.release()
        return [], 0, "no frames requested"
    cap = cap if cap is not None else open_video(path)
    if not cap.isOpened():
        return [], 0, "cannot open"
    want = set(int(i) for i in indices)
    last = max(want)
    out, i = [], 0
    try:
        while i <= last:
            if not cap.grab():
                break
            if i in want:
                ok, fr = cap.retrieve()
                if ok and fr is not None and fr.size:
                    if fr.ndim == 2:
                        fr = cv2.cvtColor(fr, cv2.COLOR_GRAY2BGR)
                    out.append((i, fr))
            i += 1
    finally:
        cap.release()
    return out, i, "" if out else "no frame could be decoded"


# ----------------------------------------------------------------------------- faces
class FaceDetector:
    """YuNet (OpenCV FaceDetectorYN) or, as a fallback, OpenCV's Haar cascade.

    detect() returns an (n, 5) float array of boxes (x, y, w, h, score) in the coordinates of the
    image passed in. Detection runs on a copy whose longer side is at most `max_side` pixels.
    """

    def __init__(self, kind="yunet", model_path=None, score_threshold=0.6, max_side=640):
        self.kind, self.max_side, self.score_threshold = kind, int(max_side), float(score_threshold)
        if kind == "yunet":
            if not model_path or not os.path.exists(model_path):
                raise FileNotFoundError(f"YuNet model not found: {model_path}")
            self.det = cv2.FaceDetectorYN.create(model_path, "", (320, 320), self.score_threshold, 0.3, 50)
        elif kind == "haar":
            path = os.path.join(cv2.data.haarcascades, "haarcascade_frontalface_default.xml")
            self.det = cv2.CascadeClassifier(path)
            if self.det.empty():
                raise RuntimeError("OpenCV Haar cascade could not be loaded")
        else:
            raise ValueError(kind)

    def detect(self, img):
        H, W = img.shape[:2]
        s = min(1.0, self.max_side / float(max(H, W)))
        small = cv2.resize(img, (max(1, int(round(W * s))), max(1, int(round(H * s)))), interpolation=cv2.INTER_AREA) if s < 1.0 else img
        if self.kind == "yunet":
            self.det.setInputSize((small.shape[1], small.shape[0]))
            _, faces = self.det.detect(small)
            if faces is None or len(faces) == 0:
                return np.zeros((0, 5), np.float32)
            boxes = np.asarray(faces, np.float32)[:, [0, 1, 2, 3, 14]].copy()
        else:
            gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
            rects = self.det.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(24, 24))
            if len(rects) == 0:
                return np.zeros((0, 5), np.float32)
            boxes = np.concatenate([np.asarray(rects, np.float32), np.ones((len(rects), 1), np.float32)], 1)
        boxes[:, :4] /= s
        keep = (boxes[:, 2] > 4) & (boxes[:, 3] > 4)
        return boxes[keep]


def iou(boxes, ref):
    """IoU of each (x, y, w, h) box with one reference box."""
    boxes = np.asarray(boxes, np.float64).reshape(-1, 4)
    x1 = np.maximum(boxes[:, 0], ref[0]); y1 = np.maximum(boxes[:, 1], ref[1])
    x2 = np.minimum(boxes[:, 0] + boxes[:, 2], ref[0] + ref[2]); y2 = np.minimum(boxes[:, 1] + boxes[:, 3], ref[1] + ref[3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    union = boxes[:, 2] * boxes[:, 3] + ref[2] * ref[3] - inter
    return inter / np.maximum(union, 1e-9)


def choose_faces(dets, forced=None):
    """One face per frame. Returns a list of (box or None, how) with how in {'detected', 'guided', 'neighbour', 'none'}.

    Without guidance: the largest face of the first frame that has one, then in each frame the detection that overlaps
    the previously chosen face most (IoU > 0.3), else again the largest.
    With guidance (forced[j] = index of the manipulated face in frame j, found by guided_choice): those frames take
    that face; every other frame takes the detection that overlaps the nearest guided face (IoU > 0.1), or none.
    Frames left without a face borrow the box of the nearest frame that has one ('neighbour').
    """
    forced = list(forced) if forced is not None else [None] * len(dets)
    anchors = {j: np.asarray(dets[j][f, :4], np.float64) for j, f in enumerate(forced) if f is not None}
    chosen, prev = [], None
    for j, b in enumerate(dets):
        if len(b) == 0:
            chosen.append(None)
            continue
        if forced[j] is not None:
            k, how = int(forced[j]), "guided"
        elif anchors:
            jn = min(anchors, key=lambda a: (abs(a - j), a))
            ov = iou(b[:, :4], anchors[jn])
            if ov.max() <= 0.1:                       # the manipulated face was not detected in this frame
                chosen.append(None)
                continue
            k, how = int(np.argmax(ov)), "detected"
        else:
            k = None
            if prev is not None:
                ov = iou(b[:, :4], prev)
                if ov.max() > 0.3:
                    k = int(np.argmax(ov))
            if k is None:
                k = int(np.argmax(b[:, 2] * b[:, 3]))
            how = "detected"
        prev = np.asarray(b[k, :4], np.float64)
        chosen.append((np.asarray(b[k], np.float64), how))
    have = [i for i, c in enumerate(chosen) if c is not None]
    out = []
    for i, c in enumerate(chosen):
        if c is not None:
            out.append(c)
        elif have:
            nb = min(have, key=lambda h: (abs(h - i), h))
            out.append((chosen[nb][0], "neighbour"))
        else:
            out.append((None, "none"))
    return out


def box_difference(fr, ref, box):
    """Mean absolute difference of blurred grey levels inside a face box, between a frame and the same frame of the
    counterpart video (the original of a manipulated video, or a manipulated version of an original)."""
    H, W = fr.shape[:2]
    if ref.shape[:2] != (H, W):
        ref = cv2.resize(ref, (W, H), interpolation=cv2.INTER_AREA)
    x, y, w, h = [int(round(float(v))) for v in box[:4]]
    x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return 0.0
    a = cv2.GaussianBlur(cv2.cvtColor(np.ascontiguousarray(fr[y0:y1, x0:x1]), cv2.COLOR_BGR2GRAY), (5, 5), 0).astype(np.float32)
    b = cv2.GaussianBlur(cv2.cvtColor(np.ascontiguousarray(ref[y0:y1, x0:x1]), cv2.COLOR_BGR2GRAY), (5, 5), 0).astype(np.float32)
    return float(np.mean(np.abs(a - b)))


def guided_choice(dets, frames, ref_frames, ratio=1.5, min_diff=3.0):
    """For each frame with two or more faces: the index of the face that differs clearly most from the counterpart
    video at the same frame index (the manipulated face), or None when the difference is not clear (for example when
    the two videos are not frame-aligned) or the counterpart frame is missing."""
    out = []
    for b, (i, fr) in zip(dets, frames):
        if len(b) < 2 or i not in ref_frames:
            out.append(None)
            continue
        d = np.array([box_difference(fr, ref_frames[i], bb) for bb in b])
        order = np.argsort(-d)
        out.append(int(order[0]) if (d[order[0]] >= min_diff and d[order[0]] >= ratio * max(d[order[1]], 1e-6)) else None)
    return out


def square_crop(img, box, scale=1.3, out_size=224):
    """Square crop of side scale * max(w, h) centred on the face box, zero-padded where it leaves the frame,
    resized to out_size and returned as RGB uint8."""
    x, y, w, h = [float(v) for v in box[:4]]
    cx, cy = x + w / 2.0, y + h / 2.0
    s = max(2, int(round(max(w, h) * float(scale))))
    x0, y0 = int(round(cx - s / 2.0)), int(round(cy - s / 2.0))
    H, W = img.shape[:2]
    sub = img[max(0, y0):min(H, y0 + s), max(0, x0):min(W, x0 + s)]
    pt, pl = max(0, -y0), max(0, -x0)
    pb, pr = max(0, y0 + s - H), max(0, x0 + s - W)
    if sub.size == 0:
        sub = np.zeros((s, s, 3), np.uint8)
    elif pt or pl or pb or pr:
        sub = cv2.copyMakeBorder(sub, pt, pb, pl, pr, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    if sub.shape[0] != s or sub.shape[1] != s:                       # rounding at the borders
        sub = cv2.resize(sub, (s, s), interpolation=cv2.INTER_LINEAR)
    interp = cv2.INTER_AREA if s > out_size else cv2.INTER_CUBIC
    crop = cv2.resize(sub, (int(out_size), int(out_size)), interpolation=interp)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


# ----------------------------------------------------------------------------- worker processes
_W = {}


def init_worker(cfg):
    """Called once in every extraction process. cfg: detector settings and the memmap registry."""
    cv2.setNumThreads(1)
    _W.clear()
    _W["cfg"] = cfg
    _W["det"] = FaceDetector(cfg["detector"], cfg.get("model_path"), cfg.get("score_threshold", 0.6), cfg.get("max_side", 640))
    _W["mm"] = {}


def _memmap(name):
    if name not in _W["mm"]:
        _W["mm"][name] = np.load(_W["cfg"]["memmaps"][name], mmap_mode="r+")
    return _W["mm"][name]


def process_video(task):
    """Decode the planned frames of one video, find one face per frame, write the crops into the memmap.

    task: dict(uid, path, set, offset, slots, window). Returns dict(uid, n_frames, fps, width, height,
    n_decoded, rows, seconds, error); rows are per slot: slot, frame, face ('detected'/'neighbour'/'none'/
    'no frame'), score and the box.
    """
    t0 = time.time()
    cfg = _W["cfg"]
    out = dict(uid=task["uid"], n_frames=-1, fps=0.0, width=0, height=0, n_decoded=0, rows=[], seconds=0.0, error="",
               n_multi_face_frames=0, ref_used=False, n_guided=0)
    try:
        hook = os.environ.get("FFB_EXTRACT_CRASH", "")              # test hook: an extraction process dies on this video (once)
        if hook and hook in task["uid"] and not os.path.exists(hook_flag := os.path.join(os.path.dirname(_W["cfg"]["memmaps"][task["set"]]), "crash_hook_used")):
            open(hook_flag, "w").close()
            os._exit(11)
        slow = float(os.environ.get("FFB_EXTRACT_SLOW", "0") or 0)    # test hook: every video takes this many seconds longer
        if slow > 0:
            time.sleep(slow)
        cap = open_video(task["path"])
        if not cap.isOpened():
            out["error"] = "cannot open"
            return out
        n, fps, (w, h) = video_info(cap)
        out.update(n_frames=n, fps=fps, width=w, height=h)
        idx = sample_indices(n, task["slots"], task["window"])
        frames, n_dec, err = read_frames(task["path"], idx, cap=cap)
        out["n_decoded"] = n_dec
        if err and not frames:
            out["error"] = err
        det = _W["det"]
        dets = [det.detect(fr) for _, fr in frames]
        forced = None
        multi = [i for (i, _), bx in zip(frames, dets) if len(bx) >= 2]
        out["n_multi_face_frames"] = len(multi)
        if multi and task.get("ref_path"):
            ref_list, _, _ = read_frames(task["ref_path"], multi)
            if ref_list:
                forced = guided_choice(dets, frames, dict(ref_list))
                out["ref_used"] = True
        chosen = choose_faces(dets, forced)
        out["n_guided"] = sum(1 for _, h in chosen if h == "guided")
        mm = _memmap(task["set"])
        size = int(cfg["crop_size"])
        got = {i: (fr, c, len(bx)) for (i, fr), c, bx in zip(frames, chosen, dets)}
        for slot, fi in enumerate(idx[: task["slots"]]):
            row = dict(slot=slot, frame=int(fi), face="no frame", n_faces=0, score=float("nan"), x=float("nan"), y=float("nan"), w=float("nan"), h=float("nan"))
            if fi in got:
                fr, (box, how), nf = got[fi]
                row["face"] = how
                row["n_faces"] = int(nf)
                if box is not None:
                    mm[task["offset"] + slot] = square_crop(fr, box, cfg.get("crop_scale", 1.3), size)
                    row.update(score=float(box[4]), x=float(box[0]), y=float(box[1]), w=float(box[2]), h=float(box[3]))
            out["rows"].append(row)
    except Exception as e:                                   # one broken video must not stop the pool
        out["error"] = f"{type(e).__name__}: {e}"
    out["seconds"] = time.time() - t0
    return out


def flush_worker(*_):
    for mm in _W.get("mm", {}).values():
        try:
            mm.flush()
        except Exception:
            pass
    return True

In [ ]:
%%writefile ffb_src/ffb_metrics.py
# 3b. library module (written to ffb_src/ and imported below)
"""Metrics for the FF++ / DFD benchmark: AUC, AP, EER, thresholds, calibration, cluster bootstrap.

Convention: label 1 = fake (manipulated), 0 = real; a higher score means "more likely fake".
"""
import numpy as np


def _clean(y, s):
    y = np.asarray(y).astype(np.int64).ravel()
    s = np.asarray(s, dtype=np.float64).ravel()
    ok = np.isfinite(s)
    return y[ok], s[ok]


def auc(y, s):
    """Area under the ROC curve (Mann-Whitney U with ties counted as one half). NaN if a class is missing."""
    y, s = _clean(y, s)
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    return float(weighted_auc(y, s, np.ones((1, len(y))))[0])


def weighted_auc(y, s, W):
    """AUC for many weightings of the same scored items at once.

    W: (B, n) non-negative weights (for a bootstrap, how often each item was drawn). Ties count one half.
    O(B * n) after a single sort, so 2,000 bootstrap resamples of a few thousand videos take well under a second.
    """
    y = np.asarray(y).astype(np.float64).ravel()
    s = np.asarray(s, dtype=np.float64).ravel()
    W = np.atleast_2d(np.asarray(W, dtype=np.float64))
    order = np.argsort(s, kind="mergesort")
    s, y, W = s[order], y[order], W[:, order]
    _, start = np.unique(s, return_index=True)
    P = np.add.reduceat(W * y, start, axis=1)
    N = np.add.reduceat(W * (1.0 - y), start, axis=1)
    below = np.cumsum(N, axis=1) - N
    num = (P * (below + 0.5 * N)).sum(1)
    den = P.sum(1) * N.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def roc_curve(y, s):
    """(fpr, tpr, thresholds) with one point per distinct score, starting at (0, 0)."""
    y, s = _clean(y, s)
    order = np.argsort(-s, kind="mergesort")
    s, y = s[order], y[order]
    distinct = np.r_[np.where(np.diff(s))[0], len(s) - 1]
    tp = np.cumsum(y)[distinct]; fp = (distinct + 1) - tp
    P, N = max(int(y.sum()), 1), max(len(y) - int(y.sum()), 1)
    fpr = np.r_[0.0, fp / N]; tpr = np.r_[0.0, tp / P]
    thr = np.r_[np.inf, s[distinct]]
    return fpr, tpr, thr


def eer(y, s):
    """Equal error rate: where the false positive rate equals the false negative rate (linear interpolation)."""
    y, s = _clean(y, s)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan")
    fpr, tpr, _ = roc_curve(y, s)
    fnr = 1.0 - tpr
    d = fpr - fnr
    i = int(np.argmax(d >= 0))
    if i == 0:
        return float(fpr[0])
    a, b = d[i - 1], d[i]
    t = 0.0 if b == a else -a / (b - a)
    return float(fpr[i - 1] + t * (fpr[i] - fpr[i - 1]))


def average_precision(y, s):
    """Average precision (area under the step precision-recall curve), fake as the positive class."""
    y, s = _clean(y, s)
    P = int(y.sum())
    if P == 0:
        return float("nan")
    order = np.argsort(-s, kind="mergesort")
    s, y = s[order], y[order]
    distinct = np.r_[np.where(np.diff(s))[0], len(s) - 1]
    tp = np.cumsum(y)[distinct]
    prec = tp / (distinct + 1.0)
    rec = tp / P
    return float(np.sum(np.diff(np.r_[0.0, rec]) * prec))


def best_threshold(y, s):
    """Threshold that maximises balanced accuracy (equivalently Youden's J) on (y, s): used on VALIDATION scores only."""
    y, s = _clean(y, s)
    fpr, tpr, thr = roc_curve(y, s)
    j = tpr - fpr
    k = int(np.argmax(j))
    if not np.isfinite(thr[k]):
        return float(np.max(s) + 1e-6)
    # midpoint between this score and the next lower distinct score, so ties do not decide the side
    lower = s[s < thr[k]]
    return float((thr[k] + lower.max()) / 2.0) if len(lower) else float(thr[k] - 1e-6)


def rates_at(y, s, t):
    """Accuracy, balanced accuracy, true positive rate (fakes caught) and true negative rate (reals passed) at threshold t (score >= t means fake)."""
    y, s = _clean(y, s)
    pred = s >= t
    tpr = float(np.mean(pred[y == 1])) if (y == 1).any() else float("nan")
    tnr = float(np.mean(~pred[y == 0])) if (y == 0).any() else float("nan")
    return dict(accuracy=float(np.mean(pred == (y == 1))), balanced_accuracy=float(np.nanmean([tpr, tnr])), tpr=tpr, tnr=tnr)


def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    return 0.5 * (1.0 + np.tanh(0.5 * z))


def ece(y, p, n_bins=15):
    """Expected calibration error of the probability of 'fake' (equal-width bins, weighted by bin size)."""
    y = np.asarray(y, dtype=np.float64).ravel(); p = np.asarray(p, dtype=np.float64).ravel()
    ok = np.isfinite(p); y, p = y[ok], p[ok]
    if len(y) == 0:
        return float("nan")
    b = np.minimum((p * n_bins).astype(int), n_bins - 1)
    tot = 0.0
    for k in range(n_bins):
        m = b == k
        if m.any():
            tot += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(tot)


def reliability_bins(y, p, n_bins=10):
    y = np.asarray(y, dtype=np.float64).ravel(); p = np.asarray(p, dtype=np.float64).ravel()
    b = np.minimum((p * n_bins).astype(int), n_bins - 1)
    rows = []
    for k in range(n_bins):
        m = b == k
        rows.append(dict(bin=k, lo=k / n_bins, hi=(k + 1) / n_bins, n=int(m.sum()),
                         mean_p=float(p[m].mean()) if m.any() else float("nan"), frac_fake=float(y[m].mean()) if m.any() else float("nan")))
    return rows


def video_scores(frame_scores, video_index, n_videos, valid=None):
    """Mean of the valid frame scores of each video (NaN for a video without a valid frame)."""
    s = np.asarray(frame_scores, dtype=np.float64)
    v = np.asarray(video_index, dtype=np.int64)
    ok = np.isfinite(s) if valid is None else (np.asarray(valid, bool) & np.isfinite(s))
    tot = np.bincount(v[ok], weights=s[ok], minlength=n_videos)
    cnt = np.bincount(v[ok], minlength=n_videos)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(cnt > 0, tot / np.maximum(cnt, 1), np.nan), cnt


def cluster_bootstrap_weights(clusters, n_boot=2000, seed=0):
    """(n_boot, n) weights: in each resample the clusters are drawn with replacement and every item of a
    drawn cluster gets that cluster's draw count. Items of one cluster (a source pair of FF++, an actor of DFD)
    therefore move together, which respects their dependence."""
    clusters = np.asarray(clusters)
    uniq, inv = np.unique(clusters, return_inverse=True)
    K = len(uniq)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, K, size=(n_boot, K))
    counts = np.zeros((n_boot, K), np.float64)
    np.add.at(counts, (np.repeat(np.arange(n_boot), K), draws.ravel()), 1.0)
    return counts[:, inv]


def multiway_bootstrap_weights(keys, n_boot=2000, seed=0):
    """(n_boot, n) weights for items that belong to one or two random factors at once.

    keys: one tuple per item, of one or two factor levels (for DFD: (actor,) for an original, (actor_a, actor_b) for a
    fake made from two actors). In each resample the levels are drawn with replacement, and an item's weight is the
    product of the draw counts of its levels (the "pigeonhole" bootstrap for crossed random effects, Owen 2007).
    With one level per item this is the ordinary cluster bootstrap.
    """
    keys = [tuple(k) for k in keys]
    levels = sorted(set(x for k in keys for x in k))
    pos = {x: i for i, x in enumerate(levels)}
    K = len(levels)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, K, size=(n_boot, K))
    counts = np.zeros((n_boot, K), np.float64)
    np.add.at(counts, (np.repeat(np.arange(n_boot), K), draws.ravel()), 1.0)
    k1 = np.array([pos[k[0]] for k in keys])
    k2 = np.array([pos[k[1]] if len(k) > 1 and k[1] != k[0] else -1 for k in keys])
    W = counts[:, k1].copy()
    two = k2 >= 0
    if two.any():
        # two different levels drawn from the same K levels: E[n_a * n_b] = (K - 1) / K, rescaled to 1
        W[:, two] *= counts[:, k2[two]] * (K / max(K - 1.0, 1.0))
    return W


def hierarchical_mean(mat, draws=None, seed=0):
    """Bootstrap over training seeds on top of a test-set bootstrap.

    mat: (S, B) array, the statistic of S trained models in B test resamples (the same resamples for every model).
    In resample b, S models are drawn with replacement and their values averaged. draws: (B, S) integer array to share
    the seed draw between two quantities computed from the same models. Returns (vector of length B, draws used)."""
    mat = np.atleast_2d(np.asarray(mat, np.float64))
    S, B = mat.shape
    if S == 1:
        return mat[0].copy(), None
    if draws is None:
        draws = np.random.default_rng(seed).integers(0, S, size=(B, S))
    return mat[draws, np.arange(B)[:, None]].mean(axis=1), draws


def percentile_interval(values, level=0.95):
    v = np.asarray(values, dtype=np.float64)
    v = v[np.isfinite(v)]
    if len(v) < 10:
        return float("nan"), float("nan")
    a = (1.0 - level) / 2.0
    return float(np.quantile(v, a)), float(np.quantile(v, 1.0 - a))


def summary(y, s, threshold=None):
    """All point metrics of one score vector (video or frame level)."""
    y, s = _clean(y, s)
    out = dict(n=int(len(y)), n_fake=int(y.sum()), n_real=int(len(y) - y.sum()), auc=auc(y, s), ap=average_precision(y, s), eer=eer(y, s))
    if threshold is not None and np.isfinite(threshold):
        r = rates_at(y, s, threshold)
        out.update({k + "_at_val_threshold": v for k, v in r.items()})
    return out

In [ ]:
%%writefile ffb_src/ffb_worker.py
# 3c. library module (written to ffb_src/ and imported below)
"""GPU worker for the FF++ / DFD benchmark.

The notebook starts one of these per GPU:  python ffb_worker.py --state DIR --worker 0 --device cuda
Each worker takes jobs from DIR/queue/pending (an atomic rename claims a job), runs them and writes the
results to DIR/results/<job_id>/. Two job types:

  clip_features  frozen CLIP image embeddings of the rows of one crop set
  train_effnet   fine-tune an EfficientNet on the given training rows, select the epoch on the validation
                 rows (video-level AUC), then score the test rows with the selected weights

A worker stops starting new jobs when the estimated end of the next job would pass the deadline in
DIR/worker_config.json, stops a running job at the deadline or when DIR/STOP exists, and exits when
the queue is empty. Nothing in here ever reads a test label.
"""
import argparse, json, os, sys, time, traceback, math, random

import numpy as np


def now():
    return time.time()


class Ctx:
    def __init__(self, state, worker, device):
        self.state, self.worker, self.device = state, int(worker), device
        self.cfg = json.load(open(os.path.join(state, "worker_config.json")))
        self.log_path = os.path.join(state, "logs", f"worker{self.worker}.log")
        os.makedirs(os.path.dirname(self.log_path), exist_ok=True)
        self.rates_path = os.path.join(state, f"worker{self.worker}_rates.json")
        self.rates = json.load(open(self.rates_path)) if os.path.exists(self.rates_path) else {}

    def log(self, *a):
        line = f"[{time.strftime('%H:%M:%S')} w{self.worker}] " + " ".join(str(x) for x in a)
        print(line, flush=True)
        with open(self.log_path, "a") as f:
            f.write(line + "\n")

    def stop_requested(self):
        return os.path.exists(os.path.join(self.state, "STOP")) or now() > float(self.cfg["deadline_epoch"])

    def save_rates(self):
        json.dump(self.rates, open(self.rates_path, "w"))


# ----------------------------------------------------------------------------- data
class CropSet:
    """Read-only view of the crop memmaps (opened lazily, also inside DataLoader worker processes)."""
    def __init__(self, registry):
        self.registry = registry
        self.mm = {}

    def get(self, name):
        if name not in self.mm:
            self.mm[name] = np.load(self.registry[name]["path"], mmap_mode="r")
        return self.mm[name]


def augment(img, rng):
    """Training augmentation in the spirit of DeepfakeBench's defaults: flip, small rotation, blur,
    brightness/contrast and JPEG re-compression, each with probability 0.5. img: HWC RGB uint8."""
    import cv2
    h, w = img.shape[:2]
    if rng.random() < 0.5:
        img = img[:, ::-1]
    if rng.random() < 0.5:
        a = rng.uniform(-10, 10)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        img = cv2.warpAffine(np.ascontiguousarray(img), M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    if rng.random() < 0.5:
        k = int(rng.choice([3, 5, 7]))
        img = cv2.GaussianBlur(np.ascontiguousarray(img), (k, k), 0)
    if rng.random() < 0.5:
        alpha = 1.0 + rng.uniform(-0.1, 0.1); beta = 255.0 * rng.uniform(-0.1, 0.1)
        img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    if rng.random() < 0.5:
        q = int(rng.integers(40, 101))
        ok, enc = cv2.imencode(".jpg", np.ascontiguousarray(img)[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), q])
        if ok:
            img = cv2.imdecode(enc, cv2.IMREAD_COLOR)[:, :, ::-1]
    return np.ascontiguousarray(img)


class CropDataset:
    """Crops from the memmaps by (set, row). Defined at module level so that DataLoader workers can be started by any method."""
    def __init__(self, registry, set_names, set_ids, rows, labels=None, train=False):
        self.registry, self.set_names, self.train = registry, list(set_names), bool(train)
        self.set_ids = np.asarray(set_ids, np.int64); self.rows = np.asarray(rows, np.int64)
        self.labels = None if labels is None else np.asarray(labels, np.float32)
        self.cs = None

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        import torch
        if self.cs is None:
            self.cs = CropSet(self.registry)
        img = np.asarray(self.cs.get(self.set_names[self.set_ids[i]])[self.rows[i]])
        if self.train:                                # torch's generator is reseeded per worker and per epoch, so augmentations differ between epochs
            img = augment(img, np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,)).item())))
        y = -1.0 if self.labels is None else float(self.labels[i])
        return torch.from_numpy(np.ascontiguousarray(img)), torch.tensor(y)


def make_dataset(registry, set_names, set_ids, rows, labels=None, train=False, seed=0):
    return CropDataset(registry, set_names, set_ids, rows, labels, train)


def loader(ds, batch, shuffle, n_workers, seed, drop_last=False, persistent=False):
    import torch, multiprocessing as mp
    g = torch.Generator(); g.manual_seed(int(seed))
    kw = dict(batch_size=int(batch), shuffle=shuffle, num_workers=int(n_workers), pin_memory=torch.cuda.is_available(), drop_last=drop_last, generator=g)
    if n_workers > 0:
        kw.update(prefetch_factor=4, persistent_workers=bool(persistent))
        if "fork" in mp.get_all_start_methods():       # Linux: fork is cheap and does not need to pickle the dataset
            kw["multiprocessing_context"] = "fork"
    return torch.utils.data.DataLoader(ds, **kw)


def to_input(x, dev, mean, std, channels_last):
    """uint8 BHWC -> normalised float BCHW on the device."""
    import torch
    x = x.to(dev, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - mean) / std
    return x.contiguous(memory_format=torch.channels_last) if channels_last else x


# ----------------------------------------------------------------------------- jobs
def predict(model, dl, dev, mean, std, amp, channels_last, ctx=None):
    import torch
    out = []
    model.eval()
    with torch.no_grad():
        for x, _ in dl:
            if ctx is not None and ctx.stop_requested():
                raise TimeoutError("stopped at the deadline or on request")
            xb = to_input(x, dev, mean, std, channels_last)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
                z = model(xb)
            out.append(z.float().reshape(-1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, np.float32)


def video_auc(z, labels, video):
    """AUC of video scores (mean of frame logits); labels and video index per frame."""
    v = np.asarray(video); n = int(v.max()) + 1 if len(v) else 0
    tot = np.bincount(v, weights=z, minlength=n); cnt = np.bincount(v, minlength=n)
    lab = np.zeros(n); np.maximum.at(lab, v, labels)
    keep = cnt > 0
    s, y = tot[keep] / cnt[keep], lab[keep]
    if y.min() == y.max():
        return float("nan")
    order = np.argsort(s, kind="mergesort"); s, y = s[order], y[order]
    _, start = np.unique(s, return_index=True)
    P = np.add.reduceat(y, start); N = np.add.reduceat(1 - y, start)
    below = np.cumsum(N) - N
    return float((P * (below + 0.5 * N)).sum() / (P.sum() * N.sum()))


def balanced_bce(z, y):
    """Class-balanced binary cross-entropy of logits z (the mean over fakes and the mean over reals, averaged)."""
    z = np.asarray(z, np.float64).ravel(); y = np.asarray(y).ravel()
    ok = np.isfinite(z); z, y = z[ok], y[ok]
    parts = [np.logaddexp(0.0, -z[y == 1]).mean()] if (y == 1).any() else []
    parts += [np.logaddexp(0.0, z[y == 0]).mean()] if (y == 0).any() else []
    return float(np.mean(parts)) if parts else float("nan")


def run_train(ctx, job, out_dir):
    import torch, timm
    cfg = ctx.cfg
    d = np.load(os.path.join(ctx.state, "jobs_data", job["job_id"] + ".npz"))
    names = job["set_names"]; reg = cfg["crop_sets"]
    seed = int(job["seed"])
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    dev = torch.device(ctx.device)
    amp = bool(cfg.get("amp", True)) and dev.type == "cuda"
    chl = dev.type == "cuda"
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    m = cfg["effnet"]
    model = timm.create_model(m["arch"], pretrained=bool(m["pretrained"]), num_classes=1)
    dcfg = getattr(model, "pretrained_cfg", {}) or {}
    mean = torch.tensor(dcfg.get("mean", (0.485, 0.456, 0.406)), device=dev).view(1, 3, 1, 1)
    std = torch.tensor(dcfg.get("std", (0.229, 0.224, 0.225)), device=dev).view(1, 3, 1, 1)
    model = model.to(dev)
    if chl:
        model = model.to(memory_format=torch.channels_last)
    ytr = d["train_labels"].astype(np.float32)
    pos_weight = torch.tensor([(ytr == 0).sum() / max((ytr == 1).sum(), 1)], device=dev, dtype=torch.float32)
    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=float(m["lr"]), betas=(0.9, 0.999), eps=1e-8, weight_decay=float(m["weight_decay"]))
    scaler = torch.amp.GradScaler("cuda", enabled=amp) if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=amp)
    nw = int(cfg.get("loader_workers", 2))
    ds_tr = make_dataset(reg, names, d["train_set"], d["train_rows"], ytr, train=True, seed=seed)
    dl_tr = loader(ds_tr, m["batch_size"], True, nw, seed, drop_last=True, persistent=True)
    ds_va = make_dataset(reg, names, d["val_set"], d["val_rows"], d["val_labels"], train=False)
    dl_va = loader(ds_va, m["eval_batch_size"], False, nw, seed, persistent=True)
    best, best_auc, best_loss, best_ep, hist, bad = None, -1.0, float("inf"), -1, [], 0
    t_train = t_eval = 0.0; n_train_seen = n_eval_seen = 0
    max_steps = int(m.get("max_steps_per_epoch") or 0)
    for ep in range(int(m["epochs"])):
        model.train()
        t0 = now(); losses = []
        for step, (x, y) in enumerate(dl_tr):
            if max_steps and step >= max_steps:
                break
            if ctx.stop_requested():
                raise TimeoutError("stopped at the deadline or on request")
            xb = to_input(x, dev, mean, std, chl); yb = y.to(dev, non_blocking=True).float()
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):
                z = model(xb).reshape(-1)
            loss = lossf(z.float(), yb)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite training loss at epoch {ep}, step {step}")
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            losses.append(float(loss.detach()))
            n_train_seen += len(yb)
        t_train += now() - t0
        t1 = now()
        zv = predict(model, dl_va, dev, mean, std, amp, chl, ctx)
        t_eval += now() - t1; n_eval_seen += len(zv)
        auc_v = video_auc(zv, d["val_labels"], d["val_video"])
        loss_v = balanced_bce(zv, d["val_labels"])
        hist.append(dict(epoch=ep + 1, train_loss=float(np.mean(losses)) if losses else float("nan"), val_video_auc=auc_v, val_loss=loss_v,
                         train_seconds=t1 - t0, val_seconds=now() - t1, lr=opt.param_groups[0]["lr"]))
        ctx.log(f"{job['job_id']}: epoch {ep + 1}/{m['epochs']} loss {hist[-1]['train_loss']:.4f} val video AUC {auc_v:.4f} val loss {loss_v:.4f} "
                f"({len(losses) * int(m['batch_size']) / max(hist[-1]['train_seconds'], 1e-9):.0f} img/s)")
        json.dump(dict(job_id=job["job_id"], epoch=ep + 1, of=int(m["epochs"]), val_auc=auc_v, t=now()), open(os.path.join(out_dir, "progress.json"), "w"))
        # the best epoch: highest validation video AUC; a tie (AUC saturates, e.g. at 1.0) goes to the lower validation loss
        if np.isfinite(auc_v) and (auc_v > best_auc + 1e-4 or (abs(auc_v - best_auc) <= 1e-4 and loss_v < best_loss)):
            best_auc, best_loss, best_ep, bad = auc_v, loss_v, ep + 1, 0
            best = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= int(m["patience"]):
                ctx.log(f"{job['job_id']}: early stop after epoch {ep + 1} (best epoch {best_ep})")
                break
    if best is None:                                  # validation AUC was never finite (one class only): keep the last weights
        best = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}; best_ep = len(hist)
    model.load_state_dict(best)
    scores = {"val": predict(model, dl_va, dev, mean, std, amp, chl, ctx)}
    for t in job["tests"]:
        ds = make_dataset(reg, names, d[f"test_{t}_set"], d[f"test_{t}_rows"], None, train=False)
        t1 = now()
        scores["test_" + t] = predict(model, loader(ds, m["eval_batch_size"], False, nw, seed), dev, mean, std, amp, chl, ctx)
        t_eval += now() - t1; n_eval_seen += len(ds)
    rows = {"rows_val": d["val_rows"], **{f"rows_test_{t}": d[f"test_{t}_rows"] for t in job["tests"]}}       # kept with the scores, checked when a later session reuses them
    np.savez_compressed(os.path.join(out_dir, "scores.npz"), **{k: v.astype(np.float32) for k, v in scores.items()}, **rows)
    import csv
    for h in hist:
        h["selected"] = int(h["epoch"] == best_ep)        # the epoch whose weights were kept
    with open(os.path.join(out_dir, "history.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(hist[0].keys())); w.writeheader(); w.writerows(hist)
    if job.get("save_checkpoint"):
        torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in best.items()}, os.path.join(out_dir, "model_best_fp16.pt"))
    r = ctx.rates.setdefault("train_effnet", {})
    if n_train_seen:
        r["sec_per_train_sample"] = t_train / n_train_seen
    if n_eval_seen:
        r["sec_per_eval_sample"] = t_eval / n_eval_seen
    ctx.save_rates()
    return dict(best_epoch=best_ep, best_val_video_auc=best_auc, best_val_loss=best_loss, epochs_run=len(hist), train_seconds=t_train, eval_seconds=t_eval,
                train_images_per_second=n_train_seen / max(t_train, 1e-9), n_train=int(len(ytr)), n_val=int(len(d["val_rows"])))


def run_clip(ctx, job, out_dir):
    import torch
    from transformers import CLIPVisionModelWithProjection
    cfg = ctx.cfg; c = cfg["clip"]
    dev = torch.device(ctx.device)
    dtype = torch.float16 if dev.type == "cuda" else torch.float32
    model = CLIPVisionModelWithProjection.from_pretrained(c["path"]).to(device=dev, dtype=dtype).eval()     # loaded in float32, then cast (no dtype keyword: it changed name between transformers 4 and 5)
    mean = torch.tensor(c["mean"], device=dev).view(1, 3, 1, 1); std = torch.tensor(c["std"], device=dev).view(1, 3, 1, 1)
    reg = cfg["crop_sets"]
    rows = np.arange(int(job["row_start"]), int(job["row_end"]))
    ds = make_dataset(reg, [job["set"]], np.zeros(len(rows), np.int64), rows, None, train=False)
    dl = loader(ds, c["batch_size"], False, int(cfg.get("loader_workers", 2)), 0)
    feats = []; t0 = now()
    size = int(c["image_size"])
    with torch.no_grad():
        for x, _ in dl:
            if ctx.stop_requested():
                raise TimeoutError("stopped at the deadline or on request")
            xb = x.to(dev, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
            if xb.shape[-1] != size:
                xb = torch.nn.functional.interpolate(xb, size=(size, size), mode="bicubic", align_corners=False).clamp_(0, 1)
            xb = ((xb - mean) / std).to(dtype)
            e = model(pixel_values=xb).image_embeds.float()
            e = e / e.norm(dim=-1, keepdim=True).clamp_min(1e-12)
            feats.append(e.cpu().numpy().astype(np.float16))
    F = np.concatenate(feats) if feats else np.zeros((0, 1), np.float16)
    np.save(os.path.join(out_dir, "features.npy"), F)
    secs = now() - t0
    ctx.rates.setdefault("clip_features", {})["sec_per_sample"] = secs / max(len(rows), 1)
    ctx.save_rates()
    return dict(rows=int(len(rows)), dim=int(F.shape[1]), seconds=secs, images_per_second=len(rows) / max(secs, 1e-9))


def estimate_seconds(ctx, job):
    if job["type"] == "train_effnet":
        r = ctx.rates.get("train_effnet", {})
        if "sec_per_train_sample" not in r:
            return None
        n_tr, n_va, n_te = job["n_train"], job["n_val"], job["n_test"]
        steps = job.get("max_steps_per_epoch") or 0
        per_epoch = min(n_tr, steps * job.get("batch_size", 32)) if steps else n_tr
        return job["epochs"] * (per_epoch * r["sec_per_train_sample"] + n_va * r.get("sec_per_eval_sample", r["sec_per_train_sample"] / 3)) + n_te * r.get("sec_per_eval_sample", 0.002)
    if job["type"] == "clip_features":
        r = ctx.rates.get("clip_features", {})
        return None if "sec_per_sample" not in r else (job["row_end"] - job["row_start"]) * r["sec_per_sample"]
    return None


def claim(ctx):
    q = os.path.join(ctx.state, "queue")
    for fn in sorted(os.listdir(os.path.join(q, "pending"))):
        if not fn.endswith(".json"):
            continue
        src = os.path.join(q, "pending", fn)
        try:
            job = json.load(open(src))
        except Exception:
            continue
        if job.get("only_worker") is not None and int(job["only_worker"]) != ctx.worker:
            continue
        est = estimate_seconds(ctx, job)
        if est is not None and now() + 1.1 * est > float(ctx.cfg["deadline_epoch"]):
            ctx.log(f"not starting {job['job_id']}: estimated {est / 60:.1f} min would pass the deadline")
            continue
        dst = os.path.join(q, "running", fn)
        try:
            os.rename(src, dst)                      # atomic: only one worker can win
        except OSError:
            continue
        return job, dst
    return None, None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--state", required=True); ap.add_argument("--worker", type=int, default=0); ap.add_argument("--device", default="cuda")
    a = ap.parse_args()
    ctx = Ctx(a.state, a.worker, a.device)
    if a.device == "cuda":
        import torch
        if not torch.cuda.is_available():
            ctx.log("CUDA requested but not available"); sys.exit(3)
        torch.cuda.set_device(0)                      # CUDA_VISIBLE_DEVICES selects the GPU
    try:
        import torch
        torch.set_num_threads(max(1, int(ctx.cfg.get("torch_threads", 1))))
    except Exception:
        pass
    ctx.log(f"started on {a.device} (pid {os.getpid()})")
    q = os.path.join(ctx.state, "queue")
    while not ctx.stop_requested():
        job, path = claim(ctx)
        if job is None:
            break
        out_dir = os.path.join(ctx.state, "results", job["job_id"]); os.makedirs(out_dir, exist_ok=True)
        json.dump(dict(worker=ctx.worker, pid=os.getpid(), t=now(), queue_file=os.path.basename(path)), open(os.path.join(out_dir, "claim.json"), "w"))
        t0 = now()
        ctx.log(f"job {job['job_id']} ({job['type']}) started")
        hook = os.environ.get("FFB_WORKER_CRASH", "")             # test hook: die abruptly (once) inside a job whose id contains the text
        if hook and hook in job["job_id"] and not os.path.exists(os.path.join(ctx.state, "crash_hook_used")):
            open(os.path.join(ctx.state, "crash_hook_used"), "w").close()
            os._exit(9)
        try:
            res = run_train(ctx, job, out_dir) if job["type"] == "train_effnet" else run_clip(ctx, job, out_dir)
            res.update(job_id=job["job_id"], status="ok", worker=ctx.worker, device=a.device, seconds=now() - t0)
            json.dump(res, open(os.path.join(out_dir, "summary.json"), "w"), indent=1)
            os.replace(path, os.path.join(q, "done", os.path.basename(path)))
            ctx.log(f"job {job['job_id']} done in {(now() - t0) / 60:.1f} min")
        except TimeoutError as e:
            json.dump(dict(job_id=job["job_id"], status="stopped", reason=str(e), seconds=now() - t0), open(os.path.join(out_dir, "summary.json"), "w"))
            os.replace(path, os.path.join(q, "stopped", os.path.basename(path)))
            ctx.log(f"job {job['job_id']} stopped: {e}")
            break
        except Exception as e:
            tb = traceback.format_exc()
            json.dump(dict(job_id=job["job_id"], status="FAILED", error=f"{type(e).__name__}: {e}", traceback=tb[-4000:], seconds=now() - t0),
                      open(os.path.join(out_dir, "summary.json"), "w"), indent=1)
            os.replace(path, os.path.join(q, "failed", os.path.basename(path)))
            ctx.log(f"job {job['job_id']} FAILED: {type(e).__name__}: {e}\n{tb[-2000:]}")
            if "out of memory" in str(e).lower():
                try:
                    import torch; torch.cuda.empty_cache()
                except Exception:
                    pass
    ctx.log("exiting")


if __name__ == "__main__":
    main()

In [ ]:
# =============================================================================
# 3d. IMPORT THE LIBRARIES AND CHECK THEM (a few seconds)
# =============================================================================
import importlib
for _m in ("ffb_video", "ffb_metrics"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])
import ffb_video as FV
import ffb_metrics as FM

def _selftest():
    rng = np.random.default_rng(0)
    out = []
    y = rng.integers(0, 2, 300); s = rng.normal(size=300) + y
    from sklearn.metrics import roc_auc_score
    out.append(("AUC equals scikit-learn", abs(FM.auc(y, s) - roc_auc_score(y, s)) < 1e-12))
    W = FM.cluster_bootstrap_weights(np.repeat(np.arange(30), 10), 50, seed=1)
    out.append(("cluster bootstrap keeps clusters together", all(len(set(W[0, k * 10:(k + 1) * 10])) == 1 for k in range(30))))
    out.append(("weighted AUC with unit weights equals AUC", abs(FM.weighted_auc(y, s, np.ones((1, 300)))[0] - FM.auc(y, s)) < 1e-12))
    out.append(("EER of separated scores is 0", FM.eer([0, 0, 1, 1], [0, 1, 2, 3]) == 0.0))
    out.append(("frame indices are spread over the window", FV.sample_indices(1000, 4, 300) == [37, 112, 187, 262]))
    img = np.zeros((100, 120, 3), np.uint8); img[40:60, 50:70] = 255
    c = FV.square_crop(img, np.array([50, 40, 20, 20, 1.0]), 1.3, 32)
    out.append(("face crop is square, RGB and centred on the face", c.shape == (32, 32, 3) and c[16, 16].min() == 255 and c[0, 0].max() == 0))
    ch = FV.choose_faces([np.zeros((0, 5)), np.array([[0, 0, 10, 10, .9], [50, 50, 30, 30, .8]])])
    out.append(("the largest face is chosen and lent to a frame without detection", ch[0][1] == "neighbour" and ch[1][0][2] == 30))
    return out

_res = _selftest()
for _n, _ok in _res:
    log(f"   self-test {'ok  ' if _ok else 'FAIL'} {_n}")
if not all(ok for _, ok in _res):
    raise RuntimeError("a library self-test failed; the libraries above were changed or a dependency behaves differently")
RUN["ok"]["libraries"] = True

## 4. Data

The notebook does not trust folder names alone. Every video is identified by its file name, which follows the official naming of FaceForensics++ (`000.mp4` for a source video, `000_003.mp4` for a manipulated one made from the pair 000/003) and of DFD (`01__exit_phone_room.mp4` for an actor's original, `01_02__exit_phone_room__YVGY8LOK.mp4` for a fake). The method comes from the nearest folder named after it, the compression from the nearest folder name that says `c23` or `c40`. Masks and models from the official download are ignored. Counts are compared with the published ones (FF++: 1,000 source videos and 1,000 per method; DFD: 363 originals and 3,068 fakes), and every difference is reported.

In [ ]:
# =============================================================================
# 4a. REHEARSAL DATA (only when DRY_RUN is True)
# =============================================================================
REHEARSAL_FACE_B64 = "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAMCAgMCAgMDAwMEAwMEBQgFBQQEBQoHBwYIDAoMDAsKCwsNDhIQDQ4RDgsLEBYQERMUFRUVDA8XGBYUGBIUFRT/2wBDAQMEBAUEBQkFBQkUDQsNFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBT/wAARCABuAF8DASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwD6S0208qPc38qvRR5cdcVfitkAAq4lvFsxwOK402dhCgUKOmcVGzlQcGm3TRWal5JAijux4+leB/tL/H5fh98PNYuND82XUfL8lLiMfJC7EAHPc9eB+lKU1Faglc9J8YfEbQPBb41PUY1uCMi3QhpMeuM8fjXMxfF648TzRx+GtFlvIGyGvr9xbQp+B+ZvyFfInwG8e33jCxnuw8U1+7lpVvrZpG3HI3KwbPOMdjwfSvpPwpolybuC7fVl3n/lkny4PfCEMwrw6uPqKTjFWsd8aEbXZ6fol74w06E3N+dJvEfJWK1R4go7fOzHJ/CorX4/abY6mbPXLG402IHabtB5sa+m4DkD3Gah13xZp2j6c/8AaEt3YiFMvdyBHTpnoFWQ/gK8J8QeLNC8WTytpl0bjj/WwKI+v8T5wfzHtisoY6tF3buP2MZdD64s/EGla/YpeaZfRX1s3IkhbI/HuPpTGYtnBwPaviGx1bW/BGqxXGmasBdzM3lRQSZ88AZYBcgED3468nBx9HfC/wCNNj4jSy0rV5Db6zMNqM8WxZT2BxkKxx0z/PFetQx8Kz5XoznnRcVdHpqw4YEmrCKFGc9KHgIGetSW0e4EGu5mCZhC8Pr+tQ6t4nt/D2mTXt7MsVvEMlmP5Cs9JW3da8W/aP8AFH9m3vhyxmlEdo3m3Mu7oSpQLx36tx7iuSpV9lByNIx5nY5j4mfHe41vUxGWe004HiNnw0mPXB78cZqj4n0aX4r+AY7Wea2skaRWaNiASg9T/k9K47UfBtzqE66qSJYsCaNs5Izg5PtjNbPhLWtL8QTz2XkvJbWQIaSNycnAPPvy3514MsRJt8x3xpJ2SDwfoPhnwvYx6FPbxSS2nztdWZKzByPvHBzjAVSAf+Weea9O8NXlvKC0GpXsc+P3c9/GGzx2D5z0FeTeIPDunTXIvdFnlguY3DGQH7ygcg+x5rZ8JJFFqKC4ZSrHcrwoVfPv8wHU/jntXE2272OnkaPT9fk1i7EUhl0S9hRgSk6FMHjGAAAT9T+FczPFor3ss+o28IuMBWEMZCtyOucnGR6c+vp0P9u2k8aQS3himbCrPGxdGP8ACCOq/Tn6nrRrnxA0Y+HrjTtTtoZoxGQLplAkXnDj39efTHXBppGT0R83/Eq9stE1uGO6uVNspWeJ/NUOjDuCDkfn04NVNQ8cNqdhB4g0yX7DHbyrE08coLrMArBuGIGSQCD0/lwXxRsYtS8SyaVbzW09gzF4HDhN4PQgnGCPbFT22nTaNon/AAj0wt7uKXbI4s5QwBHBBwAc4OOf/r1SjFRUrkNn6dfBLx7F8WPhjpOvLsW8ZDBeRoQQk6cP+B6j2Ndc0bQt0/Kvk7/gnbqi6XpnifwxdSyR3qGHUFgd9wCMCrYB6EHbn619lN5RXJwa+so1PaU1I8yUbSseXtEY2BzmvBf2rdMtbyw0G7lkVLiJ5Y0DEc7tvr7qPzr6G8sMMe1fOv7bOhXs3w80zUrKEyPaXoSSQHCwo4xvPtkKP+BVhi6fNQkkbUnaaPG9R8dy+HfBOqQq4NxPEYN5wcZHQfniu6/Yl8PWes2viC9u41Fo86x5lOASF3Fsn2K18p2viO4eR7HUH8+XOyOPIwrHjP5fr+Nfdn7Mnh5vDXwrt9QisRqLTPJdi3CjEjH7g56cKlfOQg18R6sdzvf+Gf8AwvrOrTzaLrMWGGPs4lVlJ54x1IyTWjcfsh20dmk9nfvbuV+ZFY5+n068VxusfGj4m6h4m8P6BeeDtE/szUVQs0sMsiWYPmArI5TaCuwZ5A+dcZGSPbfhL4nutTtFS5iks4NuFiYkoO2F54Ht+VdXsqa0aNueTR4h46/Y+1lrTfpuqoJwQ7AEhnx15H+elfO/xu+D/wAUPDVjNL9lfU4ADvltmyR0ycdeeTwOoFfavxB+KOoPrn9iaTby3dw524idk5PRdw5H4dgewNeFv8etI8UQ67Zar4Q1hrbSHEF5fRai7iIkA5CiTLYyMgcjuODiVRg3eJMpaJSPgEeIRFeW51S0l3xx/ZpQVx8w4Gc45x39q67w9rMFtHb+TbPbRBn3CWXe0mRjOMf1rs/2lPDUB8H3dzayy3dqsyXEEso+YISARnHPDDrzXjHgGxnu/D0si3AV7N/vuN21T06+5q5QUqXOtLM4paSsfU/7FetajpP7Q0dtLl4dXsri3MoQjcUUy8+hGzntX6G3s7RRgIefrXwh+wj4YvLrxxq3ii4vlvLLSbX7LBGDkLNL1IPQ4VT/AN9V9oTavJNKwwfyr18G2qSucU1eRWRACPWl1Pw9ZeJtLutM1K1jvLG5jMcsEq5VlPUVZjtcnJNa2mW+WyBmvRSvoYXsfnR8Xf2NdV8B+M3uNDSbUNFmLTQ5GXRVwWU+/P5CvuH9nnSYtI8EaZYSxhzDDGnTttHOD9K9CvtKF/DtMYLDlS44+n0rjbW8n0DxDf26WqoodD5ULlwMoDwSBXh4qg6M+foethakai5Xuemax4fszGi3Mcn2YgMoB2rXJWxi0+8mFuqxoAREgOcD1rAuNe1jx9qAimMllpVmxBWRsPOw/hC9lB6k9e3rU8tzqmlPJcvoi39lGu0NDIxkY4PVcAcf7xz6VySkpv3T04wSVm9TlNAkhi+IWoRzYHnqMITyT2b+ddB4l+G2ia1Z3ayWsaSXHMhCBWb6mvGfEur6zfeLXvo9Jk0ltyeWzZBUD1yPrXsulePGudBkN4sf2hI8OVOQRz8w/wAKiMraBKC6Hyh+1HpWmab8O7yyjiSNLeGReOSTjj9cV8f/AA3glvo7nRtPtZL3UNVh+yw20S7nkkLALgfrX2b8SvDE3xZu73SYEnkjnBI8hGY9RjhQe9ezfsv/ALJGjfArS21e7hN94puxzdXCjdbR4+4g525yc8k8gZ4rpoUnWTj0PNrtQ1L37NfwFT4GfCyz0aZln1m5b7XqU6fdadgMqp/uqAFH0zxmvSW07k5WuikTaMYpI1UnDDNe3y20R517mXbWpYDjHFVLzx54b8Hsw1LVIIZBz5ane/5Lk18+/tS/tQ6f8KtJuI0uDsVjAiW7DzbqUdUQ9lHdun5jP5ofEH9o3xn4+vZm/tGTSbJydtrYMU4/2n+8x9eQPYV6XIo77nI2lufsHrf7Wvw+0c4kubxh/eSFQD/30wqr4I+Lfhn4neJr6+0G4kkQCPzVmUBgwGOgJGMD9K/D51vNQdpn8+4Y8mRssfzNfd3/AATJ0yaf/hOYVUxXS/ZJYWbgZHmgg+xBry8fyuizqws/3mx+hHxA8HWPiazsriNp7W5tZBcRyWk7w7j/ABIxQglWzyPoeorT0rT7NNBM0mq65ox+VWQyLKoIOX65Yjb0Jx+NZ/hTxAuql7G6RobuH5XglGGX0/D3HWtrVvDoewRxvRiCArAZ/DNeHSla8rH0MXdJNngHxb0H/hINRj07TPFGtQec+XucKJAmTgKNuASuPmOcehrs9c0HS/Bnw2gt4wz3ckIjVpnLu3HLMScljxkn0FJ4ns7TQXe+uZS0i8Lv4x615j4s8b3Hi/UrTTrd909w6QRqP4AxCjI9eaym+aWw6k7aHpv7MWjCGPXtbkUiOZ0s4GI+8EyXI9ssB/wGvaLufzH46Vm6H4etvDOk2ml2EXlWdsgjRe59ST3JOST6mr8qbFGP1r6WjTVKmoHzlSbnNyG+T5gqJ7Pbz2q7CAUz3qCdtpyeM02tQTPw8/ab1u88bfFfUopHkXT9FjS2XJyMkBm2j+8zNj6AdhW94Q+CEnwy8IweNvGCGxvtQjb+w9JlQFmAwHuJsg4UA7VGMlmzwF5+n/hN8A9K8f8Axnt766toJbZZmv5obgHZLMqqI3P+6GkGO5x6V59+3RPq3jDxrLb6NbNHp+jW8kF4pddsLROygKM5OQA5wMcc4zWlSUpTktjlkrM+LvEOvy6nq13ch/8AWys428Dkmvqb/gnx8cfD/wAI/HV7aeKrr7BY6xHGEvpATHFIjMAJPRSH+90GBnAyR8rSadbWZBaQyMD/AJ4psl09tdwMDhe30NRVpxqw5C6c3TlzH7xa/Y6V42t4tW8La3azXIX93d6dOkwGecHaSCM15j4l8d/FjwxCwbQ7XW44wR5kUrIxH+7g/wA6/IO11ma2YPFK0bjujEH9K6nTPjl468NIo0vxnr1gqniOHUZQh+qbtp/EV5X1OS+FnoRxbjsfe/iHxJ4x8dOJ9YQaZAOlvGSWHsSa47xd8R7b4HaXbeJZUeWS0u4WRE2l2feMEBuCRy2D1x2r5Mvf2tPihd2H2SXxEJ17TPaQ+YPx28/jXl3iTxhrfi26E+tard6nKOhuJSwX6DoPwrSlgZqalJ7EVMZdNdT9xfgF+1L4V+N+j2rw3cEV7LhVKnakj/3MHlH/ANhuvYmvYbiIE1/Pb8NviVrPwv15dS0qdgrYE9qxPlXCA9GH54Ycg9K/YD9kr9qbT/i7oFhY6ldlrmYeVbXU7fP5gHMEp/vj+Fv4hjuRn2ZRT1RyQlzH0Sp8s4FMuY3ccAkda01sSsmMcetWktkB7GsLXZpfQ/Or4PfEy31SLT/EcAWBdRHl3EcTY+zXHSRMdl3HcoP8LL3Br1fxT+z/AKB8YNLv5LGVNO8W3sfm2967YV50XBhl9VYd8EjII6EH5G8Q6Rd/Cv8AaD1zQLGSKbTbm1S8uoCSqNHsLBlGPldecDkYJGcE17VpHiHVfEulaj4ZTUrnT9RWEm11O2cq8ZAxzggnIOP85rW8ZvmfUVSF1zI+GPiF8FNb8I6pq4vrKXTpNOuDDc2lxw8bA4OD0I9xXmWryBrhQvRBiv0Z/ai8P/8ACWfC/wAL63rd5LLci3vLdvJUZYRpCASx5b5wGwf0r82pWMkjE9zTStKxyN6GnCx2j0IpLnBh6YPaprRVFopIyQMZqKf5hs6elT1L6FPgbu/eq8hy5qZ8AkCq54JrRGTNrQLIauJLKQ4QI0qvjJix1b12gcsPQE9q9O+Avjm9+F3jk6dqLSW9lcusNyob7hz8si47qeQR1GQOteb+DLqey1iK6gCM1sDMUk+66gfMh9QRkEe9exeIfA8c0N9AJB9p0R5Y47hvvSQoFeIN6sEJUn1jTHc1DbTNIH7A/Af4pv8AELwo1vfyK+u6YwguyCP3q4+SXj+8Ac+6nsRXpAuCDxX59/sfeO7uy1vwVM5ZxqFu2lXIB+8oJEZ/DZH+APrX3uZiOaJJJ3Onof/Z"   # 95 x 110 JPEG of the NASA portrait of astronaut Eileen Collins (public domain), as shipped with scikit-image

def dry_make_videos(root):
    """Small synthetic copies of FF++ and DFD in three of the real folder layouts.

    Faces are the embedded portrait with a colour, size and position per synthetic 'person'; each manipulation
    method leaves a different artefact inside the face. Nothing measured on these videos says anything about
    deepfake detection: they exist to exercise discovery, extraction, training, evaluation, figures and export.
    """
    import base64, cv2
    rng = np.random.default_rng(7)
    face0 = cv2.imdecode(np.frombuffer(base64.b64decode(REHEARSAL_FACE_B64), np.uint8), cv2.IMREAD_COLOR)
    have_ffmpeg = shutil.which("ffmpeg") is not None

    def person(pid):
        r = np.random.default_rng(int(hashlib.md5(str(pid).encode()).hexdigest()[:8], 16))
        f = face0.astype(np.float32) * r.uniform(0.8, 1.15, 3)[None, None, :] + r.uniform(-12, 12)
        if r.random() < 0.5:
            f = f[:, ::-1]
        return np.clip(f, 0, 255).astype(np.uint8), r.uniform(0.9, 1.25)

    planted = os.environ.get("FFB_DRY_PLANTED", "")              # test hook: "1" = an unmistakable mark on every fake

    def manipulate(face, method, other):
        h, w = face.shape[:2]
        f = face.copy()

        yy, xx = np.mgrid[0:h, 0:w]
        inner = (((xx - w * 0.5) / (w * 0.30)) ** 2 + ((yy - h * 0.48) / (h * 0.30)) ** 2) <= 1.0
        if method in ("Deepfakes", "FaceSwap", "FaceShifter", "DFD"):
            o = cv2.resize(other, (w, h))
            if method == "Deepfakes":
                o = cv2.GaussianBlur(o, (5, 5), 0)
            if method == "FaceShifter":
                o = (0.7 * o + 0.3 * face).astype(np.uint8)
            m = inner.astype(np.float32)
            if method in ("Deepfakes", "DFD"):
                m = cv2.GaussianBlur(m, (9, 9), 0)
            f = (f * (1 - m[..., None]) + o * m[..., None]).astype(np.uint8)
        elif method == "Face2Face":
            mouth = (yy > h * 0.55) & (yy < h * 0.75) & (np.abs(xx - w * 0.5) < w * 0.22)
            shifted = np.roll(f, 3, axis=0)
            f[mouth] = shifted[mouth]
        elif method == "NeuralTextures":
            lower = inner & (yy > h * 0.5)
            tex = ((np.sin(xx * 1.7) * np.sin(yy * 1.3)) * 18).astype(np.int16)
            f = np.where(lower[..., None], np.clip(f.astype(np.int16) + tex[..., None], 0, 255), f).astype(np.uint8)
        if planted == "1":                                       # plus a small green square on the left cheek, away from eyes, nose and mouth
            f[int(h * 0.60):int(h * 0.70), int(w * 0.20):int(w * 0.30)] = (0, 255, 0)
        return f

    def write_video(path, frames, crf, fps=25):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        H, W = frames[0].shape[:2]
        if have_ffmpeg:
            p = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{W}x{H}", "-r", str(fps), "-i", "-",
                                "-c:v", "libx264", "-preset", "veryfast", "-crf", str(crf), "-pix_fmt", "yuv420p", path],
                               input=np.stack(frames).tobytes(), capture_output=True)
            if p.returncode == 0 and os.path.exists(path):
                return
        vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))      # fallback: MPEG-4 part 2; c40 imitated by JPEG damage
        for fr in frames:
            if crf >= 40:
                fr = cv2.imdecode(cv2.imencode(".jpg", fr, [int(cv2.IMWRITE_JPEG_QUALITY), 20])[1], cv2.IMREAD_COLOR)
            vw.write(fr)
        vw.release()

    def clip_frames(pid, method=None, other=None, size=(240, 320), n=40, partner=None):
        """partner: a second, larger, never manipulated person on the left (a two-person scene such as DFD's hugging scenes);
        the main person then stands on the right."""
        H, W = size
        face, sc = person(pid)
        bg_key = f"{pid}{partner}"                     # an original and its manipulated versions share background and motion, as in FF++ and DFD
        fr_rng = np.random.default_rng(int(hashlib.md5(bg_key.encode()).hexdigest()[:8], 16))
        bg = cv2.resize(fr_rng.uniform(30, 220, (H // 16 + 1, W // 16 + 1, 3)).astype(np.uint8), (W, H), interpolation=cv2.INTER_CUBIC)
        fh = int(min(H * (0.45 if partner else 0.62), face.shape[0] * sc * H / 240)); fw = int(fh * face.shape[1] / face.shape[0])
        ofs = fr_rng.uniform(0, 6.28, 2)
        if partner:
            pf = person(partner)[0]; ph = int(H * 0.62); pw = int(ph * pf.shape[1] / pf.shape[0]); pf = cv2.resize(pf, (pw, ph))
        frames = []
        for t in range(n):
            f = cv2.resize(face, (fw, fh))
            if method:
                f = manipulate(f, method, person(other)[0] if other is not None else face)
            fr = bg.copy()
            if partner:
                px, py = int(W * 0.04), int((H - ph) / 2)
                fr[py:py + ph, px:px + pw] = pf
                x = int(W * 0.62 + W * 0.04 * np.sin(t / 9.0 + ofs[0])); y = int((H - fh) / 2 + (H - fh) * 0.15 * np.cos(t / 7.0 + ofs[1]))
                x = min(x, W - fw)
            else:
                x = int((W - fw) / 2 + (W - fw) * 0.25 * np.sin(t / 9.0 + ofs[0])); y = int((H - fh) / 2 + (H - fh) * 0.2 * np.cos(t / 7.0 + ofs[1]))
            fr[y:y + fh, x:x + fw] = f
            frames.append(fr)
        return frames

    # FF++: the first pairs of each official split
    pairs = {"train": FFPP_SPLITS["train"][:6], "val": FFPP_SPLITS["val"][:2], "test": FFPP_SPLITS["test"][:2]}
    ids = {s: sorted(set(x for p in v for x in p.split("_"))) for s, v in pairs.items()}
    rows23, rows40 = [], []
    # newer Kaggle notebooks mount a dataset at /kaggle/input/datasets/<owner>/<slug>, older ones at /kaggle/input/<slug>: both are used here
    base23 = os.path.join(root, "datasets", "abbasaliyev", "ffpp-core-c23-videos")        # layout of the abbasaliyev copies
    base40 = os.path.join(root, "faceforensics_official", "FaceForensics")                  # layout of the official download script
    for split, plist in pairs.items():
        for vid in ids[split]:
            fr = clip_frames(vid)
            write_video(os.path.join(base23, "real", vid + ".mp4"), fr, 23)
            rows23.append(dict(video_id=vid, method="", label=0, split=split, path=f"real/{vid}.mp4"))
            if split == "test":
                write_video(os.path.join(base40, "original_sequences", "youtube", "c40", "videos", vid + ".mp4"), fr, 40)
        for p in plist:
            a, b = p.split("_")
            for tgt, src in ((a, b), (b, a)):
                for m in ALL_METHODS:
                    fr = clip_frames(tgt, m, src)
                    write_video(os.path.join(base23, m, f"{tgt}_{src}.mp4"), fr, 23)
                    rows23.append(dict(video_id=f"{tgt}_{src}", method=m, label=1, split=split, path=f"{m}/{tgt}_{src}.mp4"))
                    if split == "test":
                        write_video(os.path.join(base40, "manipulated_sequences", m, "c40", "videos", f"{tgt}_{src}.mp4"), fr, 40)
    pd.DataFrame(rows23).to_csv(os.path.join(base23, "index.csv"), index=False)
    # a mask video of the official layout, which must be ignored
    write_video(os.path.join(base40, "manipulated_sequences", "Deepfakes", "masks", "videos", "999_960.mp4"), clip_frames("999"), 40)

    # DFD: four actors, two scenes; the c23 copy in the layout of the sanikatiwarekar copy (no compression in its name),
    # the c40 copy in the layout of the abbasaliyev copies
    actors, scenes = ["01", "02", "03", "04"], ["exit_phone_room", "walking_outside_cafe_disgusted", "hugging_happy"]
    partner_of = {"hugging_happy": "partner"}                 # a two-person scene: a larger, never manipulated second person
    d23 = os.path.join(root, "datasets", "sanikatiwarekar", "deep-fake-detection-dfd-entire-original-dataset")
    d40 = os.path.join(root, "ffpp-dfd-c40-videos")
    rows = []
    for a in actors:
        for sc in scenes:
            fr = clip_frames("dfd" + a + sc[:4], size=(270, 480), partner=partner_of.get(sc))
            write_video(os.path.join(d23, "DFD_original sequences", "DFD_original sequences", f"{a}__{sc}.mp4"), fr, 23)
            write_video(os.path.join(d40, "actors", f"{a}__{sc}.mp4"), fr, 40)
            rows.append(dict(video_id=f"{a}__{sc}", method="", label=0, split="", path=f"actors/{a}__{sc}.mp4"))
    for a in actors:
        for b in actors:
            if a == b:
                continue
            sc = scenes[(int(a) + int(b)) % 3]
            code = hashlib.md5(f"{a}{b}{sc}".encode()).hexdigest()[:8].upper()
            fr = clip_frames("dfd" + a + sc[:4], "DFD", "dfd" + b + sc[:4], size=(270, 480), partner=partner_of.get(sc))
            name = f"{a}_{b}__{sc}__{code}"
            write_video(os.path.join(d23, "DFD_manipulated_sequences", "DFD_manipulated_sequences", name + ".mp4"), fr, 23)
            write_video(os.path.join(d40, "DeepFakeDetection", name + ".mp4"), fr, 40)
            rows.append(dict(video_id=name, method="DeepFakeDetection", label=1, split="", path=f"DeepFakeDetection/{name}.mp4"))
    pd.DataFrame(rows).to_csv(os.path.join(d40, "index.csv"), index=False)
    n = sum(len(fns) for _, _, fns in os.walk(root))
    log(f"   rehearsal videos written: {n} files under {root} ({'H.264 via ffmpeg' if have_ffmpeg else 'MPEG-4 via OpenCV'})")

if DRY_RUN:
    _t = time.time()
    if not os.path.isdir(INPUT_ROOT) or not any(os.scandir(INPUT_ROOT)):
        os.makedirs(INPUT_ROOT, exist_ok=True)
        dry_make_videos(INPUT_ROOT)
    log(f"rehearsal input ready in {time.time() - _t:.0f} s")

In [ ]:
# =============================================================================
# 4b. FIND THE VIDEOS, CHECK THE COUNTS, ASSIGN THE SPLITS
# =============================================================================
VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".webm")
RX_FF_REAL = re.compile(r"^(\d{3})$")
RX_FF_FAKE = re.compile(r"^(\d{3})_(\d{3})$")
RX_DFD_REAL = re.compile(r"^(\d{2})__([A-Za-z0-9][A-Za-z0-9_\-]*)$")
RX_DFD_FAKE = re.compile(r"^(\d{2})_(\d{2})__([A-Za-z0-9][A-Za-z0-9_\-]*?)__([A-Za-z0-9]{8})$")
RX_COMP = re.compile(r"(?<![a-z0-9])(c23|c40|c0|raw)(?![a-z0-9])")
METHOD_BY_FOLDER = {m.lower(): m for m in ALL_METHODS}
SKIP_FOLDERS = {"masks", "mask", "models", "images", "bounding_boxes", "landmarks", "faces", "frames", "__macosx"}

def try_kagglehub(slug):
    """Ask Kaggle to attach a dataset to this session (no download inside a Kaggle notebook). Returns its path or None."""
    if not (CFG["ALLOW_KAGGLEHUB"] and ON_KAGGLE and not DRY_RUN):
        return None
    try:
        import kagglehub
        p = kagglehub.dataset_download(slug)
        if p and os.path.isdir(p):
            log(f"   attached Kaggle dataset {slug} -> {p}")
            return p
    except Exception as e:
        log(f"   could not attach {slug}: {type(e).__name__}: {str(e)[:160]}")
    return None

def folder_parts(dp):
    """The folder names of dp below /kaggle/input (all of its folder names if it lies elsewhere, e.g. a kagglehub cache)."""
    a, r = os.path.abspath(dp), os.path.abspath(INPUT_ROOT)
    inside = a == r or a.startswith(r + os.sep)
    return [p for p in (os.path.relpath(a, r) if inside else a).split(os.sep) if p not in ("", ".")]

def copy_name(parts, fallback):
    """The attached copy a folder belongs to. Newer Kaggle notebooks mount a dataset at /kaggle/input/datasets/<owner>/<slug>,
    older ones at /kaggle/input/<slug>; both give <slug>."""
    if len(parts) >= 3 and parts[0].lower() == "datasets":
        return parts[2]
    return parts[0] if parts else fallback

def scan_videos(roots, name=None):
    """Every video file under the roots, identified by name. Returns (rows, ignored-count by reason)."""
    rows, ignored, seen_real = [], defaultdict(int), set()
    for root in roots:
        for dp, dns, fns in os.walk(root, followlinks=True):
            dns[:] = sorted(d for d in dns if d.lower() not in SKIP_FOLDERS)
            parts = folder_parts(dp)
            low = [p.lower() for p in parts]
            for fn in sorted(fns):
                stem, ext = os.path.splitext(fn)
                if ext.lower() not in VIDEO_EXT:
                    continue
                comp = None
                for p in reversed(low):
                    m = RX_COMP.search(p)
                    if m:
                        comp = m.group(1).replace("raw", "c0"); break
                top = name or copy_name(parts, os.path.basename(os.path.abspath(root)))
                if comp is None:                            # an override names any folder of the path, usually the copy's own folder
                    comp = next((CFG["COMPRESSION_OVERRIDES"][p] for p in reversed(parts) if p in CFG["COMPRESSION_OVERRIDES"]), None)
                method = next((METHOD_BY_FOLDER[p] for p in reversed(low) if p in METHOD_BY_FOLDER), None)
                path = os.path.join(dp, fn)
                if RX_DFD_FAKE.match(stem):
                    a, b = RX_DFD_FAKE.match(stem).group(1, 2)
                    rows.append(dict(family="dfd", video_id=stem, method="DFD", label=1, actor_a=a, actor_b=b, comp=comp, path=path, top=top))
                elif RX_DFD_REAL.match(stem):
                    a = RX_DFD_REAL.match(stem).group(1)
                    rows.append(dict(family="dfd", video_id=stem, method="real", label=0, actor_a=a, actor_b=a, comp=comp, path=path, top=top))
                elif RX_FF_FAKE.match(stem):
                    if method is None:
                        ignored["FF++ manipulated video outside a method folder"] += 1; continue
                    rows.append(dict(family="ffpp", video_id=stem, method=method, label=1, actor_a="", actor_b="", comp=comp, path=path, top=top))
                elif RX_FF_REAL.match(stem):
                    if method is not None:
                        ignored["source-video name inside a method folder"] += 1; continue
                    rows.append(dict(family="ffpp", video_id=stem, method="real", label=0, actor_a="", actor_b="", comp=comp, path=path, top=top))
                else:
                    ignored["name not in FF++ or DFD format"] += 1
    return rows, dict(ignored)

def stage_discover():
    roots = [INPUT_ROOT] if os.path.isdir(INPUT_ROOT) else []
    rows, ignored = scan_videos(roots)
    df = pd.DataFrame(rows, columns=["family", "video_id", "method", "label", "actor_a", "actor_b", "comp", "path", "top"])
    # attach what is missing (Kaggle only)
    want = {"ffpp_c23": ("ffpp", "c23"), "ffpp_c40": ("ffpp", "c40"), "dfd_c23": ("dfd", "c23"), "dfd_c40": ("dfd", "c40")}
    for key, (fam, comp) in want.items():
        sub = df[(df["family"] == fam) & (df["comp"].fillna("c23" if fam == "dfd" else "") == comp)] if len(df) else df
        if len(sub) and sub["label"].nunique() == 2:      # present with both classes (the DFD fakes inside xdxd003/ff-c23 come without originals)
            continue
        for slug in CFG["KAGGLE_DATASETS"].get(key, []):
            p = try_kagglehub(slug)
            if p:
                more, ign2 = scan_videos([p], name=slug.split("/")[-1])
                for k, v in ign2.items():
                    ignored[k] = ignored.get(k, 0) + v
                df = pd.concat([df, pd.DataFrame(more, columns=df.columns)], ignore_index=True)
                break
    if not len(df) or not (df["family"] == "ffpp").any():
        raise RuntimeError("No FaceForensics++ video was found under " + INPUT_ROOT + ".\n"
                           "  -> Add Input -> Datasets -> abbasaliyev/ffpp-core-c23-videos (and the c40 and DFD copies named in the first cell),\n"
                           "     or the folder written by the official FaceForensics download script; then run all cells again.")
    unknown = df["comp"].isna()
    if unknown.any():
        tops = sorted(df.loc[unknown, "top"].unique())
        warn(f"compression not stated in the folder names of {', '.join(tops)}: assumed c23 (the 24 GB of the known DFD copy match the c23 release; "
             "c40 is about 3 GB, raw about 1.8 TB). If that is wrong, set CFG['COMPRESSION_OVERRIDES'] = {'<folder>': 'c40'}.")
        df.loc[unknown, "comp"] = "c23"
    df = df[df["comp"].isin(["c23", "c40"])].copy()
    # the same video attached twice (two copies of one dataset): keep one
    df = df.sort_values(["family", "comp", "method", "video_id", "path"])
    dup = df.duplicated(["family", "comp", "method", "video_id"], keep="first")      # FF++ fakes of different methods share their name
    if dup.any():
        log(f"   {int(dup.sum())} duplicate video(s) found in more than one attached copy; one of each is used")
    df = df[~dup].reset_index(drop=True)

    # ---- splits ----
    split_of, pair_of = {}, {}
    for s in ("train", "val", "test"):
        for p in FFPP_SPLITS[s]:
            a, b = p.split("_")
            split_of[a] = split_of[b] = s
            pair_of[a] = pair_of[b] = p
    def ff_split(r):
        if r["method"] == "real":
            return split_of.get(r["video_id"], "unlisted"), pair_of.get(r["video_id"], r["video_id"])
        a, b = r["video_id"].split("_")
        sa, sb = split_of.get(a, "unlisted"), split_of.get(b, "unlisted")
        if sa != sb:
            return "excluded", f"{min(a, b)}_{max(a, b)}"
        return sa, pair_of.get(a, f"{min(a, b)}_{max(a, b)}")
    ff = df["family"] == "ffpp"
    sp = df[ff].apply(ff_split, axis=1, result_type="expand")
    df.loc[ff, "split"] = sp[0].values; df.loc[ff, "cluster"] = sp[1].values
    actors = sorted(set(df.loc[~ff, "actor_a"]) | set(df.loc[~ff, "actor_b"]))
    rng = np.random.default_rng(CFG["SEED"])
    perm = list(rng.permutation(actors))
    train_actors = set(perm[: int(round(len(perm) * CFG["DFD_TRAIN_ACTOR_SHARE"]))])
    def dfd_split(r):
        a, b = r["actor_a"] in train_actors, r["actor_b"] in train_actors
        return "dfd_train" if (a and b) else ("dfd_test" if not (a or b) else "dfd_mixed")
    df.loc[~ff, "split"] = df[~ff].apply(dfd_split, axis=1)
    df.loc[~ff, "cluster"] = "actor" + df.loc[~ff, "actor_a"]
    df["content"] = df["family"] + "|" + df["method"] + "|" + df["video_id"]   # the same content at another compression shares this key
    df["uid"] = df["family"] + "|" + df["comp"] + "|" + df["method"] + "|" + df["video_id"]

    # ---- QUICK / rehearsal subset ----
    if QUICK:
        q = CFG["QUICK_SUBSET"]; r = np.random.default_rng(CFG["SEED"])
        keep_pairs = set()
        for s, k in (("train", q["train_pairs"]), ("val", q["val_pairs"]), ("test", q["test_pairs"])):
            pl = sorted(set(df.loc[ff & (df["split"] == s), "cluster"]))          # pairs that are present in the attached copy
            keep_pairs |= set(str(x) for x in r.permutation(pl)[:k])
        # DFD: as many actors from each half, so that both the training and the test actors are represented
        half = max(1, q["dfd_actors"] // 2)
        sub_actors = set(r.permutation(sorted(train_actors))[:half]) | set(r.permutation(sorted(set(actors) - train_actors))[:half])
        keep = (ff & df["cluster"].isin(keep_pairs)) | (~ff & df["actor_a"].isin(sub_actors) & df["actor_b"].isin(sub_actors))
        log(f"   QUICK subset: {int(keep.sum())} of {len(df)} videos")
        df = df[keep].reset_index(drop=True)

    # ---- counts against the published ones ----
    pub = CFG["PUBLISHED"]
    inv = []
    for comp in ("c23", "c40"):
        for m in ["real"] + ALL_METHODS:
            n = int(((df["family"] == "ffpp") & (df["comp"] == comp) & (df["method"] == m)).sum())
            inv.append(dict(dataset="FF++", compression=comp, method=m, videos=n, published=pub["ffpp_real"] if m == "real" else pub["ffpp_per_method"]))
        for m, p in (("real", pub["dfd_real"]), ("DFD", pub["dfd_fake"])):
            n = int(((df["family"] == "dfd") & (df["comp"] == comp) & (df["method"] == m)).sum())
            inv.append(dict(dataset="DFD", compression=comp, method=m, videos=n, published=p))
    inv = pd.DataFrame(inv)
    inv["status"] = np.where(inv["videos"] == 0, "not attached", np.where(inv["videos"] == inv["published"], "matches", "differs"))
    if not (QUICK or DRY_RUN):
        for r in inv[inv["status"] == "differs"].itertuples():
            warn(f"{r.dataset} {r.compression} {r.method}: {r.videos} videos found, {r.published} published")
    inv.to_csv(os.path.join(DIRS["tables"], "T01_data_inventory.csv"), index=False)
    if ignored:
        log("   files ignored: " + "; ".join(f"{k}: {v}" for k, v in ignored.items()))
    by = df.groupby(["family", "comp", "split", "method"]).size().rename("videos").reset_index()
    by.to_csv(os.path.join(DIRS["tables"], "T02_videos_by_split.csv"), index=False)
    excl = df["split"].isin(["unlisted", "excluded"])
    if excl.any():
        warn(f"{int(excl.sum())} FF++ video(s) are not in the official split (or their pair spans two splits) and are not used")
    RUN["data"].update(inventory=inv.to_dict("records"), ignored=ignored, dfd_train_actors=sorted(train_actors), dfd_actors=actors,
                       sources=sorted(df["top"].unique().tolist()))
    for r in inv.itertuples():
        if r.videos:
            log(f"   {r.dataset:4s} {r.compression} {r.method:15s} {r.videos:5d} videos (published {r.published:5d}) {r.status}")
    return df.drop(columns=["top"]).reset_index(drop=True)

VID = run_stage("data", stage_discover)
if VID is None:
    raise RuntimeError("The data stage failed (see above). Nothing can run without FaceForensics++.")
HAVE = {f"{fam}_{comp}": bool(((VID["family"] == fam) & (VID["comp"] == comp)).any()) for fam in ("ffpp", "dfd") for comp in ("c23", "c40")}
HAVE["dfd_both_classes"] = {c: bool(((VID["family"] == "dfd") & (VID["comp"] == c) & (VID["label"] == 0)).any() and ((VID["family"] == "dfd") & (VID["comp"] == c) & (VID["label"] == 1)).any()) for c in ("c23", "c40")}
for _c in ("c23", "c40"):
    if HAVE[f"dfd_{_c}"] and not HAVE["dfd_both_classes"][_c]:
        warn(f"DFD {_c}: only one class was found (for example the DFD fakes inside xdxd003/ff-c23 without the actors' originals); DFD {_c} cannot be evaluated")
log(f"available: {', '.join(k for k, v in HAVE.items() if v is True) or 'nothing'}")

## 5. Models and faces

**Weights first.** The EfficientNet-B4 and CLIP ViT-L/14 weights are downloaded before anything else, so that a session without Internet stops within minutes instead of after the face extraction.

**Faces.** From every video the notebook decodes a fixed number of frames, spread evenly over its first 300 frames, finds the faces with **YuNet** (OpenCV's `FaceDetectorYN`, MIT licence; the model file is downloaded from the OpenCV model zoo and checked against its SHA-256), and stores a square crop of 1.3 × the face box at 224 × 224 pixels. Nothing is re-encoded, so the only compression a crop carries is the video's own.

*Which face.* In most frames there is one face. When a frame shows two or more (DFD has scenes with two people, of whom only one is manipulated), the notebook opens the **counterpart video** at the same frame (the original of a manipulated video, or a manipulated version of an original) and takes the face whose region differs clearly most between the two: that is the manipulated face. If no face differs clearly (for example because the two videos are not frame-aligned), it falls back to the largest face, followed from frame to frame by overlap. A frame without a usable face borrows the box of the nearest frame that has one; a video without any face is reported and left out. Table T04 reports, per set, how often frames showed several faces and how often the difference decided.

*How many frames per video:* 16 for real and 4 for manipulated FF++ training videos (with the four classic methods this gives about as many real as manipulated crops; a leave-one-method-out configuration has 16 real crops for every 12 manipulated ones), 8 for validation, 16 for the FF++ test videos and 8 for DFD.

*Time and disk.* Before extracting, the notebook measures a few videos of each set and, if the plan would not fit in its time budget or on the disk, applies the cut rules of table T03 in order. Decoding dominates the time, so the rules shorten the decoding window and drop DFD c40 before anything else. Videos are processed in a random order within each set, so that a set cut short by the time limit is a random subset; such a set is used and flagged. The plan is stored with the notebook's output, and a later session that resumes uses the same plan.

In [ ]:
# =============================================================================
# 5a. FACE DETECTOR AND MODEL WEIGHTS
# =============================================================================
def get_face_detector():
    """The YuNet model file, verified. In a rehearsal without Internet, OpenCV's Haar cascade stands in; in the benchmark
    and the quick test a missing YuNet stops the run, because crops from another detector would be another benchmark."""
    import cv2, urllib.request
    mdir = os.path.join(WORK, "ffb_models"); os.makedirs(mdir, exist_ok=True)
    path = os.path.join(mdir, FV.YUNET_FILE)
    def sha(p):
        return hashlib.sha256(open(p, "rb").read()).hexdigest()
    if not (os.path.exists(path) and sha(path) == FV.YUNET_SHA256):
        for url in FV.YUNET_URLS:
            try:
                with urllib.request.urlopen(url, timeout=60) as r, open(path + ".part", "wb") as f:
                    f.write(r.read())
                if sha(path + ".part") == FV.YUNET_SHA256:
                    os.replace(path + ".part", path)
                    log(f"   YuNet downloaded from {url.split('/')[2]} and verified")
                    break
                log(f"   YuNet from {url.split('/')[2]}: checksum differs, not used")
            except Exception as e:
                log(f"   YuNet download from {url.split('/')[2]} failed: {type(e).__name__}: {str(e)[:120]}")
    problem = None
    if os.path.exists(path) and sha(path) == FV.YUNET_SHA256:
        try:
            FV.FaceDetector("yunet", path, CFG["DET_SCORE"], CFG["DET_MAX_SIDE"]).detect(np.zeros((120, 160, 3), np.uint8))
            return "yunet", path
        except Exception as e:
            problem = f"YuNet could not be loaded by OpenCV {cv2.__version__} ({type(e).__name__}: {str(e)[:160]})"
    else:
        problem = "the YuNet model could not be downloaded (is Internet on in the session options?)"
    if not DRY_RUN:
        raise RuntimeError(problem + ". The benchmark is defined with YuNet; turn Internet on and run again.")
    warn(problem + "; this rehearsal uses OpenCV's Haar cascade instead")
    FV.FaceDetector("haar")
    return "haar", None

def prepare_models():
    """Download (or, in a rehearsal, create) the weights once, before the workers start, so that they load from the local
    cache. The two detectors are independent: if one model cannot be obtained, its jobs are left out and the other runs."""
    info = dict(clip_ok=False, effnet_ok=False, offline=True)
    if DRY_RUN:
        from transformers import CLIPVisionConfig, CLIPVisionModelWithProjection
        d = os.path.join(STATE, "tiny_clip")
        if not os.path.exists(os.path.join(d, "config.json")):
            torch.manual_seed(0)
            c = CLIPVisionConfig(hidden_size=32, intermediate_size=64, num_hidden_layers=2, num_attention_heads=2,
                                 image_size=CFG["CLIP"]["image_size"], patch_size=16, projection_dim=16)
            CLIPVisionModelWithProjection(c).save_pretrained(d)
        info.update(clip_path=d, clip_ok=True, effnet_ok=True)
        return info
    try:
        from huggingface_hub import snapshot_download
        t0 = time.time()
        p = snapshot_download(CFG["CLIP"]["model"], allow_patterns=["config.json", "preprocessor_config.json", "model.safetensors"])
        if not os.path.exists(os.path.join(p, "model.safetensors")):
            p = snapshot_download(CFG["CLIP"]["model"], allow_patterns=["config.json", "preprocessor_config.json", "pytorch_model.bin"])
        info.update(clip_path=p, clip_ok=True)
        log(f"   CLIP weights ready ({time.time() - t0:.0f} s): {p}")
    except Exception as e:
        warn(f"the CLIP weights could not be downloaded ({type(e).__name__}: {str(e)[:200]}); the CLIP probe is left out. Is Internet on?")
    try:
        import timm
        t0 = time.time()
        m = timm.create_model(CFG["EFFNET"]["arch"], pretrained=bool(CFG["EFFNET"]["pretrained"]), num_classes=1)
        pc = getattr(m, "pretrained_cfg", {}) or {}
        info["effnet_pretrained_tag"] = str(pc.get("hf_hub_id") or pc.get("tag") or "")
        info["effnet_params_m"] = round(sum(x.numel() for x in m.parameters()) / 1e6, 1)
        info["effnet_normalisation"] = dict(mean=list(pc.get("mean", ())), std=list(pc.get("std", ())))
        info["effnet_ok"] = True
        del m
        log(f"   EfficientNet weights ready ({time.time() - t0:.0f} s): {info['effnet_pretrained_tag']}, {info['effnet_params_m']} M parameters")
    except Exception as e:
        warn(f"the EfficientNet weights could not be obtained ({type(e).__name__}: {str(e)[:200]}); the EfficientNet runs are left out. Is Internet on?")
    if not (info["clip_ok"] or info["effnet_ok"]):
        raise RuntimeError("neither the CLIP nor the EfficientNet weights could be obtained; turn Internet on in the session options and run again")
    return info

MODELS = run_stage("models", prepare_models, needs=("data",))
if MODELS:
    RUN["models"] = {k: v for k, v in MODELS.items() if k != "offline"}

In [ ]:
# =============================================================================
# 5b. PLAN, MEASURE, EXTRACT
# =============================================================================
SET_ORDER = ["ff_c23_train", "ff_c23_val", "ff_c23_test", "dfd_c23", "ff_c40_test", "dfd_c40"]
SET_DOC = {
    "ff_c23_train": "FF++ c23 train split: real + the four classic methods",
    "ff_c23_val": "FF++ c23 validation split: real + the four classic methods",
    "ff_c23_test": "FF++ c23 test split: real + all five methods",
    "ff_c40_test": "FF++ c40 test split: real + all five methods",
    "dfd_c23": "DFD c23: all actors' originals and fakes",
    "dfd_c40": "DFD c40: all actors' originals and fakes",
}
REFS = {}

def counterpart_paths():
    """uid -> path of the counterpart video in the same copy and compression: the original of a manipulated video, or a
    manipulated version of an original. It is opened only for frames that show two or more faces (ffb_video.guided_choice)."""
    key = {(r.family, r.comp, r.method, r.video_id): r.path for r in VID.itertuples()}
    partner = {}
    for lst in FFPP_SPLITS.values():
        for pr in lst:
            a_, b_ = pr.split("_"); partner[a_] = b_; partner[b_] = a_
    dfd_fakes = defaultdict(list)
    for r in VID[(VID["family"] == "dfd") & (VID["label"] == 1)].itertuples():
        m = RX_DFD_FAKE.match(r.video_id)
        dfd_fakes[(r.comp, m.group(1), m.group(3))].append(r.path)
    out = {}
    for r in VID.itertuples():
        ref = None
        if r.family == "ffpp" and r.label == 1:
            ref = key.get(("ffpp", r.comp, "real", r.video_id.split("_")[0]))
        elif r.family == "ffpp":
            b_ = partner.get(r.video_id)
            ref = next((key[k] for k in [("ffpp", r.comp, m, f"{r.video_id}_{b_}") for m in ("Deepfakes", "FaceSwap", "Face2Face", "NeuralTextures", "FaceShifter")] if k in key), None)
        elif r.label == 1:
            m = RX_DFD_FAKE.match(r.video_id)
            ref = key.get(("dfd", r.comp, "real", f"{m.group(1)}__{m.group(3)}"))
        else:
            m = RX_DFD_REAL.match(r.video_id)
            lst = sorted(dfd_fakes.get((r.comp, m.group(1), m.group(2)), []))
            ref = lst[0] if lst else None
        out[r.uid] = ref
    return out

def plan_sets(frames, windows, keep=None, dfd_fake_cap=None):
    """The crop sets: which videos (in a seeded random order), how many frames each, from which decoding window."""
    v = VID
    ff = v["family"] == "ffpp"
    sel = {
        "ff_c23_train": ff & (v["comp"] == "c23") & (v["split"] == "train") & v["method"].isin(["real"] + CLASSIC),
        "ff_c23_val": ff & (v["comp"] == "c23") & (v["split"] == "val") & v["method"].isin(["real"] + CLASSIC),
        "ff_c23_test": ff & (v["comp"] == "c23") & (v["split"] == "test"),
        "ff_c40_test": ff & (v["comp"] == "c40") & (v["split"] == "test"),
        "dfd_c23": (~ff) & (v["comp"] == "c23"),
        "dfd_c40": (~ff) & (v["comp"] == "c40"),
    }
    plan = OrderedDict()
    for name in SET_ORDER:
        if keep is not None and name not in keep:
            continue
        idx = np.where(sel[name].values)[0]
        if not len(idx) or (name.startswith("dfd") and not HAVE["dfd_both_classes"].get(name[-3:], False)):
            continue
        if name.startswith("dfd") and dfd_fake_cap:
            fakes = idx[v["label"].values[idx] == 1]
            if len(fakes) > dfd_fake_cap:                    # the same videos at both compressions: chosen by content
                cont = sorted(set(v.loc[v["family"] == "dfd", "content"].values) - set(v.loc[(v["family"] == "dfd") & (v["label"] == 0), "content"].values))
                chosen = set(np.random.default_rng(CFG["SEED"]).permutation(cont)[:dfd_fake_cap])
                idx = idx[(v["label"].values[idx] == 0) | np.isin(v["content"].values[idx], list(chosen))]
        idx = np.random.default_rng([CFG["SEED"], SET_ORDER.index(name)]).permutation(idx)
        sub = v.iloc[idx]
        if name == "ff_c23_train":
            slots = np.where(sub["label"].values == 0, frames["train_real"], frames["train_fake"])
        elif name == "ff_c23_val":
            slots = np.full(len(idx), frames["val"])
        elif name.startswith("ff_"):
            slots = np.full(len(idx), frames["test"])
        else:
            slots = np.full(len(idx), frames["dfd"])
        plan[name] = dict(video_rows=idx, slots=slots.astype(int), window=int(windows["dfd" if name.startswith("dfd") else "ffpp"]))
    return plan

def plan_bytes(plan):
    per = CFG["CROP_SIZE"] ** 2 * 3
    return {k: int(p["slots"].sum()) * per for k, p in plan.items()}

W0 = int(CFG["DECODE_WINDOW"])
CUT_RULES = [   # (level, rule, decoding windows, sets left out, cap on DFD fakes per compression)
    ("L0", "the plan as configured", dict(ffpp=W0, dfd=W0), (), None),
    ("L1", "DFD decoded over half the window", dict(ffpp=W0, dfd=max(8, W0 // 2)), (), None),
    ("L2", "L1 + DFD c40 left out", dict(ffpp=W0, dfd=max(8, W0 // 2)), ("dfd_c40",), None),
    ("L3", "L2 + every video decoded over half the window", dict(ffpp=max(8, W0 // 2), dfd=max(8, W0 // 2)), ("dfd_c40",), None),
    ("L4", "L3 + at most 1,000 DFD fakes (random)", dict(ffpp=max(8, W0 // 2), dfd=max(8, W0 // 2)), ("dfd_c40",), 1000),
]

def plan_from_rule(rule):
    lvl, text, windows, drop, cap = rule
    return plan_sets(CFG["FRAMES"], windows, [s for s in SET_ORDER if s not in drop], cap)

def videos_sha():
    return hashlib.sha256("\n".join(sorted(VID["uid"])).encode()).hexdigest()

def find_saved_plan():
    """A plan stored by an earlier session of this notebook (in this session's state, or in a previous version's output
    attached as an input), made under the same protocol for the same videos."""
    cands = [os.path.join(STATE, "extraction_plan.json")]
    roots = [INPUT_ROOT] + ([os.environ["FFB_PREVIOUS_OUTPUT"]] if os.environ.get("FFB_PREVIOUS_OUTPUT") else [])     # (the variable is a test hook)
    for root in roots:
        if os.path.isdir(root):
            base = root.rstrip(os.sep).count(os.sep)
            for dp, dns, fns in os.walk(root):
                if "extraction_plan.json" in fns and os.path.basename(dp) == os.path.basename(STATE):
                    cands.append(os.path.join(dp, "extraction_plan.json"))
                if dp.count(os.sep) - base >= 3:
                    dns[:] = []
    for c in cands:
        try:
            p = json.load(open(c))
        except Exception:
            continue
        if p.get("protocol_sha256") == PROTOCOL_SHA and p.get("videos_sha256") == videos_sha() and any(r[0] == p.get("level") for r in CUT_RULES):
            return p, c
    return None, None

def run_extraction(tasks, wcfg, nproc, budget_min, label, watchdog_min):
    """Run process_video over the tasks with a pool of spawn processes.

    A process that dies (a crashing decoder, the out-of-memory killer) breaks the pool: the pool is rebuilt and the
    videos that were in flight are tried once more; a video that is in flight when the pool breaks a second time is
    recorded as failed. If nothing finishes for `watchdog_min` minutes (a stalled read), the pool is terminated and the
    same happens. No new video is started once `budget_min` (minutes since the start) has passed.
    Returns (list of dict(task, res), indices of the tasks not run)."""
    import concurrent.futures as cf, multiprocessing as mp
    from concurrent.futures.process import BrokenProcessPool
    ctx = mp.get_context("spawn")
    todo = list(range(len(tasks)))[::-1]                # a stack: popped from the end, so the planned order is kept
    tries = defaultdict(int)
    results = []
    def new_pool():
        return cf.ProcessPoolExecutor(max_workers=nproc, mp_context=ctx, initializer=FV.init_worker, initargs=(wcfg,))
    ex, pending = new_pool(), {}
    last_done = last_log = t_set = time.time()
    stopped = False
    def fail(i, why):
        t = tasks[i]
        results.append(dict(task=t, res=dict(uid=t["uid"], n_frames=-1, fps=0.0, width=0, height=0, n_decoded=0, rows=[], seconds=0.0, error=why)))
    while todo or pending:
        broken = False
        while todo and len(pending) < nproc * 4 and not stopped:
            if elapsed_min() > budget_min or out_of_time():
                stopped = True
                break
            i = todo.pop()
            try:
                pending[ex.submit(FV.process_video, tasks[i])] = i
            except BrokenProcessPool:
                todo.append(i); broken = True
                break
        if stopped and not pending:
            break
        if not broken and pending:
            fin, _ = cf.wait(list(pending), timeout=20, return_when=cf.FIRST_COMPLETED)
            for f in fin:
                i = pending.pop(f)
                try:
                    results.append(dict(task=tasks[i], res=f.result()))
                    last_done = time.time()
                except BrokenProcessPool:
                    pending[f] = i; broken = True
                except Exception as e:
                    fail(i, f"{type(e).__name__}: {e}")
            if not fin and time.time() - last_done > 60 * watchdog_min:
                warn(f"{label}: no video finished for {watchdog_min:g} min; the extraction processes are restarted")
                broken = True
        if broken:
            for p in list((getattr(ex, "_processes", None) or {}).values()):
                try:
                    p.terminate()
                except Exception:
                    pass
            try:
                ex.shutdown(wait=False, cancel_futures=True)
            except Exception:
                pass
            for f, i in list(pending.items()):
                tries[i] += 1
                if tries[i] >= 2:
                    fail(i, "the extraction process died or hung twice while this video was being processed")
                else:
                    todo.append(i)
            if pending:
                warn(f"{label}: an extraction process stopped; {len(pending)} video(s) in flight are tried again")
            pending.clear()
            ex = new_pool(); last_done = time.time()
        if time.time() - last_log > (30 if DRY_RUN else 300):
            log(f"      {label}: {len(results)}/{len(tasks)} videos, {(time.time() - t_set) / 60:.1f} min")
            last_log = time.time()
    try:
        list(ex.map(FV.flush_worker, range(nproc)))
    except Exception:
        pass
    ex.shutdown(wait=True, cancel_futures=True)
    return results, sorted(set(todo))

def stage_faces():
    if not MODELS:
        raise SkipStage("no model weights, so the crops would not be used")
    kind, model_path = get_face_detector()
    RUN["face_detector"] = dict(kind=kind, model=FV.YUNET_FILE if kind == "yunet" else "haarcascade_frontalface_default.xml",
                                score_threshold=CFG["DET_SCORE"], max_side=CFG["DET_MAX_SIDE"], crop_scale=CFG["CROP_SCALE"], crop_size=CFG["CROP_SIZE"])
    nproc = int(CFG["EXTRACT_PROCESSES"] or max(1, N_CPU))
    global REFS
    REFS = counterpart_paths()
    full = plan_from_rule(CUT_RULES[0])
    if not full:
        raise RuntimeError("no video to extract")
    wcfg = dict(detector=kind, model_path=model_path, score_threshold=CFG["DET_SCORE"], max_side=CFG["DET_MAX_SIDE"],
                crop_scale=CFG["CROP_SCALE"], crop_size=CFG["CROP_SIZE"], memmaps={})
    watchdog = 2.0 if DRY_RUN else 10.0
    def task_of(vr, name, offset, slots, window):
        u = VID.at[int(vr), "uid"]
        return dict(uid=u, path=VID.at[int(vr), "path"], set=name, offset=int(offset), slots=int(slots), window=int(window), vrow=int(vr), ref_path=REFS.get(u))

    saved, saved_from = find_saved_plan()
    if saved:
        lvl = saved["level"]
        rule = next(r for r in CUT_RULES if r[0] == lvl)
        plan = plan_from_rule(rule)
        rows = [dict(level=lvl, rule=rule[1], note=f"reused from an earlier session: {saved_from}", chosen=True)]
        log(f"   extraction plan {lvl} reused from an earlier session ({saved_from}), so that its finished jobs can be reused")
    else:
        # ---- measure: the first videos of each set's random order, in parallel ----
        probe_dir = os.path.join(CROPS_DIR, "_probe"); os.makedirs(probe_dir, exist_ok=True)
        probe_mm = os.path.join(probe_dir, "probe.npy")
        n_probe_each = 2 if DRY_RUN else max(2, nproc)
        probe_tasks, off = [], 0
        for name, p in full.items():
            for j in range(min(n_probe_each, len(p["video_rows"]))):
                t = task_of(p["video_rows"][j], "_probe", off, p["slots"][j], p["window"]); t["probe_set"] = name
                probe_tasks.append(t); off += int(p["slots"][j])
        np.lib.format.open_memmap(probe_mm, mode="w+", dtype=np.uint8, shape=(off, CFG["CROP_SIZE"], CFG["CROP_SIZE"], 3))
        res, _ = run_extraction(probe_tasks, dict(wcfg, memmaps={"_probe": probe_mm}), nproc, 1e9, "measuring", watchdog)
        spv, faces_found, frames_seen = defaultdict(list), 0, 0
        for r in res:
            spv[r["task"]["probe_set"]].append(r["res"]["seconds"])
            for x in r["res"]["rows"]:
                frames_seen += x["face"] != "no frame"; faces_found += x["face"] in ("detected", "guided")
        shutil.rmtree(probe_dir, ignore_errors=True)
        sec_per_video = {k: float(np.median(v)) for k, v in spv.items()}
        face_rate = faces_found / max(frames_seen, 1)
        log(f"   measured on {len(probe_tasks)} videos with {nproc} processes: " + ", ".join(f"{k} {v:.2f} s/video" for k, v in sec_per_video.items())
            + f"; a face was found in {face_rate:.0%} of {frames_seen} frames")
        RUN["extraction_probe"] = dict(sec_per_video=sec_per_video, face_rate=face_rate, frames=frames_seen)
        if not DRY_RUN and (frames_seen == 0 or face_rate < 0.5):
            raise RuntimeError(f"the measuring step decoded {frames_seen} frames and found a face in {face_rate:.0%} of them. Either the videos "
                               "cannot be decoded by this OpenCV build or they are not face videos; check the attached inputs.")
        # ---- choose the cut level ----
        budget_min = CFG["EXTRACT_BUDGET_MIN"] - elapsed_min()
        free_gb = shutil.disk_usage(CROPS_DIR).free / 1e9 - CFG["DISK_RESERVE_GB"]
        rows, chosen = [], None
        for rule in CUT_RULES:
            pl = plan_from_rule(rule)
            minutes = 0.0
            for k, p in pl.items():
                s0 = sec_per_video.get(k, max(sec_per_video.values()))
                minutes += len(p["video_rows"]) * s0 * (0.25 + 0.75 * p["window"] / W0)     # decoding, about 3/4 of the time, scales with the window
            minutes = minutes / nproc / 60.0 * 1.15
            gb = sum(plan_bytes(pl).values()) / 1e9
            fits = minutes <= budget_min and gb <= free_gb
            rows.append(dict(level=rule[0], rule=rule[1], sets=",".join(pl.keys()), videos=int(sum(len(p["video_rows"]) for p in pl.values())),
                             crops=int(sum(int(p["slots"].sum()) for p in pl.values())), disk_gb=round(gb, 2), estimated_minutes=round(minutes, 1),
                             budget_minutes=round(budget_min, 1), free_disk_gb=round(free_gb, 1), fits=fits))
            if fits and chosen is None:
                chosen = rule
        if chosen is None:
            chosen = CUT_RULES[-1]
            warn("even the last cut rule does not fit the time budget or the disk; it is applied and the extraction stops at the budget")
        lvl = chosen[0]
        plan = plan_from_rule(chosen)
        for r in rows:
            r["chosen"] = r["level"] == lvl
        if lvl != "L0":
            warn(f"extraction plan cut to {lvl} ({chosen[1]}) to fit the time budget or the disk; see T03")
        json.dump(dict(level=lvl, protocol_sha256=PROTOCOL_SHA, videos_sha256=videos_sha(), sec_per_video=sec_per_video,
                       written_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")),
                  open(os.path.join(STATE, "extraction_plan.json"), "w"), indent=1)
    pd.DataFrame(rows).to_csv(os.path.join(DIRS["tables"], "T03_extraction_plan.csv"), index=False)
    RUN["extraction_plan"] = dict(level=lvl, sets={k: dict(videos=len(p["video_rows"]), window=p["window"], crops=int(p["slots"].sum())) for k, p in plan.items()},
                                  frames=CFG["FRAMES"], processes=nproc)
    log(f"   plan {lvl}: " + ", ".join(f"{k} {len(p['video_rows'])} videos / {int(p['slots'].sum())} crops (window {p['window']})" for k, p in plan.items()))

    # ---- extract ----
    S = CFG["CROP_SIZE"]
    memmaps = {}
    for name, p in plan.items():
        path = os.path.join(CROPS_DIR, name + ".npy")
        np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8, shape=(int(p["slots"].sum()), S, S, 3))
        memmaps[name] = path
    wcfg["memmaps"] = memmaps
    meta, vinfo, status = [], [], {}
    t0 = time.time()
    for name, p in plan.items():
        offs = np.r_[0, np.cumsum(p["slots"])[:-1]].astype(int)
        tasks = [task_of(vr, name, o, sl, p["window"]) for vr, o, sl in zip(p["video_rows"], offs, p["slots"])]
        t_set = time.time()
        res, not_run = run_extraction(tasks, wcfg, nproc, CFG["EXTRACT_BUDGET_MIN"], name, watchdog)
        for item in res:
            t, r = item["task"], item["res"]
            vinfo.append(dict(uid=r["uid"], set=name, n_frames=r["n_frames"], fps=r["fps"], width=r["width"], height=r["height"], error=r["error"], seconds=r["seconds"],
                              multi_face_frames=r.get("n_multi_face_frames", 0), counterpart_used=bool(r.get("ref_used")), guided_frames=r.get("n_guided", 0),
                              has_counterpart=bool(t.get("ref_path"))))
            for x in r["rows"]:
                meta.append(dict(set=name, row=t["offset"] + x["slot"], uid=t["uid"], vrow=t["vrow"], slot=x["slot"], frame=x["frame"], face=x["face"],
                                 n_faces=x.get("n_faces", 0), score=x["score"], x=x["x"], y=x["y"], w=x["w"], h=x["h"]))
            for x in range(len(r["rows"]), t["slots"]):          # slots without a frame (short or unreadable video)
                meta.append(dict(set=name, row=t["offset"] + x, uid=t["uid"], vrow=t["vrow"], slot=x, frame=-1, face="no frame", n_faces=0,
                                 score=np.nan, x=np.nan, y=np.nan, w=np.nan, h=np.nan))
        done = len(tasks) - len(not_run)
        status[name] = dict(videos=len(tasks), done=done, complete=not not_run, minutes=round((time.time() - t_set) / 60, 1))
        log(f"   {name}: {done}/{len(tasks)} videos in {(time.time() - t_set) / 60:.1f} min")
        if not_run and done == 0:
            warn(f"extraction budget reached before {name} started: nothing is extracted from it or from the later sets")
        elif not_run:
            warn(f"extraction budget reached during {name}: {done} of {len(tasks)} videos done (a random subset, used and flagged); later sets are not extracted")
        if not_run:
            for later in list(plan.keys())[list(plan.keys()).index(name) + 1:]:
                status[later] = dict(videos=len(plan[later]["video_rows"]), done=0, complete=False, minutes=0.0)
            break
    C = pd.DataFrame(meta)
    if not len(C):
        raise RuntimeError("no face crop was extracted")
    C = C.sort_values(["set", "row"]).reset_index(drop=True)
    C["valid"] = C["face"].isin(["detected", "guided", "neighbour"])
    VI = pd.DataFrame(vinfo)
    vcols = VID[["uid", "family", "comp", "video_id", "method", "label", "split", "cluster", "content", "actor_a", "actor_b"]]
    C = C.merge(vcols, on="uid", how="left")
    sets_ok = [k for k, st in status.items() if st["done"] > 0]
    # ---- report ----
    rep = []
    for name in plan:
        c = C[C["set"] == name]; vi = VI[VI["set"] == name] if len(VI) else VI
        nv_none = (c.groupby(["uid", "label"])["valid"].sum() == 0) if len(c) else pd.Series(dtype=bool)
        lab = nv_none.index.get_level_values("label") if len(c) else []
        fr = c[c["face"] != "no frame"] if len(c) else c
        rep.append(dict(set=name, what=SET_DOC[name], videos_planned=len(plan[name]["video_rows"]), videos_done=status[name]["done"], complete=status[name]["complete"],
                        decoding_window=plan[name]["window"], crops=len(c), crops_valid=int(c["valid"].sum()) if len(c) else 0,
                        face_detected_share=round(float(c["face"].isin(["detected", "guided"]).mean()), 4) if len(c) else np.nan,
                        face_from_neighbour_share=round(float((c["face"] == "neighbour").mean()), 4) if len(c) else np.nan,
                        no_face_share=round(float((c["face"] == "none").mean()), 4) if len(c) else np.nan,
                        no_frame_share=round(float((c["face"] == "no frame").mean()), 4) if len(c) else np.nan,
                        frames_with_several_faces_share=round(float((fr["n_faces"] >= 2).mean()), 4) if len(fr) else np.nan,
                        face_chosen_by_difference_share=round(float((c["face"] == "guided").mean()), 4) if len(c) else np.nan,
                        videos_without_face=int(nv_none.sum()) if len(c) else 0,
                        videos_without_face_real=int(nv_none[np.asarray(lab) == 0].sum()) if len(c) else 0,
                        videos_without_face_fake=int(nv_none[np.asarray(lab) == 1].sum()) if len(c) else 0,
                        videos_with_errors=int((vi["error"] != "").sum()) if len(vi) else 0,
                        median_frames_in_video=float(vi["n_frames"].median()) if len(vi) else np.nan, minutes=status[name]["minutes"]))
    rep = pd.DataFrame(rep)
    rep.to_csv(os.path.join(DIRS["tables"], "T04_face_extraction.csv"), index=False)
    for r in rep.itertuples():
        if r.videos_done == 0:
            log(f"   {r.set:13s} 0/{r.videos_planned} videos: not extracted (time budget)")
            continue
        log(f"   {r.set:13s} {r.videos_done}/{r.videos_planned} videos, {r.crops_valid}/{r.crops} crops with a face (found {r.face_detected_share:.1%}, "
            f"from a neighbour frame {r.face_from_neighbour_share:.1%}; several faces in {r.frames_with_several_faces_share:.1%} of frames, decided by the "
            f"difference in {r.face_chosen_by_difference_share:.1%}); {r.videos_without_face} video(s) without a face ({r.videos_without_face_fake} manipulated)")
        if r.videos_without_face > 0.02 * max(r.videos_done, 1) and not DRY_RUN:
            warn(f"{r.set}: {r.videos_without_face} videos without any face ({r.videos_without_face_real} real, {r.videos_without_face_fake} manipulated)")
    # ---- fingerprint of the crops' structure: which frame of which video sits in which row, and whether it has a face ----
    fp = hashlib.sha256(json.dumps(dict(detector=kind, level=lvl), sort_keys=True).encode())
    for name in sets_ok:
        cc = C[C["set"] == name]
        fp.update((name + ":" + "|".join(f"{a},{b},{c},{d}" for a, b, c, d in zip(cc["row"], cc["uid"], cc["frame"], cc["valid"].astype(int)))).encode())
    RUN["crops_fingerprint"] = fp.hexdigest()
    RUN["crop_sets"] = {k: dict(path=memmaps[k], shape=[int(x) for x in np.load(memmaps[k], mmap_mode="r").shape], complete=status[k]["complete"],
                                videos_done=status[k]["done"], videos_planned=status[k]["videos"]) for k in plan}
    C.to_csv(os.path.join(STATE, "crops.csv.gz"), index=False, compression="gzip")
    log(f"   extraction: {len(C)} crops in {len(sets_ok)} set(s), {(time.time() - t0) / 60:.1f} min; structure fingerprint {RUN['crops_fingerprint'][:12]}")
    return dict(C=C, VI=VI, sets=sets_ok, memmaps=memmaps, plan=plan, level=lvl, status=status)

FACES = run_stage("faces", stage_faces, needs=("data", "models"))
CROPS = FACES["C"] if FACES else pd.DataFrame()
SETS_OK = FACES["sets"] if FACES else []
SET_COMPLETE = {k: v["complete"] for k, v in FACES["status"].items()} if FACES else {}

## 6. Detectors

**EfficientNet-B4**, ImageNet-pretrained (`timm`, `efficientnet_b4.ra2_in1k`), one output logit, fine-tuned end to end. The settings are *adapted from* DeepfakeBench's EfficientNet-B4 configuration. The same: Adam, learning rate 2·10⁻⁴, weight decay 5·10⁻⁴, batch 32, at most 10 epochs, and the same five augmentations (flip, rotation, blur, brightness-contrast, JPEG re-compression), each with probability 0.5. Different: crops of 224 px (256 there); timm's weights with their ImageNet normalisation (0.5/0.5 there); 16 real and 4 manipulated frames per training video with a loss weighted inversely to the class sizes (32 frames per video at the natural ratio there), so an epoch is about 23,000 crops and ten epochs about 7,000 optimiser steps (about 36,000 there); and the epoch is the one with the best **validation** video AUC (a tie, for example at 1.0, goes to the lower class-balanced validation loss), with patience 3 (a fixed number of epochs there). Mixed precision on the T4s.

**CLIP ViT-L/14**, frozen (`openai/clip-vit-large-patch14`): the L2-normalised image embedding of every crop, standardised, and a class-balanced logistic regression whose inverse regularisation C is chosen on the validation videos. This is the linear-probe recipe of UnivFD (Ojha et al., CVPR 2023) applied to face crops.

**Training configurations** (each EfficientNet configuration is trained with three seeds; the probe is deterministic):

| name | trained on | purpose |
|---|---|---|
| `main4` | FF++ c23 train split: real + Deepfakes, Face2Face, FaceSwap, NeuralTextures | in-dataset, c40, FaceShifter (unseen) and DFD (cross-dataset) |
| `lomo_<method>` | `main4` without one method, also removed from validation | leave-one-method-out: generalisation to an unseen manipulation |
| `ffdfd` | `main4` + DFD's training actors (originals and fakes, c23) | what DFD adds as training data, measured on DFD's test actors |

The GPU work runs in two worker processes, one per T4, which take jobs from a queue on disk: first the CLIP embeddings, then the EfficientNet runs in a fixed priority order (seed 0 of every configuration before the further seeds). A worker does not start a job that its own measured speed says cannot finish before the time limit, and every finished job is kept in `ffb_state/`, so a second session with this notebook's output attached runs only what is missing. While the workers train, the notebook fits the CLIP probes on the CPU.

In [ ]:
# =============================================================================
# 6a. MODEL WEIGHTS, JOBS AND THE GPU WORKERS
# =============================================================================
QDIR = os.path.join(STATE, "queue")
JOBS = {}            # job_id -> job description
RUNS = {}            # (detector, config, seed) -> dict(val=..., tests={set: (rows, logits)}, info=...)

def rows_of(C, set_name, mask=None):
    c = C[(C["set"] == set_name) & C["valid"]]
    return c if mask is None else c[mask(c)]

def train_rows(C, config):
    meths = ["real"] + config_methods(config)
    parts = [rows_of(C, "ff_c23_train", lambda c: c["method"].isin(meths))]
    if config == "ffdfd":
        if "dfd_c23" not in SETS_OK:
            raise SkipStage("DFD c23 is not available")
        parts.append(rows_of(C, "dfd_c23", lambda c: c["split"] == "dfd_train"))
    return pd.concat(parts, ignore_index=True)

def val_rows(C, config):
    meths = ["real"] + config_methods(config)
    return rows_of(C, "ff_c23_val", lambda c: c["method"].isin(meths))

TEST_SETS = [s for s in ("ff_c23_test", "ff_c40_test", "dfd_c23", "dfd_c40") if s in SETS_OK]

def effnet_job(config, seed, prio):
    jid = f"effnet__{config}__s{seed}"
    tr, va = train_rows(CROPS, config), val_rows(CROPS, config)
    names = sorted(set(tr["set"]) | set(va["set"]) | set(TEST_SETS))
    sid = {n: i for i, n in enumerate(names)}
    vv = {u: i for i, u in enumerate(sorted(va["uid"].unique()))}
    data = dict(train_set=tr["set"].map(sid).values.astype(np.int16), train_rows=tr["row"].values.astype(np.int64), train_labels=tr["label"].values.astype(np.int8),
                val_set=va["set"].map(sid).values.astype(np.int16), val_rows=va["row"].values.astype(np.int64), val_labels=va["label"].values.astype(np.int8),
                val_video=va["uid"].map(vv).values.astype(np.int64))
    n_test = 0
    for t in TEST_SETS:
        te = rows_of(CROPS, t)
        data[f"test_{t}_set"] = np.full(len(te), sid[t], np.int16); data[f"test_{t}_rows"] = te["row"].values.astype(np.int64)
        n_test += len(te)
    np.savez(os.path.join(STATE, "jobs_data", jid + ".npz"), **data)
    e = CFG["EFFNET"]
    return dict(job_id=jid, type="train_effnet", detector="effnet", config=config, seed=int(seed), priority=prio, set_names=names, tests=TEST_SETS,
                n_train=int(len(tr)), n_val=int(len(va)), n_test=int(n_test), epochs=int(e["epochs"]), batch_size=int(e["batch_size"]),
                max_steps_per_epoch=int(e.get("max_steps_per_epoch") or 0), save_checkpoint=(config == "main4"),
                protocol_sha256=PROTOCOL_SHA, crops_fingerprint=RUN.get("crops_fingerprint"))

def clip_jobs(chunk=20000):
    out = []
    for s in SETS_OK:
        n = int(RUN["crop_sets"][s]["shape"][0])
        for a in range(0, n, chunk):
            jid = f"clip__{s}__{a:07d}"
            out.append(dict(job_id=jid, type="clip_features", detector="clip", set=s, row_start=a, row_end=min(n, a + chunk), priority=0,
                            protocol_sha256=PROTOCOL_SHA, crops_fingerprint=RUN.get("crops_fingerprint")))
    return out

def job_done(jid):
    p = os.path.join(STATE, "results", jid, "summary.json")
    if not os.path.exists(p):
        return False
    try:
        s = json.load(open(p))
        j = json.load(open(os.path.join(STATE, "results", jid, "job.json")))
    except Exception:
        return False
    # scores and features are aligned to crop rows: they are reused only if the crops are the same as in this session
    if not (s.get("status") == "ok" and j.get("protocol_sha256") == PROTOCOL_SHA and j.get("crops_fingerprint") == RUN.get("crops_fingerprint")):
        return False
    if j.get("type") == "train_effnet":
        try:
            sc = np.load(os.path.join(STATE, "results", jid, "scores.npz")); d = np.load(os.path.join(STATE, "jobs_data", jid + ".npz"))
            if not np.array_equal(sc["rows_val"], d["val_rows"]):
                return False
            for t in j.get("tests", []):
                if not np.array_equal(sc[f"rows_test_{t}"], d[f"test_{t}_rows"]):
                    return False
        except Exception:
            return False
    return True

def adopt_previous_state():
    """A previous version's output attached as an input: copy its finished jobs into this session's state, if they were
    made under the same protocol and from the same crops (both are checked)."""
    roots = [INPUT_ROOT] + ([os.environ["FFB_PREVIOUS_OUTPUT"]] if os.environ.get("FFB_PREVIOUS_OUTPUT") else [])     # (the variable is a test hook)
    n, seen = 0, 0
    for root in roots:
        if not os.path.isdir(root):
            continue
        base_depth = root.rstrip(os.sep).count(os.sep)
        for dp, dns, fns in os.walk(root):
            if os.path.basename(dp) == "results" and os.path.basename(os.path.dirname(dp)) == os.path.basename(STATE) and os.path.abspath(dp) != os.path.abspath(os.path.join(STATE, "results")):
                for jid in sorted(dns):
                    src = os.path.join(dp, jid)
                    try:
                        j = json.load(open(os.path.join(src, "job.json"))); s = json.load(open(os.path.join(src, "summary.json")))
                    except Exception:
                        continue
                    seen += 1
                    if (s.get("status") == "ok" and j.get("protocol_sha256") == PROTOCOL_SHA and j.get("crops_fingerprint") == RUN.get("crops_fingerprint")
                            and not job_done(jid)):
                        dst = os.path.join(STATE, "results", jid)
                        if os.path.isdir(dst):
                            shutil.rmtree(dst)
                        shutil.copytree(src, dst)
                        n += 1
                dns[:] = []
            elif dp.count(os.sep) - base_depth >= 4:
                dns[:] = []
    if seen:
        log(f"   previous output found: {n} of {seen} finished job(s) adopted" + ("" if n == seen else " (the others were made under another protocol or from other crops, or are already here)"))
    return n

def write_queue(jobs):
    for sub in ("pending", "running", "done", "failed", "stopped"):
        d = os.path.join(QDIR, sub)
        if os.path.isdir(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
    n = 0
    for j in jobs:
        rd = os.path.join(STATE, "results", j["job_id"])
        if job_done(j["job_id"]):                     # finished earlier on the same crops: keep it as it is
            continue
        if os.path.isdir(rd):                         # anything left from an unfinished or incompatible earlier attempt
            shutil.rmtree(rd)
        os.makedirs(rd, exist_ok=True)
        json.dump(j, open(os.path.join(rd, "job.json"), "w"), indent=1)
        json.dump(j, open(os.path.join(QDIR, "pending", f"{j['priority']:03d}_{j['job_id']}.json"), "w"))
        n += 1
    return n

def worker_config(models):
    deadline = T_START + 60.0 * (CFG["HARD_STOP_MIN"] - CFG["EVAL_RESERVE_MIN"])
    cfg = dict(deadline_epoch=deadline, crop_sets={k: dict(path=v["path"], shape=v["shape"]) for k, v in RUN["crop_sets"].items() if k in SETS_OK},
               effnet=CFG["EFFNET"], clip=dict(CFG["CLIP"], path=models.get("clip_path", "")), amp=True, loader_workers=int(CFG["LOADER_WORKERS"]),
               torch_threads=1 if N_GPU else max(1, N_CPU // max(N_WORKERS, 1)))
    json.dump(cfg, open(os.path.join(STATE, "worker_config.json"), "w"), indent=1)
    return cfg

def start_worker(w, models):
    env = dict(os.environ)
    env["PYTHONPATH"] = SRC + os.pathsep + env.get("PYTHONPATH", "")
    env["OMP_NUM_THREADS"] = "1" if N_GPU else str(max(1, N_CPU // max(N_WORKERS, 1)))
    if N_GPU:
        env["CUDA_VISIBLE_DEVICES"] = str(w)
    if models.get("offline"):
        env["HF_HUB_OFFLINE"] = "1"; env["TRANSFORMERS_OFFLINE"] = "1"
    os.makedirs(os.path.join(STATE, "logs"), exist_ok=True)
    out = open(os.path.join(STATE, "logs", f"worker{w}.stdout.txt"), "a")
    return subprocess.Popen([sys.executable, "-u", os.path.join(SRC, "ffb_worker.py"), "--state", STATE, "--worker", str(w), "--device", WORKER_DEVICE],
                            stdout=out, stderr=subprocess.STDOUT, env=env, cwd=STATE)

def queue_counts():
    return {k: len([f for f in os.listdir(os.path.join(QDIR, k)) if f.endswith(".json")]) for k in ("pending", "running", "done", "failed", "stopped")}

PROBE_STATE = {"done": False}

def stage_gpu_jobs():
    if not SETS_OK or "ff_c23_train" not in SETS_OK or "ff_c23_val" not in SETS_OK:
        raise SkipStage("the FF++ training or validation crops are missing")
    os.makedirs(os.path.join(STATE, "jobs_data"), exist_ok=True)
    models = MODELS
    for s_ in ("ff_c23_train", "ff_c23_val"):
        if not SET_COMPLETE.get(s_, False):
            warn(f"{s_} was only partly extracted (time budget): the detectors are trained or selected on a random subset of it")
    adopt_previous_state()
    jobs = clip_jobs() if models["clip_ok"] else []
    prio = 1
    order = [(c, s) for s in CFG["SEEDS"][:1] for c in CFG["CONFIGS"]] + [(c, s) for s in CFG["SEEDS"][1:] for c in CFG["CONFIGS"]]
    told = set()
    for c, s in (order if models["effnet_ok"] else []):
        try:
            jobs.append(effnet_job(c, s, prio)); prio += 1
        except SkipStage as e:
            if c not in told:
                warn(f"configuration {c} not run: {e}"); told.add(c)
    for j in jobs:
        JOBS[j["job_id"]] = j
    worker_config(models)
    n_new = write_queue(jobs)
    stop = os.path.join(STATE, "STOP")
    if os.path.exists(stop):
        os.remove(stop)
    log(f"   {len(jobs)} jobs ({sum(j['type'] == 'clip_features' for j in jobs)} CLIP, {sum(j['type'] == 'train_effnet' for j in jobs)} EfficientNet); "
        f"{len(jobs) - n_new} already finished; {N_WORKERS} worker(s) on {WORKER_DEVICE}")
    if n_new:
        procs = {w: start_worker(w, models) for w in range(N_WORKERS)}
        restarts, retried, handled_exit = defaultdict(int), defaultdict(int), {}
        last = 0.0
        deadline = T_START + 60.0 * (CFG["HARD_STOP_MIN"] - CFG["EVAL_RESERVE_MIN"])
        while True:
            time.sleep(2 if DRY_RUN else 15)
            alive = {w: p for w, p in procs.items() if p.poll() is None}
            qc = queue_counts()
            # the probes, as soon as every CLIP job has finished
            if not PROBE_STATE["done"] and models["clip_ok"] and all(job_done(j["job_id"]) for j in jobs if j["type"] == "clip_features"):
                run_stage("probes", stage_probes)
                PROBE_STATE["done"] = True
            # a worker that died: its unfinished job goes back to the queue once (then it is marked failed), and the worker is started again (twice at most)
            for w, p in list(procs.items()):
                if p.poll() is None or p.returncode == 0 or w in handled_exit.get(p.pid, ()):
                    continue
                handled_exit.setdefault(p.pid, set()).add(w)
                for f in os.listdir(os.path.join(QDIR, "running")):
                    jid = f.split("_", 1)[1][:-5]
                    cp = os.path.join(STATE, "results", jid, "claim.json")
                    try:
                        cl = json.load(open(cp))
                    except Exception:
                        continue
                    if int(cl.get("worker", -1)) != w or int(cl.get("pid", -1)) != p.pid:
                        continue
                    retried[jid] += 1
                    if retried[jid] <= 1:
                        os.replace(os.path.join(QDIR, "running", f), os.path.join(QDIR, "pending", f))
                        warn(f"worker {w} died (exit code {p.returncode}) during {jid}; the job is queued again")
                    else:
                        json.dump(dict(job_id=jid, status="FAILED", error=f"the worker died twice during this job (exit code {p.returncode})", seconds=0),
                                  open(os.path.join(STATE, "results", jid, "summary.json"), "w"))
                        os.replace(os.path.join(QDIR, "running", f), os.path.join(QDIR, "failed", f))
                        warn(f"worker {w} died again during {jid}; the job is marked failed")
                if queue_counts()["pending"] and restarts[w] < 2 and time.time() < deadline:
                    restarts[w] += 1
                    procs[w] = start_worker(w, models)
                    log(f"   worker {w} started again (restart {restarts[w]} of 2)")
            if time.time() - last > (30 if DRY_RUN else 300):
                last = time.time()
                prog = []
                for f in os.listdir(os.path.join(QDIR, "running")):
                    jid = f.split("_", 1)[1][:-5]
                    pp = os.path.join(STATE, "results", jid, "progress.json")
                    if os.path.exists(pp):
                        try:
                            pr = json.load(open(pp)); prog.append(f"{jid} epoch {pr['epoch']}/{pr['of']} (val AUC {pr['val_auc']:.3f})")
                        except Exception:
                            pass
                log(f"      queue: {qc['done']} done, {qc['running']} running, {qc['pending']} waiting, {qc['failed']} failed, {qc['stopped']} stopped"
                    + (" | " + "; ".join(prog) if prog else ""))
            if not any(p.poll() is None for p in procs.values()):
                break
            if time.time() > deadline + 300:                       # the workers stop by themselves at the deadline; this is the backstop
                open(stop, "w").close()
                time.sleep(60)
                for p in procs.values():
                    if p.poll() is None:
                        p.kill()
                warn("GPU workers were stopped at the time limit")
                break
        for w, p in procs.items():
            RUN.setdefault("workers", {})[str(w)] = dict(returncode=p.returncode, restarts=restarts[w])
    if not PROBE_STATE["done"] and models["clip_ok"]:
        run_stage("probes", stage_probes)
        PROBE_STATE["done"] = True
    # ---- collect ----
    rows = []
    for j in jobs:
        jid = j["job_id"]
        sp = os.path.join(STATE, "results", jid, "summary.json")
        s = json.load(open(sp)) if os.path.exists(sp) else {"status": "not run (time limit or queue left over)"}
        rows.append(dict(job_id=jid, type=j["type"], config=j.get("config", ""), seed=j.get("seed", ""), status=s.get("status"),
                         minutes=round(s.get("seconds", 0) / 60, 1), best_epoch=s.get("best_epoch"), best_val_video_auc=s.get("best_val_video_auc"),
                         epochs_run=s.get("epochs_run"), train_images_per_second=s.get("train_images_per_second"), images_per_second=s.get("images_per_second"),
                         error=s.get("error", s.get("reason", ""))))
        if j["type"] == "train_effnet" and s.get("status") == "ok":
            try:
                if not job_done(jid):
                    raise RuntimeError("its stored rows do not match this session's crops")
                sc = np.load(os.path.join(STATE, "results", jid, "scores.npz"))
                d = np.load(os.path.join(STATE, "jobs_data", jid + ".npz"))
                names = j["set_names"]
                RUNS[("effnet", j["config"], int(j["seed"]))] = dict(
                    val=dict(rows=d["val_rows"], sets=[names[i] for i in d["val_set"]], logits=sc["val"]),
                    tests={t: (d[f"test_{t}_rows"], sc["test_" + t]) for t in j["tests"] if "test_" + t in sc.files},
                    info=dict(best_epoch=s.get("best_epoch"), val_auc=s.get("best_val_video_auc"), epochs_run=s.get("epochs_run")),
                    train_content=set(train_rows(CROPS, j["config"])["content"]) | set(val_rows(CROPS, j["config"])["content"]))
            except Exception as e:
                rows[-1]["status"] = "FAILED"; rows[-1]["error"] = f"results not readable: {type(e).__name__}: {str(e)[:150]}"
                warn(f"{jid}: finished, but its results could not be read ({type(e).__name__}: {str(e)[:150]}); left out")
    jt = pd.DataFrame(rows)
    jt.to_csv(os.path.join(DIRS["tables"], "T05_jobs.csv"), index=False)
    bad = jt[jt["status"] != "ok"]
    if len(bad):
        warn(f"{len(bad)} of {len(jt)} GPU jobs did not finish: " + ", ".join(f"{r.job_id} ({r.status})" for r in bad.itertuples()))
    for f in os.listdir(os.path.join(STATE, "logs")):
        shutil.copy(os.path.join(STATE, "logs", f), os.path.join(DIRS["logs"], f))
    return jt

In [ ]:
# =============================================================================
# 6b. CLIP PROBES (CPU; run by the loop above as soon as the embeddings exist)
# =============================================================================
CLIPF = {}

def load_clip_features():
    for s in SETS_OK:
        parts = sorted([j for j in JOBS.values() if j["type"] == "clip_features" and j["set"] == s], key=lambda j: j["row_start"])
        arrs = []
        for j in parts:
            p = os.path.join(STATE, "results", j["job_id"], "features.npy")
            if not os.path.exists(p):
                raise RuntimeError(f"CLIP features of {j['job_id']} are missing")
            arrs.append(np.load(p).astype(np.float32))
        CLIPF[s] = np.concatenate(arrs, 0)
        if len(CLIPF[s]) != RUN["crop_sets"][s]["shape"][0]:
            raise RuntimeError(f"CLIP features of {s}: {len(CLIPF[s])} rows for {RUN['crop_sets'][s]['shape'][0]} crops")

def feats(c):
    return np.concatenate([CLIPF[s][c.loc[c["set"] == s, "row"].values] for s in c["set"].unique()], 0) if len(c) else np.zeros((0, 1))

def stage_probes():
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    load_clip_features()
    out = []
    for config in CFG["CONFIGS"]:
        try:
            tr, va = train_rows(CROPS, config), val_rows(CROPS, config)
        except SkipStage as e:
            log(f"   probe {config}: not fitted ({e})"); continue
        if tr["label"].nunique() < 2 or va["label"].nunique() < 2:
            warn(f"probe {config}: training or validation crops hold one class only; not fitted"); continue
        # keep the row order of each set block, as feats() concatenates per set
        tr = pd.concat([tr[tr["set"] == s] for s in tr["set"].unique()]); va = pd.concat([va[va["set"] == s] for s in va["set"].unique()])
        Xtr, Xva = feats(tr), feats(va)
        sc = StandardScaler().fit(Xtr)
        Xtr_s, Xva_s = sc.transform(Xtr), sc.transform(Xva)
        vv = va["uid"].values
        best = None
        for C in CFG["PROBE_C_GRID"]:
            check_time("CLIP probes")
            clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
            clf.fit(Xtr_s, tr["label"].values)
            zv = clf.decision_function(Xva_s)
            vs = pd.Series(zv).groupby(vv).mean(); vl = pd.Series(va["label"].values).groupby(vv).max()
            a = FM.auc(vl.loc[vs.index].values, vs.values)
            out.append(dict(config=config, C=C, val_video_auc=a, iterations=int(np.max(clf.n_iter_)), converged=bool(np.max(clf.n_iter_) < 3000)))
            if best is None or a > best[0] + 1e-9:
                best = (a, C, clf, zv)
        a, C, clf, zv = best
        tests = {}
        for t in TEST_SETS:
            te = rows_of(CROPS, t)
            tests[t] = (te["row"].values, clf.decision_function(sc.transform(CLIPF[t][te["row"].values])))
        RUNS[("clip", config, 0)] = dict(val=dict(rows=va["row"].values, sets=list(va["set"].values), logits=zv), tests=tests,
                                         info=dict(C=C, val_auc=a), train_content=set(tr["content"]) | set(va["content"]))
        log(f"   probe {config}: C = {C}, validation video AUC {a:.4f}")
    pd.DataFrame(out).to_csv(os.path.join(DIRS["tables"], "T06_probe_selection.csv"), index=False)
    return pd.DataFrame(out)

JOBTAB = run_stage("gpu_jobs", stage_gpu_jobs, needs=("faces",))
if JOBTAB is not None and len(JOBTAB):
    # the stage itself ran, but if some jobs did not, everything drawn from it is incomplete: say so in the figures and the final report
    _failed = JOBTAB[JOBTAB["status"] == "FAILED"]
    _notrun = JOBTAB[JOBTAB["status"] != "ok"].drop(_failed.index)
    if len(_failed):
        RUN["stages"]["gpu_jobs"].update(status="FAILED", error=f"{len(_failed)} of {len(JOBTAB)} GPU jobs failed: " + "; ".join(f"{r.job_id}: {str(r.error)[:120]}" for r in _failed.itertuples()))
    elif len(_notrun):
        RUN["stages"]["gpu_jobs"].update(status="stopped", reason=f"{len(_notrun)} of {len(JOBTAB)} GPU jobs did not run before the time limit; attach this version's output as an input and run again to finish them")
log(f"runs available for evaluation: {len(RUNS)} ({sum(k[0] == 'effnet' for k in RUNS)} EfficientNet, {sum(k[0] == 'clip' for k in RUNS)} CLIP probe)")

In [ ]:
# =============================================================================
# 6c. GRAD-CAM OF THE FINE-TUNED EFFICIENTNET (main4, first seed)
# =============================================================================
def stage_gradcam():
    import timm
    seed = CFG["SEEDS"][0]
    ck = os.path.join(STATE, "results", f"effnet__main4__s{seed}", "model_best_fp16.pt")
    if not os.path.exists(ck):
        raise SkipStage("no checkpoint of main4 (first seed)")
    if "ff_c23_test" not in SETS_OK:
        raise SkipStage("no FF++ c23 test crops")
    model = timm.create_model(CFG["EFFNET"]["arch"], pretrained=False, num_classes=1)
    sd = torch.load(ck, map_location="cpu")
    model.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()})
    dcfg = getattr(model, "pretrained_cfg", {}) or {}
    dev = torch.device("cuda:0" if N_GPU else "cpu")
    model = model.to(dev).eval()
    mean = torch.tensor(dcfg.get("mean", (0.485, 0.456, 0.406)), device=dev).view(1, 3, 1, 1)
    std = torch.tensor(dcfg.get("std", (0.229, 0.224, 0.225)), device=dev).view(1, 3, 1, 1)
    target = getattr(model, "bn2", None) or getattr(model, "conv_head", None)
    if target is None:
        raise SkipStage("no final convolution block found in this architecture")
    store = {}
    def fwd_hook(m, i, o):
        store["a"] = o
        o.register_hook(lambda g: store.__setitem__("g", g))
    h = target.register_forward_hook(fwd_hook)
    mm = np.load(RUN["crop_sets"]["ff_c23_test"]["path"], mmap_mode="r")
    c = CROPS[(CROPS["set"] == "ff_c23_test") & CROPS["valid"]]
    picks = []
    for m in ["real"] + ALL_METHODS:
        uids = sorted(c.loc[c["method"] == m, "uid"].unique())[:2]
        for u in uids:
            cu = c[c["uid"] == u].sort_values("slot")
            picks.append((m, u, int(cu["row"].values[len(cu) // 2])))
    out = []
    try:
        for m, u, row in picks:
            img = np.asarray(mm[row])
            x = torch.from_numpy(img).to(dev).permute(2, 0, 1)[None].float().div(255.0)
            x = ((x - mean) / std).requires_grad_(False)
            model.zero_grad(set_to_none=True)
            with torch.enable_grad():
                z = model(x).reshape(-1)[0]
                z.backward()
            a, g = store["a"].detach(), store["g"].detach()
            cam = torch.relu((g.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True))
            cam = torch.nn.functional.interpolate(cam, size=img.shape[:2], mode="bilinear", align_corners=False)[0, 0]
            cam = (cam / cam.max().clamp_min(1e-12)).cpu().numpy()
            out.append(dict(method=m, uid=u, row=row, logit=float(z.detach()), prob_fake=float(torch.sigmoid(z.detach())), img=img, cam=cam))
    finally:
        h.remove()
    del model
    if N_GPU:
        torch.cuda.empty_cache()
    pd.DataFrame([{k: v for k, v in r.items() if k not in ("img", "cam")} for r in out]).to_csv(os.path.join(DIRS["tables"], "T14_gradcam_examples.csv"), index=False)
    return out

GRADCAM = run_stage("gradcam", stage_gradcam, needs=("gpu_jobs",), late_ok=True)

## 7. Evaluation

**Video score** = mean of the frame logits of a video's face crops. **Primary metric**: video-level AUC, with 95% intervals. The other metrics are point estimates (with their spread over seeds): frame-level AUC, average precision, equal error rate, and accuracy, balanced accuracy, true positive rate (fakes caught) and true negative rate (real videos passed) at the threshold that maximised balanced accuracy on the run's **validation** videos, and the expected calibration error of the sigmoid of the video score. AP, accuracy and calibration error depend on the ratio of real to manipulated videos, which differs between test sets (1:1 for a single method, 1:4 for the four classic methods, about 1:8.5 for all of DFD); every row of the result table carries its numbers of real and manipulated videos.

**Test sets.** `FF++ c23 | <method>`: the 140 real test videos against the 140 test videos of one method. `FF++ c23 | four classic methods`: real against Deepfakes, Face2Face, FaceSwap and NeuralTextures together. The same at c40. `DFD | all`: all actors' originals against all fakes; `DFD | test actors`: only the actors that the `ffdfd` runs never saw. `FF++ … | as many frames as DFD`: the four-classic-methods test with each video scored from as many frames as a DFD video, for a fair FF++-versus-DFD comparison. A (run, test set) pair is skipped if any of its videos, at any compression, was used to train or validate that run.

**Uncertainty.** The test videos are resampled 2,000 times by cluster: FF++ test videos by source pair (a pair's two real videos and every fake made from them move together); DFD videos by actor, two-way, because a DFD fake involves two actors (an actor is drawn n times; an original of that actor gets weight n, a fake of actors A and B weight n_A · n_B). Two intervals are given for EfficientNet: with the three trained models held fixed (`auc_ci_*_models_fixed`), and with the seeds resampled as well (`auc_ci_*`, the one drawn in the figures), which also reflects training randomness, as far as three seeds can. Differences between detectors, configurations or compressions (table T09) use the same resamples for both sides where the videos are the same, resample the seeds as well, list the per-seed differences and whether they agree in sign, and give the number of comparisons made. No p-values are computed and no correction for multiple comparisons is applied: an interval that excludes 0 is reported as such and nothing more is claimed.

In [ ]:
# =============================================================================
# 7a. METRICS FOR EVERY RUN AND TEST SET
# =============================================================================
def _frames_of(set_name):
    """Frames planned per video in a crop set (None if the set was not planned)."""
    try:
        sl = FACES["plan"][set_name]["slots"]
        return int(sl[0]) if len(sl) else None
    except Exception:
        return None
DFD_FRAMES = _frames_of("dfd_c23")
FF_TEST_FRAMES = _frames_of("ff_c23_test")

def target_list():
    T = []
    for comp, s in (("c23", "ff_c23_test"), ("c40", "ff_c40_test")):
        if s in TEST_SETS:
            T.append(dict(target=f"FF++ {comp} | four classic methods", set=s, methods=["real"] + CLASSIC, family="FF++", comp=comp, kind="ffpp_classic"))
            for m in ALL_METHODS:
                T.append(dict(target=f"FF++ {comp} | {m}", set=s, methods=["real", m], family="FF++", comp=comp, kind="ffpp_method", method=m))
            if DFD_FRAMES and FF_TEST_FRAMES and DFD_FRAMES < FF_TEST_FRAMES and any(x in TEST_SETS for x in ("dfd_c23", "dfd_c40")):
                T.append(dict(target=f"FF++ {comp} | four classic methods | as many frames as DFD", set=s, methods=["real"] + CLASSIC, family="FF++", comp=comp,
                              kind="ffpp_classic_matched", frames=DFD_FRAMES))
    for comp, s in (("c23", "dfd_c23"), ("c40", "dfd_c40")):
        if s in TEST_SETS:
            T.append(dict(target=f"DFD {comp} | all", set=s, splits=None, family="DFD", comp=comp, kind="dfd_all"))
            T.append(dict(target=f"DFD {comp} | test actors", set=s, splits=["dfd_test"], family="DFD", comp=comp, kind="dfd_test"))
    return T

def spread_pick(c, k):
    """Keep k crops per video, spread evenly over its crops in slot order (all of them if it has k or fewer)."""
    c = c.sort_values(["uid", "slot"]).reset_index(drop=True)
    pos = c.groupby("uid").cumcount().values; n_of = c.groupby("uid")["slot"].transform("size").values
    sel = np.zeros(len(c), bool)
    for n in np.unique(n_of):
        keep = np.arange(n) if n <= k else np.unique(np.floor((np.arange(k) + 0.5) * n / k).astype(int))
        sel |= (n_of == n) & np.isin(pos, keep)
    return c[sel]

def target_videos(t):
    c = CROPS[(CROPS["set"] == t["set"]) & CROPS["valid"]]
    if "methods" in t:
        c = c[c["method"].isin(t["methods"])]
    if t.get("splits"):
        c = c[c["split"].isin(t["splits"])]
    if t.get("frames"):
        c = spread_pick(c, int(t["frames"]))
    v = c.groupby("uid").agg(label=("label", "first"), method=("method", "first"), cluster=("cluster", "first"), content=("content", "first"),
                             actor_a=("actor_a", "first"), actor_b=("actor_b", "first")).reset_index()
    return v.sort_values("uid").reset_index(drop=True), c

ROWINFO = {s: CROPS[CROPS["set"] == s].set_index("row")[["uid", "label", "valid", "slot"]] for s in CROPS["set"].unique()} if len(CROPS) else {}
BOOT_W, N_CLUSTERS = {}, {}
EVAL_ROWS, BOOTS, VSCORES = [], {}, {}

def boot_keys(t, v):
    if t["family"] == "DFD":                       # two-way: a fake involves two actors
        return [(a,) if lab == 0 or a == b else (a, b) for a, b, lab in zip(v["actor_a"], v["actor_b"], v["label"])]
    return [(c,) for c in v["cluster"]]

def boot_weights(t, v):
    if t["target"] not in BOOT_W:
        seed = int(hashlib.md5(t["target"].encode()).hexdigest()[:8], 16)
        keys = boot_keys(t, v)
        BOOT_W[t["target"]] = FM.multiway_bootstrap_weights(keys, int(CFG["N_BOOT"]), seed=seed)
        N_CLUSTERS[t["target"]] = len(set(x for k in keys for x in k))
    return BOOT_W[t["target"]]

def run_val_threshold(run):
    va = run["val"]
    df = pd.DataFrame(dict(set=va["sets"], row=va["rows"], z=va["logits"]))
    df["uid"] = [ROWINFO[s].at[r, "uid"] for s, r in zip(df["set"], df["row"])]
    df["label"] = [ROWINFO[s].at[r, "label"] for s, r in zip(df["set"], df["row"])]
    g = df.groupby("uid").agg(z=("z", "mean"), label=("label", "first"))
    return FM.best_threshold(g["label"].values, g["z"].values), FM.auc(g["label"].values, g["z"].values)

def stage_evaluate():
    if not RUNS:
        raise SkipStage("no finished detector run")
    T = target_list()
    tv = {t["target"]: target_videos(t) for t in T}
    for (det, config, seed), run in sorted(RUNS.items()):
        check_time("evaluation", late=True)
        thr, val_auc = run_val_threshold(run)
        for t in T:
            if t["set"] not in run["tests"]:
                continue
            v, c = tv[t["target"]]
            if not len(v) or v["label"].nunique() < 2:
                continue
            leak = set(v["content"]) & run["train_content"]
            if leak:
                EVAL_ROWS.append(dict(detector=det, config=config, seed=seed, target=t["target"], skipped=f"{len(leak)} video(s) used in training or validation"))
                continue
            rows, z = run["tests"][t["set"]]
            zs = pd.Series(np.asarray(z, np.float64), index=np.asarray(rows))
            fz = zs.reindex(c["row"].values).values
            fr_auc, fr_eer = FM.auc(c["label"].values, fz), FM.eer(c["label"].values, fz)
            vid = pd.Series(fz).groupby(c["uid"].values).mean().reindex(v["uid"]).values
            m = FM.summary(v["label"].values, vid, threshold=thr)
            m.update(detector=det, config=config, seed=seed, target=t["target"], family=t["family"], comp=t["comp"], kind=t["kind"], method=t.get("method", ""),
                     frames_per_video=int(t.get("frames") or c.groupby("uid").size().max()),
                     frame_auc=fr_auc, frame_eer=fr_eer, ece=FM.ece(v["label"].values, FM.sigmoid(vid), CFG["ECE_BINS"]),
                     val_threshold=thr, val_video_auc=val_auc, n_frames=int(len(c)), skipped="")
            EVAL_ROWS.append(m)
            W = boot_weights(t, v)
            BOOTS[(det, config, seed, t["target"])] = FM.weighted_auc(v["label"].values, vid, W)
            VSCORES[(det, config, seed, t["target"])] = pd.DataFrame(dict(uid=v["uid"].values, label=v["label"].values, method=v["method"].values,
                                                                          cluster=v["cluster"].values, content=v["content"].values,
                                                                          actor_a=v["actor_a"].values, actor_b=v["actor_b"].values, score=vid))
    E = pd.DataFrame(EVAL_ROWS)
    E.to_csv(os.path.join(DIRS["results"], "metrics_per_run.csv"), index=False)
    return E

def ci(vals):
    return FM.percentile_interval(vals, CFG["CI_LEVEL"])

def boot_matrix(det, config, target):
    """(seeds x resamples) bootstrap AUCs of every finished seed, and the seeds in that order."""
    ks = sorted(k for k in BOOTS if k[0] == det and k[1] == config and k[3] == target)
    return (np.vstack([BOOTS[k] for k in ks]), [k[2] for k in ks]) if ks else (None, [])

def draw_seed(name):
    return int(hashlib.md5(name.encode()).hexdigest()[:8], 16)

try:                                                           # target -> crop set, to flag a test set cut short by the time budget
    SET_OF = {t["target"]: t["set"] for t in target_list()}
except Exception:
    SET_OF = {}

def stage_summarise():
    E = ST.get("evaluate")
    if E is None or not len(E):
        raise SkipStage("nothing evaluated")
    E = E[E["skipped"] == ""].copy()
    metric_cols = ["auc", "ap", "eer", "frame_auc", "frame_eer", "accuracy_at_val_threshold", "balanced_accuracy_at_val_threshold",
                   "tpr_at_val_threshold", "tnr_at_val_threshold", "ece"]
    rows = []
    for (det, config, target), g in E.groupby(["detector", "config", "target"], sort=False):
        r = dict(detector=det, config=config, target=target, family=g["family"].iat[0], comp=g["comp"].iat[0], kind=g["kind"].iat[0], method=g["method"].iat[0],
                 n_seeds=len(g), n_videos=int(g["n"].iat[0]), n_real=int(g["n_real"].iat[0]), n_fake=int(g["n_fake"].iat[0]),
                 frames_per_video=int(g["frames_per_video"].iat[0]), n_bootstrap_clusters=N_CLUSTERS.get(target),
                 test_set_complete=bool(SET_COMPLETE.get(SET_OF.get(target), True)))
        for mc in metric_cols:
            vals = g[mc].values.astype(float)
            r[mc] = float(np.nanmean(vals)); r[mc + "_sd"] = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else np.nan
            r[mc + "_min"] = float(np.nanmin(vals)); r[mc + "_max"] = float(np.nanmax(vals))
        M, seeds = boot_matrix(det, config, target)
        if M is not None:
            r["auc_ci_lo_models_fixed"], r["auc_ci_hi_models_fixed"] = ci(M.mean(0))
            h, _ = FM.hierarchical_mean(M, seed=draw_seed(det + config + target))
            r["auc_ci_lo"], r["auc_ci_hi"] = ci(h)
        else:
            r["auc_ci_lo"] = r["auc_ci_hi"] = r["auc_ci_lo_models_fixed"] = r["auc_ci_hi_models_fixed"] = np.nan
        rows.append(r)
    S = pd.DataFrame(rows)
    S.to_csv(os.path.join(DIRS["tables"], "T07_results.csv"), index=False)
    piv = S.assign(cell=lambda d: d.apply(lambda r: f"{r['auc']:.3f} [{r['auc_ci_lo']:.3f}, {r['auc_ci_hi']:.3f}]", axis=1)).pivot_table(
        index="target", columns=["detector", "config"], values="cell", aggfunc="first")
    piv.to_csv(os.path.join(DIRS["tables"], "T08_video_auc_overview.csv"))
    return S

def video_matrix(det, config, target, common=None):
    """Per seed, the video scores of a target (optionally restricted to, and ordered by, a list of contents)."""
    ks = sorted(k for k in VSCORES if k[0] == det and k[1] == config and k[3] == target)
    out = []
    for k in ks:
        x = VSCORES[k]
        out.append(x.set_index("content").loc[common].reset_index() if common is not None else x)
    return out, [k[2] for k in ks]

def compare(det_a, cfg_a, tgt_a, det_b, cfg_b, tgt_b, label, how):
    """Difference of video AUC, a - b. how: 'same videos' (shared resamples), 'same content' (two compressions of the
    same videos: rescored on the videos present at both, shared resamples) or 'independent' (different videos)."""
    same_models = det_a == det_b and cfg_a == cfg_b
    if how == "same content":
        va, sa = video_matrix(det_a, cfg_a, tgt_a); vb, sb = video_matrix(det_b, cfg_b, tgt_b)
        if not va or not vb:
            return None
        common = sorted(set(va[0]["content"]) & set(vb[0]["content"]))
        if len(common) < 10:
            return None
        va, _ = video_matrix(det_a, cfg_a, tgt_a, common); vb, _ = video_matrix(det_b, cfg_b, tgt_b, common)
        W = FM.multiway_bootstrap_weights(boot_keys(dict(family="DFD" if tgt_a.startswith("DFD") else "FF++"), va[0]), int(CFG["N_BOOT"]), seed=draw_seed(tgt_a + tgt_b))
        A = np.vstack([FM.weighted_auc(x["label"].values, x["score"].values, W) for x in va])
        B = np.vstack([FM.weighted_auc(x["label"].values, x["score"].values, W) for x in vb])
        pa_s = [FM.auc(x["label"].values, x["score"].values) for x in va]; pb_s = [FM.auc(x["label"].values, x["score"].values) for x in vb]
        note = f"same videos at both compressions ({len(common)}), shared resamples"
    else:
        A, sa = boot_matrix(det_a, cfg_a, tgt_a); B, sb = boot_matrix(det_b, cfg_b, tgt_b)
        if A is None or B is None:
            return None
        E = ST["evaluate"]; E = E[E["skipped"] == ""]
        pa_s = [float(E[(E["detector"] == det_a) & (E["config"] == cfg_a) & (E["target"] == tgt_a) & (E["seed"] == s)]["auc"].iat[0]) for s in sa]
        pb_s = [float(E[(E["detector"] == det_b) & (E["config"] == cfg_b) & (E["target"] == tgt_b) & (E["seed"] == s)]["auc"].iat[0]) for s in sb]
        note = "same videos, shared resamples" if how == "same videos" else "different videos, independent resamples"
    ha, draws = FM.hierarchical_mean(A, seed=draw_seed(label + tgt_a + cfg_a))
    hb, _ = FM.hierarchical_mean(B, draws=draws if (same_models and A.shape[0] == B.shape[0]) else None, seed=draw_seed(label + tgt_b + cfg_b + "b"))
    d = ha - hb
    lo, hi = ci(d)
    lo_f, hi_f = ci(A.mean(0) - B.mean(0))
    if same_models and sa == sb:                       # the same trained models on both sides: per-seed paired differences
        diffs = [x - y for x, y in zip(pa_s, pb_s)]
    else:                                              # different trainings: every combination of a seed of a and a seed of b
        diffs = [x - y for x in pa_s for y in pb_s]
    return dict(comparison=label, a=f"{det_a}/{cfg_a} on {tgt_a}", b=f"{det_b}/{cfg_b} on {tgt_b}", auc_a=float(np.mean(pa_s)), auc_b=float(np.mean(pb_s)),
                difference=float(np.mean(pa_s) - np.mean(pb_s)), ci_lo=lo, ci_hi=hi, ci_lo_models_fixed=lo_f, ci_hi_models_fixed=hi_f,
                excludes_zero=bool(np.isfinite(lo) and (lo > 0 or hi < 0)),
                seed_differences=";".join(f"{x:+.4f}" for x in diffs), signs_agree=bool(len(diffs) > 0 and (all(x > 0 for x in diffs) or all(x < 0 for x in diffs))),
                resampling=note + ("; seeds resampled" if max(A.shape[0], B.shape[0]) > 1 else ""), seeds_a=len(pa_s), seeds_b=len(pb_s))

def stage_compare():
    S = ST.get("summarise")
    if S is None:
        raise SkipStage("no summary")
    out = []
    have = lambda det, cfg, tgt: ((S["detector"] == det) & (S["config"] == cfg) & (S["target"] == tgt)).any()
    targets = sorted(S["target"].unique())
    for t in targets:                                                       # 1. CLIP probe vs EfficientNet
        if have("clip", "main4", t) and have("effnet", "main4", t):
            out.append(compare("clip", "main4", t, "effnet", "main4", t, "CLIP probe - EfficientNet (main4)", "same videos"))
    for det in ("effnet", "clip"):                                          # 2. compression, the same videos
        pairs = [("FF++ c40 | four classic methods", "FF++ c23 | four classic methods"), ("DFD c40 | all", "DFD c23 | all")] + \
                [(f"FF++ c40 | {m}", f"FF++ c23 | {m}") for m in ALL_METHODS]
        for a, b in pairs:
            if have(det, "main4", a) and have(det, "main4", b):
                out.append(compare(det, "main4", a, det, "main4", b, f"c40 - c23 ({det}, main4)", "same content"))
    for det in ("effnet", "clip"):                                          # 3. leave one method out
        for m in CLASSIC:
            for comp in ("c23", "c40"):
                t = f"FF++ {comp} | {m}"
                if have(det, f"lomo_{m}", t) and have(det, "main4", t):
                    out.append(compare(det, f"lomo_{m}", t, det, "main4", t, f"unseen - seen method ({det}, {comp})", "same videos"))
    for det in ("effnet", "clip"):                                          # 4. DFD as additional training data
        for t in ("DFD c23 | test actors", "DFD c40 | test actors", "FF++ c23 | four classic methods", "FF++ c40 | four classic methods"):
            if have(det, "ffdfd", t) and have(det, "main4", t):
                out.append(compare(det, "ffdfd", t, det, "main4", t, f"ffdfd - main4 ({det})", "same videos"))
    for det in ("effnet", "clip"):                                          # 5. cross-dataset gap, at the same number of frames per video
        for comp in ("c23", "c40"):
            a = f"DFD {comp} | all"
            b = f"FF++ {comp} | four classic methods | as many frames as DFD"
            if not have(det, "main4", b):
                b = f"FF++ {comp} | four classic methods"
            if have(det, "main4", a) and have(det, "main4", b):
                out.append(compare(det, "main4", a, det, "main4", b, f"DFD - FF++ ({det}, {comp})", "independent"))
    D = pd.DataFrame([r for r in out if r])
    if len(D):
        D["comparisons_in_this_table"] = len(D)
    D.to_csv(os.path.join(DIRS["tables"], "T09_comparisons.csv"), index=False)
    return D

def stage_frames_curve():
    rows = []
    for (det, config, seed), run in RUNS.items():
        if config != "main4":
            continue
        for tname in ("FF++ c23 | four classic methods", "DFD c23 | all", "FF++ c40 | four classic methods"):
            t = next((x for x in target_list() if x["target"] == tname), None)
            if t is None or t["set"] not in run["tests"]:
                continue
            v, c = target_videos(t)
            rws, z = run["tests"][t["set"]]
            zs = pd.Series(np.asarray(z, np.float64), index=np.asarray(rws))
            c = c.assign(z=zs.reindex(c["row"].values).values)
            k_max = int(c.groupby("uid").size().max())
            for k in [k for k in CFG["FRAMES_CURVE"] if k <= k_max]:       # no more frames than were extracted
                sub = spread_pick(c, k)
                vs = sub.groupby("uid")["z"].mean().reindex(v["uid"]).values
                rows.append(dict(detector=det, config=config, seed=seed, target=tname, frames=k, auc=FM.auc(v["label"].values, vs),
                                 videos_with_fewer=int((c.groupby("uid").size() < k).sum())))
    F = pd.DataFrame(rows)
    if len(F):
        F.to_csv(os.path.join(DIRS["tables"], "T10_frames_per_video.csv"), index=False)
    return F

def stage_calibration():
    rows = []
    for (det, config, seed, tname), v in VSCORES.items():
        if config != "main4" or seed != min(k[2] for k in VSCORES if k[0] == det):
            continue
        if tname not in ("FF++ c23 | four classic methods", "DFD c23 | all", "FF++ c40 | four classic methods", "DFD c40 | all"):
            continue
        p = FM.sigmoid(v["score"].values)
        e = FM.ece(v["label"].values, p, CFG["ECE_BINS"])
        for b in FM.reliability_bins(v["label"].values, p, CFG["ECE_BINS"]):
            rows.append(dict(detector=det, config=config, seed=seed, target=tname, ece_this_seed=e, **b))
    R = pd.DataFrame(rows)
    if len(R):
        R.to_csv(os.path.join(DIRS["tables"], "T11_reliability_bins.csv"), index=False)
    return R

def stage_training_history():
    rows = []
    for j in JOBS.values():
        if j["type"] != "train_effnet":
            continue
        p = os.path.join(STATE, "results", j["job_id"], "history.csv")
        if os.path.exists(p) and job_done(j["job_id"]):
            h = pd.read_csv(p); h["config"] = j["config"]; h["seed"] = j["seed"]; rows.append(h)
    H = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if len(H):
        H.to_csv(os.path.join(DIRS["tables"], "T12_training_history.csv"), index=False)
    return H

# These take seconds to a few minutes, so they may run until the later time limit (FINAL_STOP_MIN)
run_stage("evaluate", stage_evaluate, needs=("gpu_jobs",), late_ok=True)
run_stage("summarise", stage_summarise, needs=("evaluate",), late_ok=True)
run_stage("compare", stage_compare, needs=("summarise",), late_ok=True)
run_stage("frames_curve", stage_frames_curve, needs=("evaluate",), late_ok=True)
run_stage("calibration", stage_calibration, needs=("evaluate",), late_ok=True)
run_stage("history", stage_training_history, needs=("gpu_jobs",), late_ok=True)
if ST.get("summarise") is not None:
    _S = ST["summarise"]
    for _r in _S[(_S["config"] == "main4") & (_S["kind"].isin(["ffpp_classic", "dfd_all"]))].itertuples():
        log(f"   {_r.detector:6s} main4 | {_r.target:34s} video AUC {_r.auc:.4f} [{_r.auc_ci_lo:.4f}, {_r.auc_ci_hi:.4f}]"
            + (f"  seeds {_r.n_seeds}: {_r.auc_min:.4f}-{_r.auc_max:.4f}" if _r.n_seeds > 1 else ""))

## 8. Figures

Every figure is written as a 600 DPI PNG and a vector PDF (fonts embedded), together with a CSV of the numbers it plots (`tables/figure_data/`). Blue is the fine-tuned EfficientNet, orange the CLIP probe. Whiskers are 95% bootstrap intervals that resample the test clusters and, for EfficientNet, the three seeds; small hollow markers are the individual EfficientNet seeds. A figure that draws on a stage that did not finish says so in red under the plot.

In [ ]:
# =============================================================================
# 8a. STYLE AND HELPERS
# =============================================================================
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
C_EFF, C_CLIP, C_REAL, C_FAKE = "#2a78d6", "#eb6834", "#1baf7a", "#d03b3b"
DET_COLOR = {"effnet": C_EFF, "clip": C_CLIP}
DET_LABEL = {"effnet": "EfficientNet-B4 (fine-tuned)" if not DRY_RUN else "EfficientNet (rehearsal)", "clip": "CLIP ViT-L/14 probe" if not DRY_RUN else "CLIP probe (rehearsal)"}
METHOD_COLOR = {"real": "#1baf7a", "Deepfakes": "#2a78d6", "Face2Face": "#eda100", "FaceSwap": "#9b59b6", "NeuralTextures": "#eb6834", "FaceShifter": "#52514e", "DFD": "#d03b3b"}
_RC = {
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 8.5, "axes.labelsize": 7.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "axes.edgecolor": AXIS, "axes.linewidth": 0.6, "axes.labelcolor": INK2, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.5, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True, "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "pdf.fonttype": 42, "ps.fonttype": 42, "lines.linewidth": 1.5, "lines.markersize": 5.5, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "figure.dpi": 110,
}
for _k, _v in _RC.items():
    try:
        plt.rcParams[_k] = _v
    except Exception as _e:
        log(f"   matplotlib setting {_k} not applied ({type(_e).__name__})")
FIGS = []
CAPTIONS = {
    "F01_data": "Videos found per dataset, compression and method, next to the published counts (a); outcome of face detection for every sampled frame, per crop set (b).",
    "F02_example_crops": "Face crops as the detectors see them (224 x 224, 1.3 x the face box): two test videos per class, at c23 and, where available, the same frame of the same video at c40.",
    "F03_roc_ffpp": "Video-level ROC curves on the FF++ test split, real videos against the four classic methods, at c23 and c40; detectors trained on main4. Thin lines: individual EfficientNet seeds.",
    "F04_auc_per_method": "Video-level AUC on the FF++ test split for each manipulation method against the real videos; detectors trained on main4 (FaceShifter is never seen in training). Whiskers: 95% bootstrap intervals (test clusters and, for EfficientNet, the three seeds resampled); hollow markers: EfficientNet seeds.",
    "F05_unseen_method": "Leave-one-method-out: video AUC on a method's test videos when the method was in training (main4) and when it was held out of training and validation (lomo). Whiskers: 95% bootstrap intervals (test clusters and seeds).",
    "F06_compression": "Video AUC of the main4 detectors on the test videos at c23 and at c40 (each point uses the videos in which a face was found at that compression; the paired differences in T09 use the videos present at both). Whiskers: 95% bootstrap intervals.",
    "F07_cross_dataset": "Cross-dataset test: video-level ROC on all of DFD (c23) for detectors trained on FF++ (a), and video AUC on FF++ (scored from as many frames per video as DFD) and on DFD, at both compressions (b). Whiskers: 95% bootstrap intervals (FF++ by source pair, DFD two-way by actor; seeds resampled).",
    "F08_dfd_as_training_data": "DFD as additional training data: video AUC of detectors trained on FF++ only (main4) and on FF++ plus DFD's training actors (ffdfd), on DFD's test actors and on FF++. Whiskers: 95% bootstrap intervals.",
    "F09_gradcam": "Grad-CAM of the fine-tuned EfficientNet (main4, first seed) on FF++ c23 test crops, two videos per class; the number is the predicted probability of 'fake'.",
    "F10_training_curves": "EfficientNet training: training loss, validation video AUC and class-balanced validation loss per epoch, every configuration and seed; the marker is the selected epoch (the best validation AUC, a tie going to the lower validation loss).",
    "F11_frames_per_video": "Video AUC as a function of the number of frames averaged per video (frames spread evenly over those extracted; at most as many as were extracted); main4 detectors, mean over EfficientNet seeds, band: range of the seeds.",
    "F12_score_distributions": "Distribution of the video-level probability of 'fake' for real and manipulated videos; main4 detectors, first seed.",
    "F13_calibration": "Reliability diagrams of the video-level probability of 'fake' (main4, first seed; 15 equal-width bins, empty bins not drawn) in-dataset and across datasets, with the expected calibration error of the same seed and bins. Both detectors were trained with class-balancing weights.",
    "F14_auc_overview": "Video AUC of every detector and training configuration on every test set (mean over seeds). Grey: not evaluated (training overlap or set not available).",
}
N_FIGURES_PLANNED = len(CAPTIONS)
try:                                   # absorb one-time backend initialisation problems of some IPython/matplotlib pairs before the first real figure
    plt.close(plt.figure())
except Exception as _e:
    log(f"   matplotlib warm-up: {type(_e).__name__}: {_e}")
IN_NOTEBOOK = False
try:
    from IPython import get_ipython as _gip
    IN_NOTEBOOK = _gip() is not None
except Exception:
    pass

STATUS_WORD = {"FAILED": "failed", "stopped": "was stopped at the time limit", "skipped": "was skipped"}

def unfinished(stages):
    out = []
    for k in stages:
        st = RUN["stages"].get(k, {}).get("status")
        if st is not None and st != "ok":
            out.append((k, STATUS_WORD.get(st, st)))
    return out

def save_fig(fig, name, data=None, note="", uses=()):
    """PNG at 600 DPI + vector PDF + the plotted numbers as CSV."""
    if MODE != "full":
        txt = "DRY RUN\nsynthetic videos, random networks\nnot a result" if DRY_RUN else "QUICK TEST\nsmall subset, 1 seed, 2 epochs\nnot a result"
        wm = fig.text(0.5, 0.5, txt, ha="center", va="center", fontsize=20, color="#b0b0b0", alpha=0.35, rotation=24, zorder=100, fontweight="bold")
        wm.set_in_layout(False)
    part = unfinished(uses)
    if part:
        msg = "INCOMPLETE: " + "; ".join(f"stage '{k}' {w}" for k, w in part) + ". Conditions that did not run are missing from this figure."
        try:                                             # below everything already drawn, legends under the axes included
            fig.canvas.draw()
            y = min(0.0, fig.get_tightbbox(fig.canvas.get_renderer()).y0 / fig.get_figheight()) - 0.02
        except Exception:
            y = -0.02
        fig.text(0.5, y, msg, ha="center", va="top", fontsize=7, color="#b02a2a", fontweight="bold", wrap=True)
        note = (note + "; " if note else "") + msg
    png = os.path.join(DIRS["figures/png"], name + ".png"); pdf = os.path.join(DIRS["figures/pdf"], name + ".pdf")
    fig.savefig(png, dpi=int(CFG["DPI"]), bbox_inches="tight", pad_inches=0.05)
    fig.savefig(pdf, dpi=int(CFG["DPI"]), bbox_inches="tight", pad_inches=0.05)
    if IN_NOTEBOOK:
        try:
            import io
            from IPython.display import Image, display
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=110, bbox_inches="tight", pad_inches=0.05)
            print(f"{name}  (preview at 110 DPI; the {CFG['DPI']} DPI PNG and the PDF are in the ZIP)")
            display(Image(data=buf.getvalue()))
        except Exception:
            pass
    csv = ""
    if data is not None and len(data):
        os.makedirs(os.path.join(DIRS["tables"], "figure_data"), exist_ok=True)
        csv = os.path.join(DIRS["tables"], "figure_data", name + ".csv")
        data.to_csv(csv, index=False)
    FIGS[:] = [f for f in FIGS if f["name"] != name]
    FIGS.append(dict(name=name, png=png, pdf=pdf, data=csv, caption=CAPTIONS.get(name, ""), note=note, complete=not part))
    plt.close(fig)

def make_fig(name, fn, needs=()):
    """One figure per fenced call: a failed or impossible figure never blocks the others."""
    missing = [k for k in needs if not (RUN["ok"].get(k) or RUN["stages"].get(k, {}).get("status") == "stopped")]
    if missing:
        RUN["stages"]["fig:" + name] = {"status": "skipped", "reason": "needs " + ", ".join(missing)}
        return
    if out_of_time(late=True):
        RUN["stages"]["fig:" + name] = {"status": "skipped", "reason": "session time limit is close"}
        return
    try:
        if ("fig:" + name) in os.environ.get("FFB_FAIL_STAGE", "").split(","):
            raise RuntimeError("failure injected by the FFB_FAIL_STAGE test hook")
        out = fn()
        RUN["stages"]["fig:" + name] = {"status": "skipped", "reason": "no data for this figure"} if out is False else {"status": "ok"}
    except Exception as e:
        RUN["stages"]["fig:" + name] = {"status": "FAILED", "error": f"{type(e).__name__}: {e}", "traceback": traceback.format_exc()[-3000:]}
        log(f"-- figure {name}: FAILED  {type(e).__name__}: {e}")
        plt.close("all")

def dot(ax, x, y, lo=None, hi=None, color=C_EFF, marker="o", orient="v", label=None, hollow=False, z=3, ms=6.5):
    if lo is not None and np.isfinite(lo) and np.isfinite(hi):
        if orient == "v":
            ax.plot([x, x], [lo, hi], color=color, lw=1.2, solid_capstyle="round", zorder=z)
        else:
            ax.plot([lo, hi], [y, y], color=color, lw=1.2, solid_capstyle="round", zorder=z)
    ax.plot([x], [y], marker=marker, ms=ms, mfc="white" if hollow else color, mec=color if hollow else "white", mew=1.2 if hollow else 1.0, ls="none", zorder=z + 1, label=label)

def lkey(color, marker="o", label="", line=False, hollow=False, ls="-"):
    from matplotlib.lines import Line2D
    return Line2D([0], [0], color=color if line else "none", marker=marker if marker else "", ms=6, mfc="white" if hollow else color, mec=color if hollow else "white", mew=1.0, lw=1.5, ls=ls, label=label)

def legend_top(ax, handles, ncol=4, y=1.02):
    lg = ax.legend(handles=handles, loc="lower right", bbox_to_anchor=(1.0, y), ncol=ncol, handletextpad=0.4, columnspacing=1.2, borderaxespad=0.0)
    for t in lg.get_texts():
        t.set_color(INK2)
    return lg

def S_get(det, config, target):
    S = ST.get("summarise")
    if S is None:
        return None
    r = S[(S["detector"] == det) & (S["config"] == config) & (S["target"] == target)]
    return None if not len(r) else r.iloc[0]

def seeds_auc(det, config, target):
    E = ST.get("evaluate")
    if E is None or not len(E):
        return []
    e = E[(E["detector"] == det) & (E["config"] == config) & (E["target"] == target) & (E["skipped"] == "")]
    return list(e["auc"].values)

def auc_axis(ax, lo=None):
    ax.axhline(0.5, color=MUTED, lw=0.8, ls=(0, (3, 3)), zorder=1)
    ax.set_ylim(lo if lo is not None else ax.get_ylim()[0], 1.005)
    ax.set_ylabel("video AUC")

In [ ]:
# =============================================================================
# 8b. THE FIGURES
# =============================================================================
def fig_data():
    inv = pd.read_csv(os.path.join(DIRS["tables"], "T01_data_inventory.csv"))
    T4 = os.path.join(DIRS["tables"], "T04_face_extraction.csv")
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), gridspec_kw=dict(width_ratios=[1.25, 1.0], wspace=0.55))
    ax = axes[0]
    inv = inv.assign(label=lambda d: d["dataset"] + " " + d["compression"] + " " + d["method"].map(lambda m: SHORT.get(m, m)))
    inv = inv[inv["videos"] > 0].reset_index(drop=True)
    y = np.arange(len(inv))[::-1]
    ax.barh(y, inv["videos"], color=[METHOD_COLOR.get(m, MUTED) for m in inv["method"]], height=0.7, zorder=2)
    ax.plot(inv["published"], y, ls="none", marker="|", ms=9, mew=1.6, color=INK, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels(inv["label"], fontsize=6.3)
    ax.set_xlabel("videos"); ax.set_title("a  Videos found")
    legend_top(ax, [lkey(INK, "|", "published count")], ncol=1)
    ax = axes[1]
    if os.path.exists(T4):
        t4 = pd.read_csv(T4)
        parts = [("face_detected_share", "detected", C_EFF), ("face_from_neighbour_share", "from a neighbour frame", "#86b6ef"),
                 ("no_face_share", "no face", C_FAKE), ("no_frame_share", "no frame", MUTED)]
        yy = np.arange(len(t4))[::-1]; left = np.zeros(len(t4))
        for col, lab, colr in parts:
            v = t4[col].fillna(0).values
            ax.barh(yy, v, left=left, color=colr, height=0.7, label=lab, zorder=2); left += v
        ax.set_yticks(yy); ax.set_yticklabels(t4["set"], fontsize=6.5); ax.set_xlim(0, 1)
        ax.set_xlabel("share of sampled frames"); ax.set_title("b  Faces")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=6.3)
    save_fig(fig, "F01_data", data=inv, uses=("data", "faces"))

def fig_examples():
    if "ff_c23_test" not in SETS_OK:
        return False
    import cv2
    c23 = CROPS[(CROPS["set"] == "ff_c23_test") & CROPS["valid"]]
    c40 = CROPS[(CROPS["set"] == "ff_c40_test") & CROPS["valid"]] if "ff_c40_test" in SETS_OK else None
    m23 = np.load(RUN["crop_sets"]["ff_c23_test"]["path"], mmap_mode="r")
    m40 = np.load(RUN["crop_sets"]["ff_c40_test"]["path"], mmap_mode="r") if c40 is not None else None
    classes = ["real"] + ALL_METHODS
    dfd = "dfd_c23" in SETS_OK
    nrow = len(classes) + (2 if dfd else 0)
    fig, axes = plt.subplots(nrow, 4, figsize=(4.4, 1.12 * nrow), gridspec_kw=dict(wspace=0.04, hspace=0.08))
    rows = []
    def show(ax, img, title=None):
        ax.imshow(img); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
        if title:
            ax.set_title(title, fontsize=6.5, loc="center", fontweight="normal", color=INK2, pad=2)
    for i, m in enumerate(classes):
        uids = sorted(c23.loc[c23["method"] == m, "uid"].unique())[:2]
        for j in range(2):
            for k, (cc, mm, lab) in enumerate([(c23, m23, "c23"), (c40, m40, "c40")]):
                ax = axes[i, 2 * j + k]
                if j < len(uids) and cc is not None:
                    vid = uids[j].split("|")[-1]
                    r = cc[(cc["video_id"] == vid) & (cc["method"] == m)].sort_values("slot")
                    if len(r):
                        row = int(r["row"].values[0]); show(ax, np.asarray(mm[row]), lab if i == 0 else None)
                        rows.append(dict(cls=m, video_id=vid, comp=lab, row=row)); continue
                ax.axis("off")
        axes[i, 0].set_ylabel(SHORT.get(m, m), fontsize=7, rotation=0, ha="right", va="center", color=INK2); axes[i, 0].yaxis.set_visible(True)
    if dfd:
        d23 = CROPS[(CROPS["set"] == "dfd_c23") & CROPS["valid"]]
        d40 = CROPS[(CROPS["set"] == "dfd_c40") & CROPS["valid"]] if "dfd_c40" in SETS_OK else None
        md = {"c23": np.load(RUN["crop_sets"]["dfd_c23"]["path"], mmap_mode="r")}
        if d40 is not None:
            md["c40"] = np.load(RUN["crop_sets"]["dfd_c40"]["path"], mmap_mode="r")
        for i, (lab, lbl) in enumerate([("DFD real", 0), ("DFD fake", 1)]):
            vids = sorted(d23.loc[d23["label"] == lbl, "video_id"].unique())[:2]
            for j in range(2):
                for k, (cc, comp) in enumerate([(d23, "c23"), (d40, "c40")]):
                    ax = axes[len(classes) + i, 2 * j + k]
                    r = cc[cc["video_id"] == vids[j]].sort_values("slot") if (cc is not None and j < len(vids)) else []
                    if len(r):
                        row = int(r["row"].values[0]); show(ax, np.asarray(md[comp][row]))
                        rows.append(dict(cls=lab, video_id=vids[j], comp=comp, row=row))
                    else:
                        ax.axis("off")
            axes[len(classes) + i, 0].set_ylabel(lab, fontsize=7, rotation=0, ha="right", va="center", color=INK2); axes[len(classes) + i, 0].yaxis.set_visible(True)
    save_fig(fig, "F02_example_crops", data=pd.DataFrame(rows), uses=("faces",))

def roc_panel(ax, target, title):
    drew = False
    rows = []
    for det in ("effnet", "clip"):
        keys = sorted(k for k in VSCORES if k[0] == det and k[1] == "main4" and k[3] == target)
        for i, k in enumerate(keys):
            v = VSCORES[k]
            f, t, _ = FM.roc_curve(v["label"].values, v["score"].values)
            first = i == 0
            ax.plot(f, t, color=DET_COLOR[det], lw=1.6 if first else 0.7, alpha=1.0 if first else 0.55, zorder=3 if first else 2,
                    label=f"{DET_LABEL[det]} (AUC {S_get(det, 'main4', target)['auc']:.3f})" if first else None)
            rows += [dict(detector=det, seed=k[2], target=target, fpr=a, tpr=b) for a, b in zip(f, t)]
            drew = True
    ax.plot([0, 1], [0, 1], color=MUTED, lw=0.8, ls=(0, (3, 3)))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.0); ax.set_aspect("equal")
    ax.set_xlabel("false positive rate (real flagged)"); ax.set_ylabel("true positive rate (fakes caught)")
    ax.set_title(title)
    if drew:
        ax.legend(loc="lower right", fontsize=6.3)
    return drew, rows

def fig_roc_ffpp():
    tg = [t for t in ("FF++ c23 | four classic methods", "FF++ c40 | four classic methods") if S_get("effnet", "main4", t) is not None or S_get("clip", "main4", t) is not None]
    if not tg:
        return False
    fig, axes = plt.subplots(1, len(tg), figsize=(3.4 * len(tg), 3.2), squeeze=False)
    rows = []
    for ax, t, letter in zip(axes[0], tg, "ab"):
        _, r = roc_panel(ax, t, f"{letter}  {t.replace(' | four classic methods', '')}: real vs DF+F2F+FS+NT"); rows += r
    save_fig(fig, "F03_roc_ffpp", data=pd.DataFrame(rows), uses=("gpu_jobs", "evaluate"))

def per_method_panel(ax, comp, config_of=lambda m: "main4", seen_note=True):
    rows = []
    xs = np.arange(len(ALL_METHODS))
    for di, det in enumerate(("effnet", "clip")):
        for i, m in enumerate(ALL_METHODS):
            t = f"FF++ {comp} | {m}"; cfgn = config_of(m)
            r = S_get(det, cfgn, t)
            if r is None:
                continue
            x = i + (di - 0.5) * 0.28
            dot(ax, x, r["auc"], r["auc_ci_lo"], r["auc_ci_hi"], DET_COLOR[det])
            for a in seeds_auc(det, cfgn, t) if det == "effnet" and r["n_seeds"] > 1 else []:
                ax.plot([x + 0.09], [a], marker="o", ms=3, mfc="white", mec=DET_COLOR[det], mew=0.8, ls="none", zorder=5)
            rows.append(dict(detector=det, config=cfgn, target=t, auc=r["auc"], ci_lo=r["auc_ci_lo"], ci_hi=r["auc_ci_hi"]))
    ax.set_xticks(xs); ax.set_xticklabels([SHORT[m] + ("\n(unseen)" if m == "FaceShifter" and seen_note else "") for m in ALL_METHODS])
    ax.axvspan(len(ALL_METHODS) - 1.5, len(ALL_METHODS) - 0.5, color="#f3f2ee", zorder=0)
    return rows

def fig_per_method():
    comps = [c for c in ("c23", "c40") if any(S_get(d, "main4", f"FF++ {c} | Deepfakes") is not None for d in ("effnet", "clip"))]
    if not comps:
        return False
    fig, axes = plt.subplots(1, len(comps), figsize=(3.6 * len(comps), 2.8), squeeze=False, sharey=True)
    rows = []
    for ax, comp, letter in zip(axes[0], comps, "ab"):
        rows += per_method_panel(ax, comp)
        ax.set_title(f"{letter}  FF++ {comp} test split")
        auc_axis(ax)
    legend_top(axes[0][-1], [lkey(C_EFF, label=DET_LABEL["effnet"]), lkey(C_CLIP, label=DET_LABEL["clip"]), lkey(C_EFF, "o", "individual seeds", hollow=True)], ncol=3, y=1.08)
    save_fig(fig, "F04_auc_per_method", data=pd.DataFrame(rows), uses=("gpu_jobs", "evaluate"))

def fig_unseen_method():
    rows = []
    comps = [c for c in ("c23", "c40") if any(S_get(d, f"lomo_{m}", f"FF++ {c} | {m}") is not None for d in ("effnet", "clip") for m in CLASSIC)]
    if not comps:
        return False
    fig, axes = plt.subplots(1, len(comps), figsize=(3.6 * len(comps), 2.8), squeeze=False, sharey=True)
    for ax, comp, letter in zip(axes[0], comps, "ab"):
        for di, det in enumerate(("effnet", "clip")):
            for i, m in enumerate(CLASSIC):
                t = f"FF++ {comp} | {m}"
                a, b = S_get(det, "main4", t), S_get(det, f"lomo_{m}", t)
                x = i + (di - 0.5) * 0.32
                if a is not None and b is not None:
                    ax.plot([x, x], [a["auc"], b["auc"]], color=DET_COLOR[det], lw=0.8, alpha=0.6, zorder=2)
                if a is not None:
                    dot(ax, x - 0.06, a["auc"], a["auc_ci_lo"], a["auc_ci_hi"], DET_COLOR[det], "o")
                    rows.append(dict(detector=det, method=m, comp=comp, condition="seen (main4)", auc=a["auc"], ci_lo=a["auc_ci_lo"], ci_hi=a["auc_ci_hi"]))
                if b is not None:
                    dot(ax, x + 0.06, b["auc"], b["auc_ci_lo"], b["auc_ci_hi"], DET_COLOR[det], "D", hollow=True)
                    rows.append(dict(detector=det, method=m, comp=comp, condition="held out (lomo)", auc=b["auc"], ci_lo=b["auc_ci_lo"], ci_hi=b["auc_ci_hi"]))
        ax.set_xticks(range(len(CLASSIC))); ax.set_xticklabels([SHORT[m] for m in CLASSIC])
        ax.set_title(f"{letter}  held-out method, FF++ {comp}"); auc_axis(ax)
    legend_top(axes[0][-1], [lkey(C_EFF, label="EfficientNet"), lkey(C_CLIP, label="CLIP probe"), lkey(INK2, "o", "seen in training"), lkey(INK2, "D", "held out", hollow=True)], ncol=4, y=1.08)
    save_fig(fig, "F05_unseen_method", data=pd.DataFrame(rows), uses=("gpu_jobs", "evaluate"))

def fig_compression():
    pairs = [("FF++ | four classic", "FF++ c23 | four classic methods", "FF++ c40 | four classic methods")] + \
            [(f"FF++ | {SHORT[m]}", f"FF++ c23 | {m}", f"FF++ c40 | {m}") for m in ALL_METHODS] + [("DFD | all", "DFD c23 | all", "DFD c40 | all")]
    pairs = [p for p in pairs if any(S_get(d, "main4", p[1]) is not None and S_get(d, "main4", p[2]) is not None for d in ("effnet", "clip"))]
    if not pairs:
        return False
    fig, ax = plt.subplots(figsize=(7.2, 2.9))
    rows = []
    for i, (lab, a_t, b_t) in enumerate(pairs):
        for di, det in enumerate(("effnet", "clip")):
            a, b = S_get(det, "main4", a_t), S_get(det, "main4", b_t)
            if a is None or b is None:
                continue
            x = i + (di - 0.5) * 0.34
            ax.plot([x - 0.07, x + 0.07], [a["auc"], b["auc"]], color=DET_COLOR[det], lw=0.8, alpha=0.6, zorder=2)
            dot(ax, x - 0.07, a["auc"], a["auc_ci_lo"], a["auc_ci_hi"], DET_COLOR[det], "o")
            dot(ax, x + 0.07, b["auc"], b["auc_ci_lo"], b["auc_ci_hi"], DET_COLOR[det], "s", hollow=True)
            rows += [dict(detector=det, test=lab, comp="c23", auc=a["auc"], ci_lo=a["auc_ci_lo"], ci_hi=a["auc_ci_hi"]),
                     dict(detector=det, test=lab, comp="c40", auc=b["auc"], ci_lo=b["auc_ci_lo"], ci_hi=b["auc_ci_hi"])]
    ax.set_xticks(range(len(pairs))); ax.set_xticklabels([p[0] for p in pairs], fontsize=6.6)
    ax.set_title("Compression: the test videos at c23 and at c40 (main4)"); auc_axis(ax)
    legend_top(ax, [lkey(C_EFF, label="EfficientNet"), lkey(C_CLIP, label="CLIP probe"), lkey(INK2, "o", "c23"), lkey(INK2, "s", "c40", hollow=True)], ncol=4, y=1.06)
    save_fig(fig, "F06_compression", data=pd.DataFrame(rows), uses=("gpu_jobs", "evaluate"))

def fig_cross_dataset():
    if not any(S_get(d, "main4", "DFD c23 | all") is not None for d in ("effnet", "clip")):
        return False
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1), gridspec_kw=dict(width_ratios=[1, 1.15], wspace=0.35))
    _, rows = roc_panel(axes[0], "DFD c23 | all", "a  DFD c23, all videos (trained on FF++)")
    ax = axes[1]
    def matched(comp):          # FF++ scored from as many frames per video as DFD, when that test set exists
        t = f"FF++ {comp} | four classic methods | as many frames as DFD"
        return t if any(S_get(d, "main4", t) is not None for d in ("effnet", "clip")) else f"FF++ {comp} | four classic methods"
    tests = [matched("c23"), matched("c40"), "DFD c23 | all", "DFD c40 | all"]
    k = DFD_FRAMES if (DFD_FRAMES and "as many frames" in tests[0]) else None
    labs = [f"FF++ c23" + (f"\n({k} frames)" if k else ""), f"FF++ c40" + (f"\n({k} frames)" if k else ""), "DFD c23", "DFD c40"]
    rows2 = []
    for i, t in enumerate(tests):
        for di, det in enumerate(("effnet", "clip")):
            r = S_get(det, "main4", t)
            if r is None:
                continue
            x = i + (di - 0.5) * 0.3
            dot(ax, x, r["auc"], r["auc_ci_lo"], r["auc_ci_hi"], DET_COLOR[det])
            for a in seeds_auc(det, "main4", t) if det == "effnet" and r["n_seeds"] > 1 else []:
                ax.plot([x + 0.09], [a], marker="o", ms=3, mfc="white", mec=DET_COLOR[det], mew=0.8, ls="none", zorder=5)
            rows2.append(dict(detector=det, target=t, auc=r["auc"], ci_lo=r["auc_ci_lo"], ci_hi=r["auc_ci_hi"]))
    ax.axvspan(1.5, 3.5, color="#f3f2ee", zorder=0)
    ax.set_xticks(range(4)); ax.set_xticklabels(labs); ax.set_title("b  in-dataset and cross-dataset (main4)"); auc_axis(ax)
    save_fig(fig, "F07_cross_dataset", data=pd.DataFrame(rows + rows2), uses=("gpu_jobs", "evaluate"))

def fig_dfd_training():
    tests = [t for t in ("DFD c23 | test actors", "DFD c40 | test actors", "FF++ c23 | four classic methods", "FF++ c40 | four classic methods")
             if any(S_get(d, "ffdfd", t) is not None for d in ("effnet", "clip"))]
    if not tests:
        return False
    fig, ax = plt.subplots(figsize=(6.0, 2.8))
    rows = []
    for i, t in enumerate(tests):
        for di, det in enumerate(("effnet", "clip")):
            for ci_, cfgn in enumerate(("main4", "ffdfd")):
                r = S_get(det, cfgn, t)
                if r is None:
                    continue
                x = i + (di - 0.5) * 0.4 + (ci_ - 0.5) * 0.14
                dot(ax, x, r["auc"], r["auc_ci_lo"], r["auc_ci_hi"], DET_COLOR[det], "o" if cfgn == "main4" else "^", hollow=cfgn == "main4")
                rows.append(dict(detector=det, config=cfgn, target=t, auc=r["auc"], ci_lo=r["auc_ci_lo"], ci_hi=r["auc_ci_hi"]))
    ax.set_xticks(range(len(tests))); ax.set_xticklabels([t.replace(" | four classic methods", "").replace(" | test actors", "\ntest actors") for t in tests])
    ax.set_title("DFD as additional training data"); auc_axis(ax)
    legend_top(ax, [lkey(C_EFF, label="EfficientNet"), lkey(C_CLIP, label="CLIP probe"), lkey(INK2, "o", "FF++ only (main4)", hollow=True), lkey(INK2, "^", "FF++ + DFD train actors")], ncol=2, y=1.04)
    save_fig(fig, "F08_dfd_as_training_data", data=pd.DataFrame(rows), uses=("gpu_jobs", "evaluate"))

def fig_gradcam():
    if not GRADCAM:
        return False
    import matplotlib.cm as cm
    classes = [m for m in ["real"] + ALL_METHODS if any(g["method"] == m for g in GRADCAM)]
    fig, axes = plt.subplots(len(classes), 4, figsize=(4.4, 1.15 * len(classes)), gridspec_kw=dict(wspace=0.04, hspace=0.18), squeeze=False)
    for i, m in enumerate(classes):
        ex = [g for g in GRADCAM if g["method"] == m][:2]
        for j in range(2):
            a1, a2 = axes[i, 2 * j], axes[i, 2 * j + 1]
            for a in (a1, a2):
                a.set_xticks([]); a.set_yticks([]); a.grid(False)
                for s in a.spines.values():
                    s.set_visible(False)
            if j >= len(ex):
                a1.axis("off"); a2.axis("off"); continue
            g = ex[j]
            a1.imshow(g["img"])
            heat = cm.get_cmap("inferno")(g["cam"])[..., :3] if hasattr(cm, "get_cmap") else plt.colormaps["inferno"](g["cam"])[..., :3]
            a2.imshow(np.clip(0.55 * g["img"] / 255.0 + 0.45 * heat, 0, 1))
            a2.set_title(f"p(fake) {g['prob_fake']:.2f}", fontsize=6, loc="center", fontweight="normal", color=INK2, pad=2)
        axes[i, 0].set_ylabel(SHORT.get(m, m), fontsize=7, rotation=0, ha="right", va="center", color=INK2); axes[i, 0].yaxis.set_visible(True)
    save_fig(fig, "F09_gradcam", data=pd.DataFrame([{k: v for k, v in g.items() if k not in ("img", "cam")} for g in GRADCAM]), uses=("gradcam",))

def fig_training():
    H = ST.get("history")
    if H is None or not len(H):
        return False
    cfgs = [c for c in CFG["CONFIGS"] if c in set(H["config"])]
    has_vl = "val_loss" in H.columns and H["val_loss"].notna().any()
    fig, axes = plt.subplots(1, 3 if has_vl else 2, figsize=(7.2, 2.6), gridspec_kw=dict(wspace=0.38))
    cmap = plt.colormaps.get("tab10") if hasattr(plt, "colormaps") else plt.cm.tab10
    for ci_, c in enumerate(cfgs):
        col = METHOD_COLOR.get(c[5:], C_EFF) if c.startswith("lomo_") else ({"main4": INK, "ffdfd": C_FAKE}.get(c, cmap(ci_)))
        for s, g in H[H["config"] == c].groupby("seed"):
            g = g.sort_values("epoch")
            axes[0].plot(g["epoch"], g["train_loss"], color=col, lw=1.0, alpha=0.8, label=c if s == min(H[H["config"] == c]["seed"]) else None)
            axes[1].plot(g["epoch"], g["val_video_auc"], color=col, lw=1.0, alpha=0.8)
            if has_vl:
                axes[2].plot(g["epoch"], g["val_loss"], color=col, lw=1.0, alpha=0.8)
            sel = g[g["selected"] == 1] if "selected" in g.columns else g.loc[[g["val_video_auc"].idxmax()]] if g["val_video_auc"].notna().any() else g.iloc[:0]
            for k, col_name in ((1, "val_video_auc"), (2, "val_loss")):
                if len(sel) and k < len(axes) and col_name in sel.columns:
                    axes[k].plot(sel["epoch"], sel[col_name], marker="o", ms=4, color=col, mec="white", ls="none")
    from matplotlib.ticker import MaxNLocator
    for ax in axes:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("epoch")
    axes[0].set_ylabel("training loss"); axes[0].set_title("a  training loss")
    axes[1].set_ylabel("validation video AUC"); axes[1].set_title("b  validation AUC")
    if has_vl:
        axes[2].set_ylabel("validation loss (balanced)"); axes[2].set_title("c  validation loss")
    hl, lb = axes[0].get_legend_handles_labels()
    fig.legend(hl, lb, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=min(len(lb), 6), fontsize=6.5, handlelength=1.6, columnspacing=1.2)
    save_fig(fig, "F10_training_curves", data=H, uses=("gpu_jobs",))

def fig_frames():
    F = ST.get("frames_curve")
    if F is None or not len(F):
        return False
    tg = list(dict.fromkeys(F["target"]))
    fig, axes = plt.subplots(1, len(tg), figsize=(2.6 * len(tg), 2.6), squeeze=False, sharey=True)
    rows = []
    for ax, t, letter in zip(axes[0], tg, "abc"):
        for det in ("effnet", "clip"):
            g = F[(F["detector"] == det) & (F["target"] == t)].groupby("frames")["auc"].agg(["mean", "min", "max"]).reset_index()
            if not len(g):
                continue
            ax.plot(g["frames"], g["mean"], color=DET_COLOR[det], marker="o", ms=4, label=DET_LABEL[det])
            if det == "effnet":
                ax.fill_between(g["frames"], g["min"], g["max"], color=DET_COLOR[det], alpha=0.15, lw=0)
            rows += [dict(detector=det, target=t, **r) for r in g.to_dict("records")]
        ax.set_xscale("log", base=2); ax.set_xticks(CFG["FRAMES_CURVE"]); ax.set_xticklabels(CFG["FRAMES_CURVE"])
        ax.set_xlabel("frames averaged per video"); ax.set_title(f"{letter}  {t.replace(' | four classic methods', '')}", fontsize=7.5)
    axes[0][0].set_ylabel("video AUC"); axes[0][0].legend(fontsize=6)
    save_fig(fig, "F11_frames_per_video", data=pd.DataFrame(rows), uses=("evaluate", "frames_curve"))

def fig_scores():
    tests = [t for t in ("FF++ c23 | four classic methods", "FF++ c40 | four classic methods", "DFD c23 | all") if any(k[1] == "main4" and k[3] == t for k in VSCORES)]
    if not tests:
        return False
    dets = [d for d in ("effnet", "clip") if any(k[0] == d for k in VSCORES)]
    fig, axes = plt.subplots(len(dets), len(tests), figsize=(2.4 * len(tests), 1.9 * len(dets)), squeeze=False, sharex=True)
    rows = []
    bins = np.linspace(0, 1, 26)
    for i, det in enumerate(dets):
        for j, t in enumerate(tests):
            ax = axes[i, j]
            ks = sorted(k for k in VSCORES if k[0] == det and k[1] == "main4" and k[3] == t)
            if not ks:
                ax.axis("off"); continue
            v = VSCORES[ks[0]]
            p = FM.sigmoid(v["score"].values)
            for lab, col, name in ((0, C_REAL, "real"), (1, C_FAKE, "manipulated")):
                ax.hist(p[v["label"].values == lab], bins=bins, color=col, alpha=0.55, density=True, label=name)
            ax.set_title(f"{DET_LABEL[det].split(' (')[0].split(' probe')[0]} | {t.replace(' | four classic methods', '')}", fontsize=7)
            rows += [dict(detector=det, target=t, uid=u, label=l, prob_fake=q) for u, l, q in zip(v["uid"], v["label"], p)]
            if i == len(dets) - 1:
                ax.set_xlabel("video probability of 'fake'")
            if j == 0:
                ax.set_ylabel("density")
    axes[0][0].legend(fontsize=6)
    save_fig(fig, "F12_score_distributions", data=pd.DataFrame(rows), uses=("evaluate",))

def fig_calibration():
    R = ST.get("calibration")
    if R is None or not len(R):
        return False
    tg = list(dict.fromkeys(R["target"]))
    fig, axes = plt.subplots(1, len(tg), figsize=(2.3 * len(tg), 2.5), squeeze=False, sharey=True)
    S = ST.get("summarise")
    for ax, t in zip(axes[0], tg):
        for det in ("effnet", "clip"):
            g = R[(R["detector"] == det) & (R["target"] == t) & (R["n"] > 0)]
            if not len(g):
                continue
            ax.plot(g["mean_p"], g["frac_fake"], marker="o", ms=3.5, color=DET_COLOR[det], label=f"{'EffNet' if det == 'effnet' else 'CLIP'}: ECE {float(g['ece_this_seed'].iat[0]):.3f}")
        ax.plot([0, 1], [0, 1], color=MUTED, lw=0.8, ls=(0, (3, 3)))
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
        ax.set_xlabel("mean predicted p(fake)"); ax.set_title(t.replace(" | four classic methods", ""), fontsize=7.5); ax.legend(fontsize=6, loc="upper left")
    axes[0][0].set_ylabel("observed share of fakes")
    save_fig(fig, "F13_calibration", data=R, uses=("evaluate", "calibration"))

def fig_overview():
    S = ST.get("summarise")
    if S is None or not len(S):
        return False
    cols = [(d, c) for d in ("effnet", "clip") for c in CFG["CONFIGS"] if ((S["detector"] == d) & (S["config"] == c)).any()]
    targets = [t["target"] for t in target_list() if (S["target"] == t["target"]).any()]
    M = np.full((len(targets), len(cols)), np.nan)
    for i, t in enumerate(targets):
        for j, (d, c) in enumerate(cols):
            r = S_get(d, c, t)
            if r is not None:
                M[i, j] = r["auc"]
    fig, ax = plt.subplots(figsize=(0.62 * len(cols) + 2.6, 0.24 * len(targets) + 1.2))
    cmap = plt.colormaps["viridis"].copy() if hasattr(plt, "colormaps") else plt.cm.viridis
    cmap.set_bad("#e6e6e6")
    im = ax.imshow(np.ma.masked_invalid(M), aspect="auto", cmap=cmap, vmin=0.5, vmax=1.0, interpolation="nearest")
    for i in range(len(targets)):
        for j in range(len(cols)):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=5.6, color="white" if M[i, j] < 0.8 else INK)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([f"{'EffNet' if d == 'effnet' else 'CLIP'}\n" + (f"w/o {SHORT[c[5:]]}" if c.startswith("lomo_") else c) for d, c in cols], fontsize=6, rotation=0)
    ax.set_yticks(range(len(targets))); ax.set_yticklabels(targets, fontsize=6)
    ax.grid(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02); cb.set_label("video AUC"); cb.outline.set_visible(False)
    ax.set_title("Video AUC: detector x training configuration x test set")
    save_fig(fig, "F14_auc_overview", data=S[["detector", "config", "target", "auc", "auc_ci_lo", "auc_ci_hi", "n_seeds"]], uses=("gpu_jobs", "evaluate"))

for _name, _fn, _needs in [
    ("F01_data", fig_data, ("data",)),
    ("F02_example_crops", fig_examples, ("faces",)),
    ("F03_roc_ffpp", fig_roc_ffpp, ("summarise",)),
    ("F04_auc_per_method", fig_per_method, ("summarise",)),
    ("F05_unseen_method", fig_unseen_method, ("summarise",)),
    ("F06_compression", fig_compression, ("summarise",)),
    ("F07_cross_dataset", fig_cross_dataset, ("summarise",)),
    ("F08_dfd_as_training_data", fig_dfd_training, ("summarise",)),
    ("F09_gradcam", fig_gradcam, ("gradcam",)),
    ("F10_training_curves", fig_training, ("history",)),
    ("F11_frames_per_video", fig_frames, ("frames_curve",)),
    ("F12_score_distributions", fig_scores, ("evaluate",)),
    ("F13_calibration", fig_calibration, ("calibration",)),
    ("F14_auc_overview", fig_overview, ("summarise",)),
]:
    make_fig(_name, _fn, _needs)
log(f"figures written: {len(FIGS)} of {N_FIGURES_PLANNED}")

## 9. What this benchmark does not establish

* **It is a benchmark, not a new method.** Fine-tuned EfficientNet-B4 and a linear probe on frozen CLIP are established baselines (DeepfakeBench; UnivFD). What the notebook adds is a clean, reproducible protocol on Kaggle hardware: the official split, model selection on validation only, a leakage check, clustered intervals, and the c40, unseen-method and DFD conditions in one run. A paper built on it needs its own contribution and a literature check of its own.
* **Frames, not all of the video.** Frames are sampled from the first 300 frames of each video (16, 8 or 4 per video); published FF++ protocols often use 32 frames from the whole video. Numbers are therefore not directly comparable with papers that use other frame counts, resolutions (DeepfakeBench uses 256 px) or face detectors.
* **One face per frame, found by YuNet.** When a frame shows several faces, the face that differs from the counterpart video is taken; when no face differs clearly, the largest face is followed. If a manipulated video and its original are not frame-aligned, the difference cannot decide and a wrong face may be taken in a multi-person scene. Table T04 reports how often frames showed several faces and how often the difference decided; figure F02 shows sample crops.
* **The `ffdfd` runs are selected on FF++ validation data only.** Their epoch, probe strength and decision threshold come from FF++ validation videos, so accuracy-type metrics on DFD use a threshold chosen on FF++; DFD's training actors are used for training only.
* **DFD comes from the same release.** DFD is distributed by the FaceForensics project and its fakes were made by Google/Jigsaw with a different pipeline and different actors, so it is a cross-dataset test; it is not "in the wild". Other cross-dataset sets (Celeb-DF, DFDC, WildDeepfake) are not included.
* **Mirrors.** The Kaggle copies are unofficial; the notebook checks counts and file names, not content. The inventory table T01 lists what was found against the published numbers.
* **Seeds and intervals.** Three seeds describe training variability only roughly; the intervals that resample them are correspondingly rough. The intervals resample test videos by cluster (DFD two-way by actor), so they speak to these test sets, not to other datasets. No hypothesis tests are run, the many comparisons of table T09 are not corrected for multiplicity, and nothing should be read into an interval beyond what it states.
* **The CLIP probe is deterministic**, so it has no seed spread; its intervals cover test-set sampling only.
* **Calibration is measured on class-reweighted models.** The EfficientNet loss weights the classes inversely to their number of training crops and the probe uses balanced class weights, so both are pushed towards a 50/50 prior; part of the calibration error on DFD (363 real against 3,068 fake videos) comes from that prior, not only from the shift in appearance.

In [ ]:
# =============================================================================
# 10. EXPORT: tables, manifest, validation, ZIP
# =============================================================================
TABLE_DOC = {
    "T01": "videos found per dataset, compression and method, next to the published counts",
    "T02": "videos per split and method",
    "T03": "extraction plan: estimated time and disk for each cut rule (the one applied is marked)",
    "T04": "face extraction per crop set: videos, crops, share of frames with a detected face",
    "T05": "every GPU job: status, minutes, selected epoch, validation AUC, throughput",
    "T06": "CLIP probe: validation video AUC for each C",
    "T07": "all metrics per detector, configuration and test set: mean, SD, minimum and maximum over seeds; the 95% interval of the video AUC (seeds resampled) and with the models fixed; numbers of real and manipulated videos and of bootstrap clusters; test_set_complete is False when the time budget stopped the face extraction of that test set early (a random subset of its videos is used; see T04)",
    "T08": "overview: video AUC [95% interval] per test set (rows) and detector/configuration (columns)",
    "T09": "differences of video AUC with 95% intervals (seeds resampled; also with the models fixed), per-seed differences and whether they agree in sign: CLIP vs EfficientNet, c40 vs c23, unseen vs seen method, ffdfd vs main4, DFD vs FF++ (at the same number of frames per video)",
    "T10": "video AUC as a function of the number of frames per video",
    "T11": "reliability-diagram bins (calibration)",
    "T12": "EfficientNet training history per epoch: training loss, validation video AUC, class-balanced validation loss; selected = 1 marks the epoch whose weights were kept",
    "T14": "Grad-CAM examples (which videos, predicted probability)",
}

def stage_export():
    if RUN.get("mode") != MODE:
        raise RuntimeError("DRY_RUN or QUICK was changed after the first cell had run. Run all cells again from the top.")
    problems = []
    def step(label, fn, default=None):
        try:
            return fn()
        except Exception as e:
            problems.append(f"export step '{label}' failed: {type(e).__name__}: {e}")
            log(f"   export step '{label}' FAILED: {type(e).__name__}: {e}")
            return default
    step("figure list", lambda: pd.DataFrame(FIGS).to_csv(os.path.join(DIRS["metadata"], "figures.csv"), index=False))
    step("video list", lambda: VID.drop(columns=["path"]).to_csv(os.path.join(DIRS["metadata"], "videos.csv.gz"), index=False, compression="gzip"))
    step("crop list", lambda: CROPS.to_csv(os.path.join(DIRS["metadata"], "crops.csv.gz"), index=False, compression="gzip") if len(CROPS) else None)
    def scores_table():
        out = []
        for (det, config, seed), run in RUNS.items():
            for s, (rows, z) in run["tests"].items():
                out.append(pd.DataFrame(dict(detector=det, config=config, seed=seed, set=s, row=np.asarray(rows), logit=np.asarray(z, np.float32))))
        if out:
            pd.concat(out, ignore_index=True).to_csv(os.path.join(DIRS["results"], "frame_scores.csv.gz"), index=False, compression="gzip")
    step("frame scores", scores_table)
    def video_scores():
        out = [v.assign(detector=k[0], config=k[1], seed=k[2], target=k[3]) for k, v in VSCORES.items()]
        if out:
            pd.concat(out, ignore_index=True).to_csv(os.path.join(DIRS["results"], "video_scores.csv.gz"), index=False, compression="gzip")
    step("video scores", video_scores)

    val = dict(figures=[], problems=problems)
    def validate_figures():
        for f in FIGS:
            ok_png = os.path.exists(f["png"]) and os.path.getsize(f["png"]) > 5000
            ok_pdf = os.path.exists(f["pdf"]) and open(f["pdf"], "rb").read(5) == b"%PDF-"
            dpi, size = None, None
            try:
                from PIL import Image
                with Image.open(f["png"]) as im:
                    dpi = im.info.get("dpi"); size = im.size
                dpi = None if dpi is None else float(dpi[0])
            except Exception:
                pass
            ok_dpi = dpi is not None and abs(dpi - CFG["DPI"]) < 1.0
            val["figures"].append(dict(name=f["name"], png=bool(ok_png), pdf=bool(ok_pdf), png_dpi=dpi, png_pixels=None if size is None else list(size), dpi_ok=bool(ok_dpi), complete=bool(f.get("complete", True))))
            if not (ok_png and ok_pdf and ok_dpi):
                problems.append(f"figure {f['name']}: png={ok_png} pdf={ok_pdf} dpi={dpi}")
    step("figure validation", validate_figures)
    by = lambda status: [k for k, v in RUN["stages"].items() if v.get("status") == status]
    failed, skipped, stopped = by("FAILED"), by("skipped"), by("stopped")
    val.update(stages_failed=failed, stages_skipped=skipped, stages_stopped=stopped, n_figures=len(FIGS), n_figures_planned=N_FIGURES_PLANNED,
               figures_incomplete=[f["name"] for f in FIGS if not f.get("complete", True)])
    if not FIGS:
        problems.append("no figure was written")
    timed_out = [k[4:] for k, v in RUN["stages"].items() if k.startswith("fig:") and v.get("status") == "skipped" and "time limit" in str(v.get("reason", ""))]
    if timed_out:
        problems.append(f"{len(timed_out)} figure(s) were not drawn because the session time limit was reached: {', '.join(timed_out)}")
    RUN["validation"] = val
    RUN["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    RUN["minutes_total"] = round(elapsed_min(), 1)
    RUN["config"] = {k: v for k, v in CFG.items()}
    RUN["figures"] = [f["name"] for f in FIGS]
    RUN["stages"]["export"] = {"status": "ok", "note": "status at the time the manifest was written, inside the export stage"}

    def stamp_tables():
        """Rehearsal and quick test: every CSV gets a first column saying what it is."""
        word = "DRY_RUN_not_a_result" if DRY_RUN else "QUICK_TEST_not_a_result"
        n = 0
        for root, _, fns in os.walk(OUT):
            for fn in fns:
                if not (fn.endswith(".csv") or fn.endswith(".csv.gz")):
                    continue
                fp = os.path.join(root, fn)
                try:
                    df = pd.read_csv(fp, dtype=str, keep_default_na=False)
                except Exception:
                    continue
                if word not in df.columns:
                    df.insert(0, word, "synthetic rehearsal" if DRY_RUN else "small subset")
                    df.to_csv(fp, index=False, compression="gzip" if fn.endswith(".gz") else None)
                    n += 1
        return n
    if MODE != "full":
        step("stamp on the tables", stamp_tables)

    def readme_text():
        fig_lines = chr(10).join(f"  {f['name']}{'' if f.get('complete', True) else '  [INCOMPLETE]'}: {f.get('caption', '')}" for f in FIGS) or "  none"
        tab_lines = chr(10).join(f"  {k}: {v}" for k, v in TABLE_DOC.items() if any(n.startswith(k) for n in os.listdir(DIRS["tables"])))
        stamp = {"dryrun": "*** DRY RUN: synthetic videos and tiny random networks. NOTHING IN THIS ARCHIVE IS A RESULT. ***",
                 "quick": "*** QUICK TEST: a small subset of the videos, one seed, two epochs. NOT THE BENCHMARK. ***", "full": "Full run."}[MODE]
        return f"""Face-forgery benchmark on FaceForensics++ and DFD -- run artifacts
====================================================================
{stamp}

figures/png/   every figure as PNG at {CFG['DPI']} DPI
figures/pdf/   the same figures as vector PDF (TrueType fonts embedded)
tables/        T01-T14 as CSV; tables/figure_data/ holds the numbers plotted in each figure
results/       metrics of every run on every test set; frame and video scores of every run
metadata/      protocol.json (written at the start), run_manifest.json, videos.csv.gz, crops.csv.gz, figures.csv
logs/          run.log and the GPU workers' logs
src/           the library modules exactly as executed

Read first: tables/T08_video_auc_overview.csv, tables/T09_comparisons.csv, metadata/run_manifest.json (warnings, skipped stages).
Uncertainty: the _sd/_min/_max columns describe the EfficientNet seeds. auc_ci_lo/auc_ci_hi are 95% bootstrap intervals
that resample the test videos by cluster (FF++: source pairs; DFD: two-way by actor) and the EfficientNet seeds;
auc_ci_*_models_fixed hold the trained models fixed. T09 makes many comparisons without a multiplicity correction.

Figures written: {len(FIGS)} of {N_FIGURES_PLANNED}
Stages failed: {failed or 'none'}
Stages stopped at the time limit: {stopped or 'none'}
Stages skipped: {skipped or 'none'}
Protocol sha256: {PROTOCOL_SHA}

FIGURES
{fig_lines}

TABLES
{tab_lines}
"""
    step("README", lambda: open(os.path.join(OUT, "README.txt"), "w").write(readme_text()))
    def write_manifest():
        with open(os.path.join(DIRS["metadata"], "run_manifest.json"), "w") as f:
            json.dump(RUN, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    step("manifest", write_manifest)
    zip_name = CFG["ZIP_NAME"].replace(".zip", {"dryrun": "_DRYRUN", "quick": "_QUICKTEST", "full": ""}[MODE] + ".zip")
    zip_path = os.path.join(WORK, zip_name)
    def write_zip():
        skipped_files = []
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
            for root, _, fns in os.walk(OUT):
                for fn in sorted(fns):
                    full = os.path.join(root, fn)
                    try:
                        zf.write(full, arcname=os.path.relpath(full, OUT))
                    except Exception as e:
                        skipped_files.append(f"{fn}: {type(e).__name__}")
            for fn in sorted(os.listdir(SRC)):
                if fn.endswith(".py"):
                    zf.write(os.path.join(SRC, fn), arcname=os.path.join("src", fn))
        with zipfile.ZipFile(zip_path) as zf:
            bad = zf.testzip(); names = zf.namelist()
        n_png = sum(n.startswith("figures/png/") and n.endswith(".png") for n in names)
        n_pdf = sum(n.startswith("figures/pdf/") and n.endswith(".pdf") for n in names)
        found = []
        if skipped_files:
            found.append("files that could not be added to the ZIP: " + "; ".join(skipped_files))
        if bad is not None:
            found.append(f"the ZIP has a corrupt member: {bad}")
        if n_png != len(FIGS) or n_pdf != len(FIGS):
            found.append(f"the ZIP holds {n_png} PNG and {n_pdf} PDF files for {len(FIGS)} figures")
        return n_png, n_pdf, found
    n_png, n_pdf, found = write_zip()
    if found:
        problems.extend(found)
        step("manifest", write_manifest)
        n_png, n_pdf, _ = write_zip()
    return dict(zip=zip_path, n_png=n_png, n_pdf=n_pdf, size_mb=os.path.getsize(zip_path) / 1e6, val=val)

EXP = run_stage("export", stage_export)

def final_report():
    print("=" * 100)
    by = lambda status: [k for k, v in RUN["stages"].items() if v.get("status") == status]
    failed, skipped, stopped = by("FAILED"), by("skipped"), by("stopped")
    nodata = [k[4:] for k, v in RUN["stages"].items() if k.startswith("fig:") and v.get("status") == "skipped" and v.get("reason") == "no data for this figure"]
    n_expected = N_FIGURES_PLANNED - len(nodata)
    if not EXP:
        head = "RUN ENDED WITHOUT A ZIP: the export stage failed (see below)."
    elif failed or stopped or len(FIGS) < n_expected:
        head = (f"RUN FINISHED, BUT NOT COMPLETE: {len(failed)} stage(s) failed, {len(stopped)} stopped at the time limit, "
                f"{len(FIGS)} of {n_expected} figures written. The ZIP holds everything that was produced.")
    else:
        head = f"RUN COMPLETE: every stage ran, {len(FIGS)} figures written, ZIP ready."
    if nodata:
        head += f"\n({len(nodata)} figure(s) need data that is not attached: {', '.join(nodata)})"
    pre = {"dryrun": "DRY RUN on synthetic videos with tiny random networks. Nothing below is a result.\n",
           "quick": "QUICK TEST on a small subset (one seed, two epochs). Nothing below is a benchmark result.\n", "full": ""}[MODE]
    print(pre + head)
    print(f"finished in {elapsed_min():.1f} min | stages ok: {sum(v.get('status') == 'ok' for v in RUN['stages'].values())}, failed: {len(failed)}, stopped: {len(stopped)}, skipped: {len(skipped)}")
    if failed:
        print("\n!!! FAILED STAGES (the run continued so that everything else is kept):")
        for k in failed:
            print(f"    {k}: {RUN['stages'][k].get('error')}")
    if stopped:
        print("\n!!! STOPPED AT THE TIME LIMIT (attach this version's output as an input and run again; finished jobs are reused):")
        for k in stopped:
            print(f"    {k}: {RUN['stages'][k].get('reason')}")
    skipped = [k for k in skipped if k[4:] not in nodata]
    if skipped:
        print("\nskipped:")
        for k in skipped:
            print(f"    {k}: {RUN['stages'][k].get('reason')}")
    if RUN["warnings"]:
        print(f"\n{len(RUN['warnings'])} warning(s):")
        for w in RUN["warnings"]:
            print("    -", w)
    jt = ST.get("gpu_jobs")
    if jt is not None and len(jt):
        n_ok = int((jt["status"] == "ok").sum())
        print(f"\nGPU jobs: {n_ok} of {len(jt)} finished" + ("" if n_ok == len(jt) else " -> attach this version's output as an input and run the notebook again to finish the rest"))
    S = ST.get("summarise")
    if S is not None and len(S):
        print("\nvideo AUC [95% interval], main4:")
        for r in S[(S["config"] == "main4") & S["kind"].isin(["ffpp_classic", "dfd_all"])].itertuples():
            print(f"    {r.detector:6s} {r.target:34s} {r.auc:.4f} [{r.auc_ci_lo:.4f}, {r.auc_ci_hi:.4f}]" + (f"   seeds {r.n_seeds}: {r.auc_min:.4f} - {r.auc_max:.4f}" if r.n_seeds > 1 else ""))
    if not EXP:
        print(f"\n!!! The export stage failed, so there is no ZIP. Figures and tables written so far are in {os.path.basename(OUT)}/ in the Output tab.")
        return
    if EXP["val"]["problems"]:
        print("\nvalidation problems:")
        for p in EXP["val"]["problems"]:
            print("    -", p)
    print(f"\nZIP: {EXP['zip']}  ({EXP['size_mb']:.1f} MB; {EXP['n_png']} figures as PNG at {CFG['DPI']} DPI + {EXP['n_pdf']} as PDF, tables, manifest)")
    print("\nTo download: open the Output tab of this notebook version (after 'Save & Run All'), or click the link below in an interactive session.")
    try:
        from IPython.display import FileLink, display
        display(FileLink(os.path.relpath(EXP["zip"], os.getcwd())))
    except Exception:
        pass

try:
    final_report()
except Exception as _e:
    print(f"(the final summary could not be printed: {type(_e).__name__}: {_e}; see metadata/run_manifest.json in the output folder)")
print("=" * 100)